# F1 Podium Prediction – Milestone 1
**Task:** For each driver in each race, predict whether they finish on the podium (top 3), using only information available after qualifying and before the race starts.

## 1. Setup & Imports

In [ ]:
# ---- Imports ----
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score, precision_score, recall_score,
                             roc_curve, precision_recall_curve, confusion_matrix)
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
import xgboost as xgb
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow import keras
import shap
try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:      # lime is not in every Kaggle image: install it once (needs Internet switched on in the notebook settings)
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lime"])
    from lime.lime_tabular import LimeTabularExplainer

# ---- Find the data folder automatically (works on Kaggle and in Claude) ----
def find_data_dir():
    """Find the folder that contains results.csv."""
    for root in ["/kaggle/input", "data"]:
        if os.path.exists(root):
            for path, dirs, files in os.walk(root):
                if "results.csv" in files:
                    return path
    raise FileNotFoundError("Couldn't find results.csv — is the dataset attached?")

# ---- Constants ----
SEED = 42
DATA_DIR = find_data_dir()
OUTPUT_DIR = "clean_data"    # folder for files we save (Decision 14)
TARGET = "podium"
TRAIN_START_YEAR = 2003 # first season in the modeling table (Decision 4)
TRAIN_END_YEAR = 2018  # last training season (Decision 10)
VAL_END_YEAR = 2021    # last validation season; test = later seasons (Decision 10)
FIGSIZE = (10, 5)

# ---- Seeds ----
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)   # Keras models (Section 10)

# ---- Display settings ----
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = FIGSIZE

# ---- Shared evaluation function (used by every model) ----
results_table = []

def evaluate(name, y_true, y_prob, split, threshold=0.5):
    """Score one model on one split and add a row to results_table."""
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    row = {
        "model": name, "split": split, "threshold": threshold,
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
    }
    results_table.append(row)
    return row

# ---- Quick check ----
print("DATA_DIR =", DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

## 2. Data Loading
**Goal:** Load the 14 raw CSV tables exactly as they are in the files, and check we have the right dataset.

**Uses:** DATA_DIR

**Creates:** raw, table_shapes

**Decision 13:** We read every CSV as-is (`keep_default_na=False`). The placeholder `\N` stays as text in `raw`; it is turned into NaN later, in Section 5 (Cleaning), on copies. pandas guesses the column types, and we load all 14 tables, including `lap_times`.

**Why:**
- The brief makes sentinel (`\N`) handling a graded *cleaning* step and asks for EDA before vs after cleaning, so we need a visible "before".
- pandas' default read silently turns strings like `"NA"` or `"null"` into NaN, which could change real values without us noticing.
- It keeps `raw` truly untouched, as the team conventions require.

**Evidence:** `table_shapes` (cell 2.4): the `n_backslash_N` and `n_default_na_strings` columns.

**Rule that follows:** only Sections 3–4 (Audit, EDA-before) read `raw`. From Section 5 on, we use `clean`.

In [ ]:
# Read every table exactly as it is in the file (Decision 13):
#   keep_default_na=False -> "\N" and empty cells stay as text; nothing becomes NaN
#   low_memory=False      -> pandas reads the whole column before choosing its type
expected_tables = [
    "circuits", "constructors", "constructor_results", "constructor_standings",
    "drivers", "driver_standings", "races", "results", "qualifying",
    "lap_times", "pit_stops", "sprint_results", "seasons", "status",
]

raw = {}
for name in expected_tables:
    path = os.path.join(DATA_DIR, f"{name}.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing table: {name}.csv was not found in {DATA_DIR}")
    raw[name] = pd.read_csv(path, keep_default_na=False, low_memory=False)

print(f"Loaded {len(raw)} tables from {DATA_DIR}")

### 2.3 Loading checks
These checks stop Run All early if we loaded the wrong dataset or a changed version: 14 tables, no empty tables, no repeated column names, the columns later sections need, and ID columns read as integers (a hidden `\N` in a key would turn it into text).

In [ ]:
# Loading checks: stop Run All early if we loaded the wrong or a changed dataset.
assert len(raw) == 14, f"Expected 14 tables, got {len(raw)}"

for name, df in raw.items():
    assert len(df) > 0, f"{name} has no rows"
    # read_csv renames repeated headers to "col.1", "col.2", ... so look for that pattern
    renamed = [c for c in df.columns if pd.Series([c]).str.match(r".+\.\d+$").iloc[0]]
    assert not renamed, f"{name} has repeated column names: {renamed}"

# Columns that later sections depend on
required_columns = {
    "results":               ["raceId", "driverId", "constructorId", "grid", "positionOrder", "statusId"],
    "qualifying":            ["raceId", "driverId", "position", "q1", "q2", "q3"],
    "races":                 ["raceId", "year", "round", "circuitId", "date"],
    "drivers":               ["driverId", "dob", "nationality"],
    "circuits":              ["circuitId", "country"],
    "driver_standings":      ["raceId", "driverId", "points", "position"],
    "constructor_standings": ["raceId", "constructorId", "points", "position"],
    "sprint_results":        ["raceId", "driverId", "grid", "positionOrder"],
    "status":                ["statusId", "status"],
}
for name, cols in required_columns.items():
    missing = [c for c in cols if c not in raw[name].columns]
    assert not missing, f"{name} is missing columns: {missing}"

# ID columns must be read as integers. If a "\N" were hiding in a key,
# pandas would read the whole column as text and this check would fail.
id_columns = {
    "circuits":              ["circuitId"],
    "constructors":          ["constructorId"],
    "drivers":               ["driverId"],
    "status":                ["statusId"],
    "seasons":               ["year"],
    "races":                 ["raceId", "year", "round", "circuitId"],
    "results":               ["resultId", "raceId", "driverId", "constructorId", "statusId"],
    "qualifying":            ["qualifyId", "raceId", "driverId", "constructorId"],
    "sprint_results":        ["resultId", "raceId", "driverId", "constructorId", "statusId"],
    "driver_standings":      ["driverStandingsId", "raceId", "driverId"],
    "constructor_standings": ["constructorStandingsId", "raceId", "constructorId"],
    "constructor_results":   ["constructorResultsId", "raceId", "constructorId"],
    "lap_times":             ["raceId", "driverId", "lap"],
    "pit_stops":             ["raceId", "driverId", "stop"],
}
for name, cols in id_columns.items():
    for col in cols:
        assert pd.api.types.is_integer_dtype(raw[name][col]), \
            f"{name}.{col} was read as {raw[name][col].dtype}, not integer"

print("All loading checks passed")

### 2.4 Size and placeholders per table
One row per table: size, memory, and how many cells hold the `\N` placeholder, an empty string, or a string pandas would normally turn into NaN. This is the evidence for Decision 13 and feeds Section 3 (Audit) and Section 4 (EDA).

In [ ]:
# Strings that pandas turns into NaN by default (pandas read_csv docs, na_values).
# "" is left out here because empty cells are counted separately.
pandas_default_na_strings = [
    "#N/A", "#N/A N/A", "#NA", "-1.#IND", "-1.#QNAN", "-NaN", "-nan", "1.#IND",
    "1.#QNAN", "<NA>", "N/A", "NA", "NULL", "NaN", "None", "n/a", "nan", "null",
]

shape_rows = []
for name, df in raw.items():
    text_cols = df.select_dtypes(include=["object", "string"])
    shape_rows.append({
        "table": name,
        "n_rows": len(df),
        "n_cols": df.shape[1],
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1e6, 1),
        "n_backslash_N": int((text_cols == "\\N").sum().sum()),
        "n_empty_string": int((text_cols == "").sum().sum()),
        "n_default_na_strings": int(text_cols.isin(pandas_default_na_strings).sum().sum()),
        "n_text_cols": text_cols.shape[1],
    })

table_shapes = (pd.DataFrame(shape_rows)
                .sort_values("n_rows", ascending=False)
                .reset_index(drop=True))
table_shapes

**What `table_shapes` shows:**
- **Size:** `lap_times` is by far the largest table (589,081 rows, ~57 MB in memory); the lookup tables (`status`, `circuits`, `seasons`) are tiny. `results`, our main table, has 26,759 rows (one per driver per race, before duplicates are removed).
- **`\N` placeholders:** 160,076 cells in 6 tables hold `\N` instead of a value. Most are in `results` (122,887: mainly race-result columns like `time`, `milliseconds`, `fastestLap`, which are empty when a driver didn't finish or the data wasn't recorded), `constructor_results` (12,608), `qualifying` (11,600) and `races` (11,349). Because of them, number columns like `results.position` are stored as text until Section 5.
- **Two different missing markers:** `qualifying` uses `\N` **and** empty cells (68, in `q2`/`q3`) for "no time". pandas' default read would have turned only the empty cells into NaN and left `\N` as text, giving two kinds of missing in one column. Reading as-is (Decision 13) keeps both visible, so Section 5 can convert both on purpose.
- **Default-NA strings:** 0 cells hold strings like `"NA"` or `"null"`. So in this dataset the default read would not have changed real values. Decision 13 is the safer choice here, not a required one; its main reasons are the visible before/after and the `qualifying` point above.

### 2.6 First look at the core tables
The first rows of `results`, `qualifying` and `races` as loaded (`\N` still visible), plus a check that names with accents were read correctly (utf-8).

In [ ]:
# First rows of the 3 core tables (as loaded: "\N" still visible)
display(raw["results"].head(3))
display(raw["qualifying"].head(3))
display(raw["races"].head(3))

# Spot-check: names with accents (non-ASCII letters) read correctly -> utf-8 is fine
drivers_preview = raw["drivers"]
accented = drivers_preview[drivers_preview["surname"].str.contains(r"[^\x00-\x7F]", regex=True)]
print(f"Drivers with accented surnames: {len(accented)}")
display(accented[["forename", "surname", "nationality"]].head(5))

### 2.7 Table guide: what each table is, and when its information is known

| Table | One row = | Key | Links to | Years in data | When is it known? |
|---|---|---|---|---|---|
| circuits | a track | circuitId | — | — | Fixed info → **safe** |
| constructors | a team | constructorId | — | — | Fixed → **safe** |
| drivers | a driver | driverId | — | — | Fixed (dob, nationality) → **safe** |
| seasons | a year | year | — | 1950–2024 | Fixed → **safe** |
| status | an end-of-race reason | statusId | — | — | Lookup; a result's status is **after the race** |
| races | a race on the calendar | raceId | circuits, seasons | 1950–2024 | Known in advance → **safe** |
| qualifying | a driver's qualifying session | qualifyId | races, drivers, constructors | **1994–2024 only** | After qualifying → **safe (our cutoff)** |
| results | a driver in a race | resultId | races, drivers, constructors, status | 1950–2024 | Source of the **target**. `grid` is pre-race; all other result columns are **after the race** |
| sprint_results | a driver in a sprint | resultId | races, drivers, constructors, status | 2021–2024 (18 sprints) | Before Sunday's race → **Decision #9** needed |
| driver_standings | a driver's standing **after** a race | driverStandingsId | races, drivers | | **After race X** (it includes race X's points) → use the previous round only |
| constructor_standings | a team's standing **after** a race | constructorStandingsId | races, constructors | | **After race X** → use the previous round only |
| constructor_results | a team's points in a race | constructorResultsId | races, constructors | | **After the race** |
| lap_times | a driver on one lap | (raceId, driverId, lap) | races, drivers | 1996–2024 | **During the race** → past races only, if ever used |
| pit_stops | one pit stop | (raceId, driverId, stop) | races, drivers | 2011–2024 | **During the race** → past races only, if ever used |

Proof that standings include the race itself: for the 2009 Australian GP (raceId 1, the first race of that season), every driver's standings points equal their points from that race.

### 2.8 Notes for later sections (nothing is fixed here)
- **Qualifying exists from 1994 but is only complete from 2003**: in 1996–2002 most races have no qualifying rows (e.g. only 6% of 2001 results rows have one). Key evidence for Decision 4 (training years = 2003–2024); see the qualifying-coverage chart in Section 4.
- **Missing markers:** `\N` everywhere, plus empty cells in `qualifying.q2`/`q3`. Section 5 must convert **both** to NaN.
- **`sprint_results` has no `rank` or `fastestLapSpeed` columns**, unlike `results` → the two tables can't be stacked blindly.
- **`pit_stops.time` is a clock time** (e.g. `17:05:23`), not a duration; the stop length is in `duration` / `milliseconds`.
- **`races` session dates** (`fp1_date` … `quali_date`, `sprint_date`) are `\N` before 2021.
- **`constructor_results.status`** is `\N` in 12,608 of 12,625 rows; the other 17 are `"D"` (disqualified).
- **Times stored as text:** `qualifying.q1/q2/q3`, `results.fastestLapTime`, `lap_times.time` look like `"1:27.452"` → convert to seconds in Section 5.
- **Every race in `races` has results**, so there are no cancelled or future races to drop.

## 3. Audit
**Goal:** Check the raw data for problems: column types, missing values, primary and foreign keys, and the one-row-per-driver-per-race grain. This section only finds and records problems; it fixes nothing (fixes happen in Section 5, Cleaning).

**Uses:** raw, table_shapes

**Creates:** dtype_summary, missing_summary, key_audit, results_dups

### 3.1 Data types
Because `\N` sits inside some columns, pandas stores them as text even when the real values are numbers, dates or times. For every text column we test whether **all its real values** (ignoring `\N` and empty cells) look like a number, a date (`2009-03-29`), a lap time (`1:26.572`) or a clock time (`06:00:00`). The tests run on copies, so `raw` is not changed. The result, `dtype_summary`, is the conversion to-do list for Section 5.

In [ ]:
# In raw, missing values are the text "\N" or an empty cell (Decision 13).
sentinels = ["\\N", ""]

# Patterns we test text values against (tests run on copies; raw is not changed)
patterns = {
    "looks_date":    r"\d{4}-\d{2}-\d{2}",       # 2009-03-29
    "looks_laptime": r"\d+:\d{2}\.\d+",           # 1:26.572  (minutes:seconds.ms)
    "looks_clock":   r"\d{2}:\d{2}:\d{2}",        # 06:00:00  (time of day)
}

dtype_rows = []
for name, df in raw.items():
    for col in df.columns:
        s = df[col]
        row = {"table": name, "column": col, "dtype": str(s.dtype),
               "looks_numeric": False, "looks_date": False,
               "looks_laptime": False, "looks_clock": False}
        if s.dtype == object:
            real = s[~s.isin(sentinels)].astype(str)   # only the real values
            row["example"] = real.iloc[0] if len(real) else None
            if len(real):
                row["looks_numeric"] = bool(pd.to_numeric(real, errors="coerce").notna().all())
                for flag, pattern in patterns.items():
                    row[flag] = bool(real.str.fullmatch(pattern).all())
        else:
            row["example"] = s.iloc[0]
        dtype_rows.append(row)

dtype_summary = pd.DataFrame(dtype_rows)
flag_cols = ["looks_numeric", "looks_date", "looks_laptime", "looks_clock"]
text_cols = dtype_summary["dtype"] == "object"

# Text columns whose real values are all numbers, dates or times -> convert in Section 5
needs_conversion = dtype_summary[text_cols & dtype_summary[flag_cols].any(axis=1)]
print(f"{len(needs_conversion)} text columns need converting in Section 5:")
display(needs_conversion[["table", "column", "example"] + flag_cols].reset_index(drop=True))

# Text columns that match none of the patterns -> check they really are text
other_text = dtype_summary[text_cols & ~dtype_summary[flag_cols].any(axis=1)]
print(f"{len(other_text)} other text columns (names, codes, URLs, or mixed formats):")
display(other_text[["table", "column", "example"]].reset_index(drop=True))

**What `dtype_summary` shows:**
- **29 text columns need converting in Section 5:** 10 to numbers (e.g. `results.position`, `milliseconds`), 7 to dates (`races.date`, `drivers.dob`, 5 session dates), 5 lap times to seconds (`qualifying.q1`–`q3`, both `fastestLapTime`), and 7 clock times (`races.time`, 5 session times, `pit_stops.time`).
- **Most other text columns really are text:** names, refs, URLs, nationality, `status`, and `positionText` (it mixes positions with letter codes such as `R` for retired; see 3.2).
- **Four columns have mixed formats** and need special handling in Section 5:
  - `lap_times.time`: almost all laps look like `1:38.109`, but 43 laps took over an hour (`2:05:05.152`, e.g. laps run during a red flag), so the time converter must handle hours too. `milliseconds` already holds the same value as a number.
  - `pit_stops.duration`: mostly seconds (`26.898`), but 517 of 11,371 stops are written as `m:ss.sss` (e.g. `16:44.718`), often stops during a red flag. `milliseconds` already holds the same value as a number.
  - `results.time` / `sprint_results.time`: the winner's row holds the **total race time** (`1:34:50.616`), the other finishers hold the **gap to the winner** (`+5.478`). The column is not one quantity, so we should not convert it; `milliseconds` already holds each driver's total time as a number.
- **`pit_stops.time` is a time of day** (when the stop happened), not how long it took.
- **Leakage reminder:** the result and timing columns (`position`, `milliseconds`, `fastestLap…`, `lap_times`, `pit_stops`) are after the race. Converting them is for EDA and Q3; they never become features for the same race.

### 3.2 Missing values and hidden sentinels
In `raw`, missing values are the text `\N` or an empty cell (Decision 13), so we count those strings. `.isna()` would find nothing. We also look at "hidden sentinels": values that are not missing but carry a special meaning, like `grid = 0` and the letter codes in `positionText`.

In [ ]:
# Count missing values per column. In raw they are the text "\N" or "" (Decision 13),
# so we compare strings; .isna() would find nothing.
missing_rows = []
for name, df in raw.items():
    for col in df.columns:
        s = df[col]
        if s.dtype != object:          # number columns were read cleanly -> nothing missing
            continue
        n_backslash_N = int((s == "\\N").sum())
        n_empty = int((s == "").sum())
        if n_backslash_N + n_empty > 0:
            missing_rows.append({
                "table": name, "column": col, "n_rows": len(s),
                "n_backslash_N": n_backslash_N, "n_empty": n_empty,
                "n_missing": n_backslash_N + n_empty,
                "pct_missing": round(100 * (n_backslash_N + n_empty) / len(s), 1),
            })

missing_summary = (pd.DataFrame(missing_rows)
                   .sort_values("pct_missing", ascending=False)
                   .reset_index(drop=True))

# Self-check: totals per table must match table_shapes from Section 2
totals = missing_summary.groupby("table")[["n_backslash_N", "n_empty"]].sum()
expected = table_shapes.set_index("table")[["n_backslash_N", "n_empty_string"]]
expected = expected[(expected > 0).any(axis=1)]
assert (totals["n_backslash_N"] == expected.loc[totals.index, "n_backslash_N"]).all()
assert (totals["n_empty"] == expected.loc[totals.index, "n_empty_string"]).all()
assert set(totals.index) == set(expected.index)

print(f"{len(missing_summary)} columns in {missing_summary['table'].nunique()} tables have missing values "
      f"({missing_summary['n_missing'].sum():,} cells). Totals match table_shapes.")
missing_summary

**What `missing_summary` shows:** 160,144 missing cells (160,076 `\N` + 68 empty) in 30 columns of 6 tables. They fall into three kinds:

| Kind | Columns | Why missing | What it means for us |
|---|---|---|---|
| **Expected (structural)** | `races` session dates/times (92–99%), `races.time` (65%), `drivers.number` / `code` (old drivers), `qualifying.q2` / `q3` | Session schedules only recorded from 2021; old races have no start time; permanent numbers and 3-letter codes are modern; drivers knocked out in Q1/Q2 have no Q2/Q3 time | Normal, not a data error. `q2`/`q3` missing even carries information (didn't reach Q2/Q3) |
| **After the race** | `results.position`, `time`, `milliseconds`, `fastestLap…`, `rank`; same in `sprint_results` | Driver didn't finish / wasn't classified, or lap data wasn't recorded for old races | **Leakage:** never use these, or whether they are missing, as features for the same race |
| **To check** | `qualifying.q1` (156 rows), `results.number` (6), `constructor_results.status` (99.9%) | `q1` missing mostly for back-of-grid drivers (median qualifying position 20) who set no time; `status` is only filled when a team was disqualified (`D`) | Small; handled in Cleaning (NaN). `constructor_results.status` is almost empty and not needed |

In [ ]:
# Hidden sentinels: values that are not missing but have a special meaning.

# 1) grid = 0 -> the driver did not start from a grid slot (pit-lane start or did not start)
grid_zero = pd.DataFrame([
    {"table": t, "n_rows": len(raw[t]),
     "n_grid_0": int((raw[t]["grid"] == 0).sum()),
     "pct_grid_0": round(100 * (raw[t]["grid"] == 0).mean(), 1)}
    for t in ["results", "sprint_results"]
])
display(grid_zero)

# 2) positionText letter codes -> why "position" is missing
position_codes = (pd.concat([
        raw[t].loc[~raw[t]["positionText"].astype(str).str.isdigit(), "positionText"]
              .value_counts().rename(t)
        for t in ["results", "sprint_results", "driver_standings", "constructor_standings"]
    ], axis=1)
    .fillna(0).astype(int))
position_codes.index.name = "positionText code"
display(position_codes)

# 3) In results: which codes do the grid = 0 rows have? (evidence for Decision #8)
res = raw["results"]
grid0_codes = (res.loc[res["grid"] == 0, "positionText"]
               .where(lambda x: ~x.str.isdigit(), "finished (number)")
               .value_counts()
               .rename("n_rows_with_grid_0"))
display(grid0_codes.to_frame())

**What the hidden sentinels show:**
- **`positionText` codes** explain why `position` is missing: `R` retired, `F` failed to qualify, `W` withdrew, `N` not classified, `D` disqualified, `E` excluded. They cover 10,951 of the 10,953 missing `results.position` values. The other 2 rows (2005 Australian GP, raceId 71) have a number in `positionText` but `\N` in `position`, a small data inconsistency. Our target uses `positionOrder`, which is never missing, so it is not affected.
- **`grid = 0` means two different things:**
  - **Did not start** (~1,550 rows, mostly before 2000): failed to qualify (`F`, 1,348) or withdrew (`W`, 203). These drivers are in `results` but **never raced**.
  - **Started from the pit lane** (60 rows that finished, all from 2000 onwards): a real start, just not from a grid slot.
- **Why it matters:** the "did not qualify" rows are not real race entries. If they stay in the modeling table they add ~1,550 easy "no podium" rows with grid = 0, and the model could learn "grid 0 = no podium" from drivers who never started. This is the main evidence for **Decision #8** (grid = 0 / non-starters). Grid charts follow in Section 4.

### 3.3 Primary keys
A primary key identifies each row, so it must be **unique** and **never empty**. We check every table's ID column, the two tables that use a combination of columns as their key (`lap_times`, `pit_stops`), and "natural keys": combinations that should also be unique even though the table has its own ID (e.g. one standings row per driver per race). `n_duplicates` counts extra rows: every repeat of a key after its first appearance.

In [ ]:
def is_empty(s):
    """True where a value is missing: NaN, "\\N" or "" (raw keeps sentinels as text, Decision 13)."""
    return s.isna() | s.astype(str).isin(sentinels)

def check_pk(table, cols, key_type="PK"):
    """Check that a key is never empty and unique. Returns one row for key_audit."""
    df = raw[table]
    empty = pd.concat([is_empty(df[c]) for c in cols], axis=1).any(axis=1)
    return {
        "table": table, "key": " + ".join(cols), "key_type": key_type,
        "n_rows": len(df),
        "n_empty": int(empty.sum()),
        # extra rows: every repeat of a key after its first appearance
        "n_duplicates": int(df.duplicated(subset=cols, keep="first").sum()),
    }

pk_checks = [
    # single-column IDs
    ("circuits", ["circuitId"]), ("constructors", ["constructorId"]),
    ("drivers", ["driverId"]), ("status", ["statusId"]), ("seasons", ["year"]),
    ("races", ["raceId"]), ("results", ["resultId"]), ("qualifying", ["qualifyId"]),
    ("sprint_results", ["resultId"]),
    ("driver_standings", ["driverStandingsId"]),
    ("constructor_standings", ["constructorStandingsId"]),
    ("constructor_results", ["constructorResultsId"]),
    # composite keys (no ID column in the file)
    ("lap_times", ["raceId", "driverId", "lap"]),
    ("pit_stops", ["raceId", "driverId", "stop"]),
]
natural_checks = [
    # combinations that should also be unique, even though the table has its own ID
    ("races", ["year", "round"]),
    ("driver_standings", ["raceId", "driverId"]),
    ("constructor_standings", ["raceId", "constructorId"]),
    ("constructor_results", ["raceId", "constructorId"]),
]
# The (raceId, driverId) grain of results / qualifying / sprint_results is checked in 3.6.

pk_audit = pd.DataFrame(
    [check_pk(t, cols, "PK") for t, cols in pk_checks]
    + [check_pk(t, cols, "natural key") for t, cols in natural_checks]
)
n_problems = int((pk_audit[["n_empty", "n_duplicates"]].sum(axis=1) > 0).sum())
print(f"{len(pk_audit)} keys checked, {n_problems} with empty values or duplicates.")
pk_audit

### 3.4 Foreign keys
A foreign key is a column that points to another table (e.g. `results.driverId` → `drivers`). Every value must exist in the table it points to; a value with no match is a **broken link**. Empty values are counted separately, not as broken. We check all 23 links between the 14 tables.

In [ ]:
def check_fk(child, col, parent, parent_col=None):
    """Check that every value in child[col] exists in parent[parent_col]. Returns one row for key_audit."""
    parent_col = parent_col or col
    values = raw[child][col]
    empty = is_empty(values)
    real = values[~empty]
    broken = real[~real.isin(set(raw[parent][parent_col]))]
    return {
        "table": child, "key": f"{col} -> {parent}.{parent_col}", "key_type": "FK",
        "n_rows": len(values),
        "n_empty": int(empty.sum()),
        "n_broken": int(len(broken)),
        "broken_examples": sorted(broken.unique().tolist())[:5],
    }

fk_checks = [
    ("results", "raceId", "races"), ("results", "driverId", "drivers"),
    ("results", "constructorId", "constructors"), ("results", "statusId", "status"),
    ("sprint_results", "raceId", "races"), ("sprint_results", "driverId", "drivers"),
    ("sprint_results", "constructorId", "constructors"), ("sprint_results", "statusId", "status"),
    ("qualifying", "raceId", "races"), ("qualifying", "driverId", "drivers"),
    ("qualifying", "constructorId", "constructors"),
    ("races", "circuitId", "circuits"), ("races", "year", "seasons"),
    ("driver_standings", "raceId", "races"), ("driver_standings", "driverId", "drivers"),
    ("constructor_standings", "raceId", "races"),
    ("constructor_standings", "constructorId", "constructors"),
    ("constructor_results", "raceId", "races"),
    ("constructor_results", "constructorId", "constructors"),
    ("lap_times", "raceId", "races"), ("lap_times", "driverId", "drivers"),
    ("pit_stops", "raceId", "races"), ("pit_stops", "driverId", "drivers"),
]

fk_audit = pd.DataFrame([check_fk(child, col, parent) for child, col, parent in fk_checks])
n_problems = int((fk_audit[["n_empty", "n_broken"]].sum(axis=1) > 0).sum())
print(f"{len(fk_audit)} links checked, {n_problems} with empty values or broken links.")
fk_audit

**What the key checks show:** all 18 primary/natural keys are unique and never empty, and all 23 links point to rows that exist. So the tables join cleanly: no rows will be lost or duplicated by a broken key when we combine them later, and Cleaning does not need any key fixes.

This is not the same as "one row per driver per race": `results` can still hold the same driver twice in one race with two different `resultId`s (shared drives). That is checked in 3.6.

We tested that the checks really catch problems: on a copy of the data with a planted duplicate, an empty key and two broken links, they reported exactly those.

### 3.5 Key audit summary
All primary-key, natural-key and foreign-key checks from 3.3 and 3.4 in one table: key | # empty | # duplicates | # broken links. `<NA>` means the check does not apply (duplicates are checked for primary keys, broken links for foreign keys). Rows with a problem are sorted to the top.

In [ ]:
# One summary table for all key checks (TODO 1.4): key | # empty | # duplicates | # broken links
key_audit = pd.concat([pk_audit, fk_audit], ignore_index=True)
key_audit = key_audit[["table", "key", "key_type", "n_rows", "n_empty", "n_duplicates", "n_broken"]]
# <NA> = this check does not apply (duplicates are checked for PKs, broken links for FKs)
key_audit[["n_duplicates", "n_broken"]] = key_audit[["n_duplicates", "n_broken"]].astype("Int64")

key_audit["has_problem"] = key_audit[["n_empty", "n_duplicates", "n_broken"]].fillna(0).sum(axis=1) > 0
key_audit = key_audit.sort_values(["has_problem", "key_type", "table"],
                                  ascending=[False, True, True]).reset_index(drop=True)

print(key_audit.groupby("key_type")["has_problem"].agg(checked="size", with_problems="sum"))
key_audit

**What `key_audit` shows:** 41 key checks, 0 problems. Every ID is unique and never empty, every combination that should be unique is, and every link between tables points to a row that exists. The tables can be joined safely and **Cleaning needs no key fixes**.

### 3.6 Grain: one row per driver per race
Our modeling table needs exactly one row per `(raceId, driverId)`, because each row is one prediction. In early F1, drivers sometimes **shared cars**: a driver could take over a teammate's car during the race, so `results` lists him once per car. We count these repeats, look at when they happen, and look at what tells the copies apart. This is the evidence for Decision #5 (duplicate rule); the rule itself is chosen and applied in Cleaning.

In [ ]:
# How often does the same driver appear more than once in the same race?
grain_key = ["raceId", "driverId"]

def check_grain(table):
    """Count (raceId, driverId) pairs that appear more than once in a table."""
    df = raw[table]
    repeated = df.duplicated(subset=grain_key, keep=False)
    return {
        "table": table, "n_rows": len(df),
        "n_pairs_repeated": int(df.loc[repeated, grain_key].drop_duplicates().shape[0]),
        "n_rows_involved": int(repeated.sum()),
        "n_extra_rows": int(df.duplicated(subset=grain_key, keep="first").sum()),
    }

grain_check = pd.DataFrame([check_grain(t) for t in ["results", "qualifying", "sprint_results"]])
grain_check

In [ ]:
# All results rows that belong to a repeated (raceId, driverId) pair.
# Names are joined in only to make the rows readable; raw is not changed.
res = raw["results"]
results_dups = (
    res[res.duplicated(subset=grain_key, keep=False)]
    .merge(raw["races"][["raceId", "year", "name"]].rename(columns={"name": "race"}), on="raceId")
    .merge(raw["drivers"][["driverId", "forename", "surname"]], on="driverId")
    .merge(raw["constructors"][["constructorId", "name"]].rename(columns={"name": "constructor"}),
           on="constructorId")
    .merge(raw["status"], on="statusId")
    .sort_values(["year", "raceId", "driverId", "resultId"])
    .reset_index(drop=True)
)
print(f"{len(results_dups)} rows in {results_dups.groupby(grain_key).ngroups} repeated pairs, "
      f"years {results_dups['year'].min()}–{results_dups['year'].max()}")

In [ ]:
# Chart: repeated driver-race pairs per season (all seasons shown, so the pattern over time is visible)
all_years = range(raw["races"]["year"].min(), raw["races"]["year"].max() + 1)
pairs_per_year = (results_dups.drop_duplicates(subset=grain_key)
                  .groupby("year").size()
                  .reindex(all_years, fill_value=0))

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.bar(pairs_per_year.index, pairs_per_year.values, color="#2a78d6", width=0.8)
ax.set_title("Same driver listed more than once in a race (shared drives), per season")
ax.set_xlabel("Season")
ax.set_ylabel("Repeated (raceId, driverId) pairs")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

**What tells the copies apart (evidence for Decision #5):**
- Each row is a **different car** (the car number differs in all 85 pairs; the team differs in only 16). The driver drove more than one car in the race.
- **`grid` cannot pick "the car he started in":** in 82 of 85 pairs *both* cars have a real grid slot, because the second car was started by a teammate before the driver took it over. Only 3 pairs are all grid = 0.
- **Trap: "keep the first row" is not neutral.** `resultId` follows the finishing order in 1,074 of 1,125 races, so keeping the lowest `resultId` would partly choose rows by race result, which the TA rule forbids.
- **It affects the target:** in 22 of the 85 pairs at least one of the cars finished on the podium, so the rule decides whether these 22 driver-races count as a podium.
- **Link to Decision #6:** 21 of the 85 pairs are in the Indianapolis 500. If the Indy 500 is dropped, 64 pairs remain.
- **Candidate rules to compare in Cleaning** (none uses race results): (a) drop all rows of these 85 pairs (176 rows, 0.7% of `results`); (b) keep the car with the better starting slot (lowest grid > 0); (c) merge the pair into one row (best grid; podium if any car was on the podium) — note that (c) uses the result to build the target, which is fine for the label but must be justified.

### 3.7 Audit findings → what happens next

| # | Finding | Found in | Fixed / used in | Decision |
|---|---|---|---|---|
| 1 | Missing values are `\N` (160,076 cells) and empty cells (68, `qualifying.q2`/`q3`) | 2.4, 3.2 | 5: turn both into NaN | 13 |
| 2 | 29 columns stored as text: 10 numbers, 7 dates, 5 lap times, 7 clock times | 3.1 | 5: convert types | — |
| 3 | Mixed formats: `lap_times.time` (43 laps over an hour), `pit_stops.duration` (517 in `m:ss.sss`), `results.time` (total time vs gap) | 3.1 | 5: use the `milliseconds` columns instead | — |
| 4 | Missing race-result columns mean "didn't finish" | 3.2 | 8: never a feature (leakage) | — |
| 5 | `grid = 0`: ~1,550 drivers who never started (failed to qualify / withdrew) + pit-lane starters | 3.2 | 5: drop `positionText` F / W (1,704 rows); keep the 87 pit-lane starters | **8** |
| 6 | 2 rows (2005 Australian GP) with `position` = `\N` but a number in `positionText` | 3.2 | none needed: target uses `positionOrder` | — |
| 7 | All 41 key checks pass: no empty keys, duplicates or broken links | 3.3–3.5 | nothing to fix | — |
| 8 | 85 driver-race pairs listed more than once (176 rows, 1950–1978); 22 involve a podium | 3.6 | 5: drop all rows of repeated pairs (61 pairs / 125 rows left after Decisions 6 and 8), then `assert` unique | **5** |
| 9 | Indianapolis 500: separate drivers, 21 of the 85 duplicate pairs | 3.6 | 5: drop the 11 Indy 500 races (405 rows) | **6** |
| 10 | Qualifying only complete from 2003 | 2.8 | 8: modeling table uses 2003–2024 (`TRAIN_START_YEAR`) | **4** |
| 11 | Standings include race X; sprint results exist | 2.7 | 8: use previous round; sprint call | 9 (bundle 3) |

**Order in Section 5:** Decision 6 → Decision 8 → Decision 5 → `assert` (raceId, driverId) unique. Expected `results` rows: 26,759 → 26,354 → 24,650 → 24,525.

## 4. EDA — Before Cleaning
**Goal:** Show what the raw data looks like before any cleaning: coverage over time, table sizes, field sizes, missing values, the grid, retirements and class balance. These charts are the evidence for Decisions 3, 4 and 8, and the "before" picture that Section 6 (EDA-after) compares against.

**Uses:** raw, table_shapes, missing_summary, TRAIN_START_YEAR

**Creates:** quali_coverage, field_size, podium_by_grid, season_rates (small tables behind the charts)

Note: this is the data **before** cleaning, so it still contains the Indianapolis 500, drivers who never started, and shared-drive duplicates.

### 4.1 Qualifying coverage per season ⭐
Qualifying features (position, gap to pole, reached Q3) are one of our main feature groups. Before choosing which seasons to model, we check for how many driver-race rows in `results` a qualifying row actually exists, season by season. A row counts as covered if the same (raceId, driverId) appears in `qualifying`.

In [ ]:
# Share of results rows (driver-races) that have a matching qualifying row, per season
coverage = (raw["results"][["raceId", "driverId"]]
            .merge(raw["races"][["raceId", "year"]], on="raceId")
            .merge(raw["qualifying"][["raceId", "driverId"]].drop_duplicates(),
                   on=["raceId", "driverId"], how="left", indicator=True))
coverage["has_quali"] = coverage["_merge"] == "both"
quali_coverage = (coverage.groupby("year")["has_quali"].mean() * 100).rename("pct_with_quali")

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(quali_coverage.index, quali_coverage.values, color="#2a78d6", linewidth=2,
        marker="o", markersize=4)
ax.axvline(TRAIN_START_YEAR, color="#52514e", linestyle="--", linewidth=1)
ax.text(TRAIN_START_YEAR + 0.5, 50, f"TRAIN_START_YEAR = {TRAIN_START_YEAR}",
        color="#52514e", fontsize=9)
ax.set_title("Share of driver-race rows with qualifying data, per season")
ax.set_xlabel("Season")
ax.set_ylabel("Results rows with a qualifying row (%)")
ax.set_ylim(0, 105)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

# The years around the change, as numbers
display(quali_coverage.loc[1993:2004].round(0).astype(int).to_frame().T)

**What the chart shows:** there is no qualifying data at all before 1994. In 1994–1995 it is nearly complete, but from 1996 to 2002 most races are missing (down to 6% of rows in 2001). From 2003 every season is complete (lowest season: 98.9%). So "qualifying exists from 1994" is not enough: a model trained on 1994–2002 would see qualifying features mostly missing, unlike the seasons it is tested on.

**Decision 4 — Training years:** the modeling table (`model_df`, Section 8) uses seasons **2003–2024** (`TRAIN_START_YEAR = 2003`). Older seasons stay in `clean` for EDA and the three data-engineering questions.

**Why:** qualifying features are a main feature group, and 2003 is the first season from which qualifying data is complete. Starting earlier would mix in seasons where these features are mostly missing (1996–2002) or absent (before 1994), so the model would learn from a different situation than the one it is tested on. 2003–2024 still gives 8,910 driver-races in 428 races (14.4% podiums) after cleaning.

**Evidence:** the chart above (coverage 1996 44%, 1999 18%, 2001 6%, 2002 12%; 2003–2024 ≥ 98.9%). To back this up with an experiment, Section 11 compares a model trained on 2003+ with one trained on 1994+ (with missing qualifying).

### 4.2 Rows per table
How big each raw table is. The sizes range from 75 rows (`seasons`) to 589,081 (`lap_times`), so we use a log scale; otherwise every table except `lap_times` would be invisible.

In [ ]:
# Rows per table (log scale: lap_times is ~8,000x bigger than seasons)
sizes = table_shapes.sort_values("n_rows")

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.barh(sizes["table"], sizes["n_rows"], color="#2a78d6", height=0.7)
ax.set_xscale("log")
for y, n in enumerate(sizes["n_rows"]):
    ax.text(n * 1.1, y, f"{n:,}", va="center", fontsize=8, color="#52514e")
ax.set_title("Number of rows per raw table")
ax.set_xlabel("Rows (log scale)")
ax.set_ylabel("Table")
ax.set_xlim(right=sizes["n_rows"].max() * 4)
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

**What the chart shows:** `lap_times` dominates (589,081 rows, one per driver per lap from 1996). The tables we need most for the model are much smaller: `results` 26,759 rows (one per driver per race, before cleaning), `qualifying` 10,494, the standings tables 13–35 thousand. The lookup tables (`drivers`, `constructors`, `circuits`, `status`, `seasons`) are small. `lap_times` and `pit_stops` describe what happened **during** a race, so they can only be used for past races, if at all.

### 4.3 Races per season
How many races each season had. This matters later: per-season numbers (e.g. podium rate) are averaged over very different numbers of races.

In [ ]:
# Number of races per season
races_per_season = raw["races"].groupby("year").size().rename("n_races")

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(races_per_season.index, races_per_season.values, color="#2a78d6", linewidth=2,
        marker="o", markersize=4)
ax.set_title("Number of races per season")
ax.set_xlabel("Season")
ax.set_ylabel("Races")
ax.set_ylim(0, races_per_season.max() + 2)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

print(f"From {races_per_season.iloc[0]} races in {races_per_season.index[0]} "
      f"to {races_per_season.iloc[-1]} in {races_per_season.index[-1]} "
      f"(max {races_per_season.max()}, total {races_per_season.sum():,}).")

**What the chart shows:** calendars grew from 7 races (1950) to 24 (2024); 1,125 races in total. The 2020 dip (17 races) is the COVID season, worth remembering when choosing a validation season (Decision 10). In our modeling years (2003–2024) every season has 16–24 races, so each season contributes a similar amount of data.

### 4.4 Drivers per race: all entries vs starters
`results` lists every driver **entered** in a race, including drivers who failed to qualify (`positionText` = F) or withdrew (W) and therefore never started. We compare the average number of entries per race with the average number of **starters** (all rows except F / W) per season. The gap shows how many non-starters are in the raw data (Decision 8).

In [ ]:
# Average number of drivers per race, per season:
#   all entries = every results row; starters = without positionText F (failed to qualify) / W (withdrew)
entries = raw["results"].merge(raw["races"][["raceId", "year"]], on="raceId")
entries["is_starter"] = ~entries["positionText"].isin(["F", "W"])

per_race = entries.groupby(["year", "raceId"]).agg(all_entries=("driverId", "size"),
                                                   starters=("is_starter", "sum"))
field_size = per_race.groupby("year").mean().round(1)

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(field_size.index, field_size["all_entries"], color="#2a78d6", linewidth=2,
        label="All entries in results")
ax.plot(field_size.index, field_size["starters"], color="#eb6834", linewidth=2,
        label="Starters (without failed to qualify / withdrew)")
ax.set_title("Average number of drivers per race, per season")
ax.set_xlabel("Season")
ax.set_ylabel("Drivers per race (average)")
ax.set_ylim(0, field_size["all_entries"].max() + 3)
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

gap = field_size["all_entries"] - field_size["starters"]
print(f"Largest gap: {gap.max():.1f} drivers per race in {gap.idxmax()} "
      f"({field_size.loc[gap.idxmax(), 'all_entries']} entries vs {field_size.loc[gap.idxmax(), 'starters']} starters). "
      f"Seasons with a gap of more than 1 driver per race: {gap[gap > 1].index.min()}–{gap[gap > 1].index.max()}.")

**What the chart shows:** from the 1950s to the early 1990s, many entered drivers never started. The peak is 1989, with on average **38.8 entries per race but only 26.1 starters**: about 13 drivers per race failed to (pre)qualify. From about 1995 the two lines overlap, because entry lists were limited and almost everyone started. This is the evidence for **Decision 8**: the raw `results` table counts non-starters as if they were in the race; they can never finish on the podium, so they would add easy "no" rows. Cleaning removes them (1,704 rows). In our modeling years (2003–2024) the effect is small, but EDA-after and Q1–Q3 use all years. (The 1950s values also include the Indianapolis 500, with ~33-car fields; Decision 6 removes it.)

### 4.5 Missing values per column (before cleaning)
The share of missing values (`\N` or empty) in every column that has any, from `missing_summary` (3.2). This is the "before" picture; Section 6 shows the same after cleaning.

In [ ]:
# % missing per column (only columns with missing values), from missing_summary (3.2)
miss = missing_summary.assign(label=missing_summary["table"] + "." + missing_summary["column"])
miss = miss.sort_values("pct_missing")

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.barh(miss["label"], miss["pct_missing"], color="#2a78d6", height=0.7)
ax.set_title("Missing values per column in the raw data (\\N or empty), before cleaning")
ax.set_xlabel("Missing (% of rows)")
ax.set_ylabel("Table.column")
ax.set_xlim(0, 100)
ax.tick_params(axis="y", labelsize=7)
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

**What the chart shows:** missing values fall into clear groups (see the table in 3.2):
- **~92–99% — schedule and old metadata:** `races` session dates/times (only recorded from 2021), `constructor_results.status` (only filled for disqualifications), `drivers.number` / `code` (modern only). Not needed for the model.
- **~68–71% — race results:** `results.time`, `milliseconds`, `fastestLap…`, `rank`: missing when a driver didn't finish or the data wasn't recorded. These are **after the race** and are never used as features.
- **44–65% — `qualifying.q2` / `q3`:** missing for drivers knocked out in Q1/Q2. That is information ("didn't reach Q3"), not an error.
- **41% — `results.position`:** missing for drivers not classified; the target uses `positionOrder`, which is never missing.
- **Under 6%:** sprint columns, `qualifying.q1` (156 rows), `results.number` (6 rows, too small to see).
After cleaning (Section 5) all of these become real NaN; Section 6 compares the before and after.

### 4.6 Distribution of `grid`
How many results rows start from each grid slot. `grid = 0` means "no grid slot": either the driver never started (failed to qualify / withdrew) or started from the pit lane (see 3.2).

In [ ]:
# How many results rows start from each grid slot? grid = 0 means "no grid slot".
grid_counts = raw["results"]["grid"].value_counts().sort_index()
colors = ["#eb6834" if g == 0 else "#2a78d6" for g in grid_counts.index]

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.bar(grid_counts.index, grid_counts.values, color=colors, width=0.8)
ax.text(0.8, grid_counts.loc[0], f"grid = 0: {grid_counts.loc[0]:,} rows",
        color="#eb6834", fontsize=9, va="bottom")
ax.set_title("Results rows per starting grid slot (grid = 0 highlighted)")
ax.set_xlabel("Grid slot (0 = no grid slot)")
ax.set_ylabel("Results rows")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

print(f"grid = 0: {grid_counts.loc[0]:,} rows ({grid_counts.loc[0] / grid_counts.sum():.1%}); "
      f"grid slots 1–{grid_counts.index.max()}; slots above 26: {grid_counts.loc[27:].sum():,} rows.")

**What the chart shows:** slots 1–20 each appear in about 1,100 rows (one per race), and fewer races had 21–26 starters, so those bars fall off; slots above 26 are rare (169 rows). **`grid = 0` is the single biggest value (1,638 rows, 6.1%)**: it is not a real starting position but a marker for "no grid slot". 3.2 showed what it hides: 1,551 drivers who never started and 87 pit-lane starters. This is why `grid` cannot be used as-is (Decision 8; the pit-lane flag is for Section 8).

### 4.7 Podium rate by grid position
For each grid slot (0–26): the share of rows that finished on the podium (`positionOrder <= 3`). Slots above 26 are left out because they have too few rows (169 in total) for a stable rate.

In [ ]:
# Podium rate (positionOrder <= 3) for each grid slot 0–26
grid_df = raw["results"][["grid", "positionOrder", "positionText"]].copy()
grid_df["podium"] = grid_df["positionOrder"] <= 3
podium_by_grid = (grid_df[grid_df["grid"] <= 26]
                  .groupby("grid")["podium"].agg(n_rows="size", podium_rate="mean"))
podium_by_grid["podium_rate"] = (podium_by_grid["podium_rate"] * 100).round(1)
colors = ["#eb6834" if g == 0 else "#2a78d6" for g in podium_by_grid.index]

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.bar(podium_by_grid.index, podium_by_grid["podium_rate"], color=colors, width=0.8)
ax.text(0, 1, "0%", color="#eb6834", fontsize=9, ha="center", va="bottom")
ax.set_title("Podium rate by starting grid slot (grid = 0 highlighted)")
ax.set_xlabel("Grid slot (0 = no grid slot)")
ax.set_ylabel("Finished on the podium (%)")
ax.set_xticks(range(0, 27, 2))
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

display(podium_by_grid.loc[[0, 1, 2, 3, 4, 5, 6, 10, 15, 20]].T)

# grid = 0 split into non-starters (F / W) and pit-lane starters
g0 = grid_df[grid_df["grid"] == 0]
non_starter = g0["positionText"].isin(["F", "W"])
print(f"grid = 0, did not start (F/W): {non_starter.sum():,} rows, {g0.loc[non_starter, 'podium'].sum()} podiums | "
      f"grid = 0, started from the pit lane: {(~non_starter).sum()} rows, {g0.loc[~non_starter, 'podium'].sum()} podiums")

**What the chart shows:** starting position is strongly linked to a podium: **63.5% from pole**, 46% from 3rd, 25% from 5th, under 6% from 10th, and almost never from 15th or further back. This makes grid the obvious first feature (bundle 3) and the reason for the "grid ≤ 3" rule baseline. **`grid = 0` has a 0% podium rate**: none of the 1,551 non-starters (they never raced) and none of the 87 pit-lane starters finished on the podium. If the non-starters stayed in the data, the model would learn "grid 0 = no podium" from drivers who were never in the race. That is the evidence for **Decision 8** (drop them; keep pit-lane starters as real starts).

### 4.8 Retirement rate per season
The share of **starters** (all rows except `positionText` F / W) who retired from the race (`positionText` = R), per season. It shows how much less often cars break down or crash out today than in earlier eras.

In [ ]:
# Retirement rate per season: share of starters (no F / W) with positionText "R" (retired)
starters = entries[entries["is_starter"]]
season_rates = (starters.assign(retired=starters["positionText"] == "R",
                                podium=starters["positionOrder"] <= 3)
                .groupby("year")[["retired", "podium"]].mean() * 100).round(1)
season_rates.columns = ["retirement_rate", "podium_rate"]

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(season_rates.index, season_rates["retirement_rate"], color="#2a78d6", linewidth=2,
        marker="o", markersize=4)
ax.set_title("Share of starters who retired from the race, per season")
ax.set_xlabel("Season")
ax.set_ylabel("Retired (% of starters)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

decades = season_rates["retirement_rate"].groupby(season_rates.index // 10 * 10).mean().round(0).astype(int)
print("Average retirement rate per decade (%):", decades.to_dict())

**What the chart shows:** until the late 1990s, **40–50% of starters retired** in an average season; since 2000 the rate has dropped steadily to about **12% in the 2020s** (9% in 2024). Cars became far more reliable. Two consequences: (1) the eras behave very differently, which supports modeling a consistent period (Decision 4) and using a time-based split; (2) in modern seasons fewer surprises come from breakdowns, so pre-race information like grid and form should matter more. Bundle 2's Q3 looks at which retirements were mechanical.

### 4.9 Class balance: how rare is a podium?
Our target is `podium = positionOrder <= 3`. Only 3 drivers per race finish on the podium, so the "yes" class is small. We show the share of starters who reached the podium, per season and overall. This is the evidence for Decision 3 (PR-AUC as main metric).

In [ ]:
# Class balance: share of starters who finished on the podium, per season
overall_all = (raw["results"]["positionOrder"] <= 3).mean() * 100
overall_starters = (starters["positionOrder"] <= 3).mean() * 100
overall_modeling = (starters.loc[starters["year"] >= TRAIN_START_YEAR, "positionOrder"] <= 3).mean() * 100

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(season_rates.index, season_rates["podium_rate"], color="#2a78d6", linewidth=2,
        marker="o", markersize=4)
ax.axhline(overall_starters, color="#52514e", linestyle="--", linewidth=1)
ax.text(1975, 4, f"dashed line = all seasons: {overall_starters:.1f}%",
        color="#52514e", fontsize=9)
ax.axvline(TRAIN_START_YEAR, color="#52514e", linestyle=":", linewidth=1)
ax.text(TRAIN_START_YEAR + 0.5, 22, f"TRAIN_START_YEAR = {TRAIN_START_YEAR}",
        color="#52514e", fontsize=9)
ax.set_title("Class balance: share of starters who finished on the podium, per season")
ax.set_xlabel("Season")
ax.set_ylabel("Podium (% of starters)")
ax.set_ylim(0, 25)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

print(f"Podium share: all raw rows {overall_all:.1f}% | starters {overall_starters:.1f}% | "
      f"starters from {TRAIN_START_YEAR} on {overall_modeling:.1f}%")

**What the chart shows:** only **13.6% of starters** finish on the podium (12.7% of all raw rows, because non-starters count as "no"); in our modeling seasons (2003+) it is **14.4%**. So about **6 in 7 rows are "no podium"**: a model that always says "no" would be about 86% accurate and completely useless. That is why accuracy is not used and **PR-AUC is our main metric** (Decision 3); it measures how well the rare podium finishers are found.

The per-season share is mostly mechanical: about 3 ÷ the number of starters (see 4.4), so it is lowest in the crowded 1980s fields (about 11–12%) and highest when fields were small (1969: 18.9%).

**Data note:** in 18 races of 1950–1960, more than 3 drivers count as "podium": when two drivers shared a car, both are listed with the same `positionOrder` (e.g. 1950 Italian GP: Ascari and Serafini, both 2nd in car #48). This does not affect the model (2003+), but EDA and Q1/Q2 on early seasons include these shared podiums.

## 5. Cleaning
**Goal:** Turn the raw tables into `clean`: real missing values (NaN), correct types, and the agreed rows removed (Decisions 6 → 8 → 5), with a visible before/after for every step.

**Uses:** raw, dtype_summary (3.1), sentinels (3.1), fk_checks (3.4), grain_key (3.6), OUTPUT_DIR

**Creates:** clean, time_to_seconds(), dtype_changes, cleaning_log, clean_data/clean.pkl

From here on, every section uses **`clean`**, never `raw` (Decision 13). Cleaning does **not** fill missing values (that happens in the model Pipeline, on training data only) and does **not** drop rows with missing values; the only rows removed are those of Decisions 6, 8 and 5.

### 5.1 Copies
`clean` starts as an independent copy of every raw table. We also save a fingerprint of `raw`, so 5.5 can prove that cleaning never changed the raw data.

In [ ]:
# clean = independent copies of every raw table; raw itself is never changed
clean = {name: df.copy() for name, df in raw.items()}

# Fingerprint of raw, so we can prove at the end (5.5) that raw was not modified
raw_fingerprint = {name: pd.util.hash_pandas_object(df, index=True).sum()
                   for name, df in raw.items()}
print(f"clean: {len(clean)} tables copied from raw ({sum(len(df) for df in clean.values()):,} rows in total)")

### 5.2 Missing markers → NaN (Decision 13)
In `raw` we kept the placeholders visible: `\N` everywhere, and empty cells in `qualifying.q2`/`q3`. Now both become real missing values (NaN), so pandas and the models treat them as missing. The table checks that every marker became exactly one NaN and none are left.

In [ ]:
# Turn both missing markers ("\N" and empty strings) into real NaN, in every table.
# sentinels = ["\\N", ""] was defined in 3.1.
missing_rows = []
for name, df in clean.items():
    markers_before = int(df.isin(sentinels).sum().sum())
    nan_before = int(df.isna().sum().sum())
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].where(~df[col].isin(sentinels))   # marker -> NaN, everything else unchanged
    missing_rows.append({"table": name,
                         "markers_before": markers_before,
                         "nan_before": nan_before,
                         "nan_after": int(df.isna().sum().sum()),
                         "markers_after": int(df.isin(sentinels).sum().sum())})

missing_check = pd.DataFrame(missing_rows).sort_values("markers_before", ascending=False).reset_index(drop=True)

# Every marker became exactly one NaN, and none are left
assert (missing_check["nan_before"] == 0).all()
assert (missing_check["markers_before"] == missing_check["nan_after"]).all()
assert missing_check["markers_after"].sum() == 0
print(f"{missing_check['markers_before'].sum():,} missing markers before -> "
      f"{missing_check['nan_after'].sum():,} NaN after; markers left: {missing_check['markers_after'].sum()}")
missing_check

**What the table shows:** all 160,144 missing markers (160,076 `\N` + 68 empty cells) are now NaN — exactly one NaN per marker, none left, and there were no NaN before (Decision 13 kept everything as text). Both kinds of marker in `qualifying` (11,600 `\N` + 68 empty) are now the same NaN. Missing values are **not filled**: e.g. a missing `q3` still means "did not reach Q3", and filling belongs in the model Pipeline (training data only).

### 5.3 Correct types (semantic typing)
Columns that hold numbers, dates or times but were stored as text are converted **in place**, keeping their names (Decision 14). The list comes from `dtype_summary` (3.1):
- **numbers** → `pd.to_numeric(..., errors="raise")`: an unexpected value stops the notebook instead of silently becoming NaN;
- **dates** → `pd.to_datetime(..., format="%Y-%m-%d")`;
- **times** → seconds with our helper `time_to_seconds()` (`"1:26.572"` → 86.572; also handles `h:mm:ss.sss` and plain seconds, the mixed formats found in 3.1).

In [ ]:
def time_to_seconds(value):
    """Convert a time string to seconds: '1:26.572' -> 86.572, '2:05:05.152' -> 7505.152,
    '26.898' -> 26.898. Missing values stay NaN."""
    if pd.isna(value):
        return np.nan
    seconds = 0.0
    for part in str(value).split(":"):
        seconds = seconds * 60 + float(part)
    return seconds

# Which columns to convert, taken from dtype_summary (3.1)
flags = dtype_summary[dtype_summary["dtype"] == "object"]
numeric_cols  = list(flags.loc[flags["looks_numeric"], ["table", "column"]].itertuples(index=False))
date_cols     = list(flags.loc[flags["looks_date"], ["table", "column"]].itertuples(index=False))
laptime_cols  = list(flags.loc[flags["looks_laptime"], ["table", "column"]].itertuples(index=False))
# Two mixed-format columns (3.1) that time_to_seconds() also handles
laptime_cols += [("lap_times", "time"), ("pit_stops", "duration")]

dtype_rows = []
def convert(table, col, kind, func):
    before = str(clean[table][col].dtype)
    clean[table][col] = func(clean[table][col])
    dtype_rows.append({"table": table, "column": col, "kind": kind,
                       "dtype_before": before, "dtype_after": str(clean[table][col].dtype)})

for t, c in numeric_cols:
    convert(t, c, "number", lambda s: pd.to_numeric(s, errors="raise"))
for t, c in date_cols:
    convert(t, c, "date", lambda s: pd.to_datetime(s, format="%Y-%m-%d"))
for t, c in laptime_cols:
    convert(t, c, "time -> seconds", lambda s: s.map(time_to_seconds))

dtype_changes = pd.DataFrame(dtype_rows)
print(f"{len(dtype_changes)} columns converted: "
      + ", ".join(f"{n} {k}" for k, n in dtype_changes["kind"].value_counts().items()))
dtype_changes

In [ ]:
# Check the time conversion against the milliseconds column that sits next to it
for table, col in [("lap_times", "time"), ("pit_stops", "duration")]:
    df = clean[table]
    diff = (df[col] - df["milliseconds"] / 1000).abs()
    assert diff.max() < 0.0005, f"{table}.{col}: conversion differs from milliseconds"
    print(f"{table}.{col}: {len(df):,} rows, converted seconds = milliseconds / 1000 on every row "
          f"(largest difference {diff.max():.6f} s)")

# Spot-check a few known values
assert time_to_seconds("1:26.572") == 86.572
assert abs(time_to_seconds("2:05:05.152") - 7505.152) < 1e-9
assert time_to_seconds("26.898") == 26.898
assert np.isnan(time_to_seconds(np.nan))
print("time_to_seconds() spot checks passed")

**What changed:** 24 columns now have their real type (`dtype_changes`): 10 numbers, 7 dates, and 7 times converted to seconds. Numbers with missing values become `float64` (NaN is a float). The conversion is proven correct: for all 589,081 laps and 11,371 pit stops, our seconds equal the dataset's own `milliseconds / 1000` exactly — including the 43 laps over an hour and the 517 pit stops written as `m:ss.sss`.

**Deliberately not converted** (Decision 14): `results.time` / `sprint_results.time` (total time for the winner, gap for everyone else → use `milliseconds`); the 7 clock times (`races.time`, the 5 session times, `pit_stops.time`: times of day, not used); `positionText` (codes like `R`, `F`).

**For later sections:** qualifying times are now in seconds (median about 87 s). One `q1` value is about 1,000 s, a clear outlier (a lap that was not a real flying lap); it is left as is here and should be handled where qualifying features are built (Section 8).

### 5.4 Apply the agreed decisions (rows removed)
Three decisions remove rows. They are applied in a fixed order (6 → 8 → 5) so that Decision 5 only removes real shared drives. Every step is logged in `cleaning_log`.

**Decision 6 — Indianapolis 500:** drop all 11 Indy 500 races (1950–1960) from **every** table that has a `raceId` (`races`, `results`, standings, `constructor_results`).

**Why:** it was effectively a separate series: only 0.6% of its drivers raced any other Grand Prix that season, and 99.8% were American, which would distort the home-advantage question (Q2). It also adds 21 of the 85 duplicate pairs (3.6). Dropping it from every race-level table keeps `clean` consistent (no race without results).

**Evidence:** driver overlap 0.6%, American drivers 99.8% (`results` + `races` + `drivers`); duplicates in 3.6.

In [ ]:
# Every row-removal step is logged here
cleaning_log = []
def log_step(step, table, before, after):
    cleaning_log.append({"step": step, "table": table, "rows_before": before,
                         "rows_removed": before - after, "rows_after": after})

# Decision 6: drop the Indianapolis 500 from every table that has a raceId
indy_race_ids = clean["races"].loc[clean["races"]["name"] == "Indianapolis 500", "raceId"]
assert len(indy_race_ids) == 11

for name, df in clean.items():
    if "raceId" in df.columns:
        before = len(df)
        clean[name] = df[~df["raceId"].isin(indy_race_ids)].reset_index(drop=True)
        if len(clean[name]) < before:
            log_step("Decision 6: drop Indy 500", name, before, len(clean[name]))

pd.DataFrame(cleaning_log)

**Decision 8 — Non-starters:** drop drivers who never started the race: `positionText` **F** (failed to qualify / prequalify) or **W** (withdrew). Keep the other `grid = 0` rows: they are pit-lane starts.

**Why:** our unit is a driver *in* the race at the moment it starts. Non-starters can never finish on the podium; keeping them would add easy "no" rows and teach the model "grid 0 = no podium" from drivers who never raced. **Not leakage:** `positionText` only decides *which rows we predict for*, it is never a feature, and whether a driver qualified or withdrew is known before the start.

**Evidence:** 3.2 (grid = 0: F 1,348, W 203, finished 60…), chart 4.4 (1989: 38.8 entries vs 26.1 starters per race), charts 4.6 and 4.7 (grid = 0 has a 0% podium rate).

In [ ]:
# Decision 8: drop drivers who never started (F = failed to qualify, W = withdrew)
before = len(clean["results"])
clean["results"] = clean["results"][~clean["results"]["positionText"].isin(["F", "W"])].reset_index(drop=True)
log_step("Decision 8: drop non-starters (F/W)", "results", before, len(clean["results"]))

remaining_grid0 = int((clean["results"]["grid"] == 0).sum())
print(f"Removed {before - len(clean['results']):,} non-starters; "
      f"{remaining_grid0} rows with grid = 0 remain (pit-lane starters, kept)")

**Decision 5 — Shared drives:** drop every row of a (raceId, driverId) pair that appears more than once.

**Why:** the copies are different cars driven by the same driver in one race, and no pre-race information tells which car is "his": in 82 of 85 pairs both cars had a real grid slot. "Keep the first row" is not neutral, because `resultId` follows the finishing order in 1,074 of 1,125 races, so it would choose by race result. Dropping uses no race results, costs 0.5% of rows, and affects no modeling season (all pairs are before 2003).

**Evidence:** 3.6 (`results_dups`, duplicates-per-season chart, "what tells the copies apart").

In [ ]:
# Decision 5: drop every row of a (raceId, driverId) pair that appears more than once
before = len(clean["results"])
repeated = clean["results"].duplicated(subset=grain_key, keep=False)
n_pairs = clean["results"].loc[repeated, grain_key].drop_duplicates().shape[0]
clean["results"] = clean["results"][~repeated].reset_index(drop=True)
log_step("Decision 5: drop shared-drive pairs", "results", before, len(clean["results"]))
print(f"Removed {before - len(clean['results'])} rows ({n_pairs} repeated pairs)")

cleaning_log = pd.DataFrame(cleaning_log)
assert len(clean["results"]) == 24525
cleaning_log

**What was removed (`cleaning_log`):** `results` went from 26,759 to **24,525 rows** (−2,234, 8.3%): 405 Indy 500 rows, 1,704 non-starters, 125 shared-drive rows (61 pairs). Decision 6 also removed the 11 Indy races from `races` and their rows from the standings tables (669 + 26) and `constructor_results` (21). Because 6 and 8 ran first, Decision 5 only removed real shared drives (e.g. the 1978 Ertl case — two "did not qualify" rows — was already gone after Decision 8). 87 pit-lane starters (grid = 0) are kept. This table is the "what was removed" evidence for Section 6 (EDA-after).

### 5.5 Final checks
What every later section can rely on: one row per driver per race, no missing markers, every link between tables still valid (the same 23 links as 3.4), a target for every row, and `raw` untouched.

In [ ]:
# Final checks: what every later section can rely on
res = clean["results"]

# 1) One row per driver per race
assert not res.duplicated(subset=grain_key).any(), "duplicate (raceId, driverId) in results"

# 2) No missing markers left anywhere
for name, df in clean.items():
    assert not df.isin(sentinels).any().any(), f"missing markers left in {name}"

# 3) Every link still points to an existing row (same 23 links as 3.4)
for child, col, parent in fk_checks:
    values = clean[child][col].dropna()
    assert values.isin(clean[parent][col]).all(), f"broken link {child}.{col} -> {parent}"

# 4) The target can be built for every row
assert res["positionOrder"].notna().all()

# 5) raw was not changed by any cleaning step
assert all(pd.util.hash_pandas_object(raw[n], index=True).sum() == raw_fingerprint[n] for n in raw)

print(f"All final checks passed: {len(res):,} results rows, unique (raceId, driverId), "
      f"no missing markers, {len(fk_checks)} links valid, target complete, raw unchanged.")

### 5.6 Save `clean` for teammates (Decision 14)
The whole `clean` dict is saved as one file, `clean_data/clean.pkl`. A pickle keeps every type (dates, numbers, NaN) and loads back as the exact same dict.

**How teammates use it:** in your scratch notebook, add this notebook's output as data, then:
`clean = pd.read_pickle("<path to the added data>/clean_data/clean.pkl")`
Load it only on Kaggle (the file is tied to Kaggle's pandas 2.3.3).

In [ ]:
# Save the whole clean dict as one file for teammates (Decision 14)
os.makedirs(OUTPUT_DIR, exist_ok=True)
clean_path = os.path.join(OUTPUT_DIR, "clean.pkl")
pd.to_pickle(clean, clean_path)

# Reload test: the file must give back exactly the same tables
reloaded = pd.read_pickle(clean_path)
assert reloaded.keys() == clean.keys()
assert all(reloaded[name].equals(clean[name]) for name in clean)
print(f"Saved {clean_path} ({os.path.getsize(clean_path) / 1e6:.1f} MB, {len(clean)} tables); "
      f"reload test passed.")

## 6. EDA – After Cleaning

**Goal:** Show what cleaning changed. Rows, missing values and the distributions from Section 4 are computed again on `clean`, and every chart puts "before" (raw) next to "after" (clean).

**Uses:** `raw`, `clean`, `cleaning_log` (5.4), `missing_check` (5.2), `missing_summary` (3.2), `field_size` (4.4)

**Creates:** `rows_compare`, `miss_compare`, `cleaning_summary` (plus a few helper tables)

### 6.1 Rows before vs after
How many rows each table lost in Section 5. Only Decisions 6, 8 and 5 remove rows, so the table is checked against `cleaning_log` (5.4). The chart follows the `results` table step by step.

In [ ]:
# Rows per table: raw vs clean (only Decision 6, 8 and 5 remove rows)
rows_compare = pd.DataFrame({"rows_raw": {k: len(v) for k, v in raw.items()},
                             "rows_clean": {k: len(v) for k, v in clean.items()}})
rows_compare["rows_removed"] = rows_compare["rows_raw"] - rows_compare["rows_clean"]
rows_compare["pct_removed"] = (rows_compare["rows_removed"] / rows_compare["rows_raw"] * 100).round(1)
rows_compare = rows_compare.sort_values("rows_raw", ascending=False)

# The table must agree with cleaning_log (5.4): same removals, nothing else changed
log_removed = cleaning_log.groupby("table")["rows_removed"].sum()
changed = rows_compare.loc[rows_compare["rows_removed"] > 0, "rows_removed"]
assert changed.sort_index().equals(log_removed.sort_index()), "rows_compare and cleaning_log disagree"
display(rows_compare[rows_compare["rows_removed"] > 0])

# Chart: results rows after each cleaning step
res_log = cleaning_log[cleaning_log["table"] == "results"].reset_index(drop=True)
step_labels = ["raw results"] + [s.split(":")[0] for s in res_log["step"]]
step_rows = [res_log.loc[0, "rows_before"]] + list(res_log["rows_after"])
bar_colors = ["#2a78d6"] * (len(step_rows) - 1) + ["#eb6834"]

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.bar(step_labels, step_rows, color=bar_colors, width=0.6)
for i, v in enumerate(step_rows):
    ax.text(i, v + 150, f"{v:,}", ha="center", fontsize=9)
    if i > 0:
        ax.text(i, v / 2, f"−{step_rows[i - 1] - v:,}", ha="center", color="white", fontsize=10)
ax.set_title("Rows in results after each cleaning step")
ax.set_xlabel("Cleaning step (applied in this order)")
ax.set_ylabel("Rows in results")
ax.set_ylim(0, max(step_rows) * 1.12)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

**What the table and chart show:** 5 of the 14 tables lost rows; the other 9 are unchanged. `results` went from 26,759 to **24,525 rows** (−2,234, 8.3%): −405 Indianapolis 500 rows (Decision 6), −1,704 non-starters (Decision 8) and −125 shared-drive rows (Decision 5). Decision 6 also removed the 11 Indy races from `races` and their rows from `driver_standings` (−669), `constructor_standings` (−26) and `constructor_results` (−21). The assert in the cell guarantees that this table and `cleaning_log` agree, so nothing else was removed.

### 6.2 Missing values before vs after
In `raw` a missing value was the text `\N` or an empty cell; in `clean` it is a real NaN (5.2). We compare the share of missing values per column of `results`, the table whose missing values matter most, with the "before" picture in 4.5.

In [ ]:
# Missing values in results: before (raw markers) vs after (NaN in clean), % of rows per column
before_pct = (missing_summary[missing_summary["table"] == "results"]
              .set_index("column")["pct_missing"])
after_pct = (clean["results"].isna().mean() * 100).round(1)
after_pct = after_pct[after_pct > 0]
miss_compare = (pd.DataFrame({"before_pct": before_pct, "after_pct": after_pct})
                .fillna(0).sort_values("before_pct"))

fig, ax = plt.subplots(figsize=FIGSIZE)
pos = np.arange(len(miss_compare))
ax.barh(pos - 0.2, miss_compare["before_pct"], height=0.38, color="#2a78d6", label="Before: raw (\\N or empty)")
ax.barh(pos + 0.2, miss_compare["after_pct"], height=0.38, color="#eb6834", label="After: clean (NaN)")
ax.set_yticks(pos)
ax.set_yticklabels(miss_compare.index)
ax.set_title("Missing values per column in results, before vs after cleaning")
ax.set_xlabel("Missing (% of rows)")
ax.set_ylabel("Column of results")
ax.set_xlim(0, 100)
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

nan_total_clean = sum(int(df.isna().sum().sum()) for df in clean.values())
print(f"Markers in raw: {missing_check['markers_before'].sum():,} -> NaN right after 5.2: {missing_check['nan_after'].sum():,} "
      f"-> NaN in clean after the row removals: {nan_total_clean:,} (markers left: 0)")
miss_compare["change_pts"] = (miss_compare["after_pct"] - miss_compare["before_pct"]).round(1)
miss_compare

**What the chart shows:** nothing was filled in, so the pattern is the same as in 4.5. The share of missing values falls a little in every column that has any (`position` 40.9% → 36.7%, `time` and `milliseconds` 71.3% → 69.2%, the fastest-lap columns 69.2% → 66.4%, `rank` 68.2% → 65.5%). The reason is that the 1,704 removed non-starters are empty in these columns (100% for `position`, `time`, `milliseconds` and `fastestLap`), so removing them lowers the share. In total, 160,144 text markers became 160,144 NaN (5.2), and 144,954 NaN remain after the row removals; no marker is left.

The values that stay missing are real. `position` is empty for drivers who were not classified. The fastest-lap columns (`fastestLap`, `rank`, `fastestLapTime`, `fastestLapSpeed`) are **empty for every row before 2004** (the dataset records them from 2004; from 2004 on only 4% are missing), so their missing share is an era effect. `time` and `milliseconds` depend on how the race ended for the driver. All of these columns describe the race itself, so they are **never features** (leakage); they stay in `clean` only because the target and Q3 need related columns. `number` has 6 missing rows (0.0%).

### 6.3 Grid distribution before vs after
The same chart as 4.6, now with both versions side by side. `grid = 0` was the single biggest value in `raw` because it mixed non-starters and pit-lane starters (Decision 8).

In [ ]:
# Grid distribution before vs after cleaning (compare with 4.6)
grid_before = raw["results"]["grid"].value_counts().sort_index()
grid_after = clean["results"]["grid"].value_counts().sort_index()
grid_slots = np.arange(0, 27)
gb = grid_before.reindex(grid_slots, fill_value=0)
ga = grid_after.reindex(grid_slots, fill_value=0)

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.bar(grid_slots - 0.2, gb.values, width=0.4, color="#2a78d6", label="Before cleaning (raw)")
ax.bar(grid_slots + 0.2, ga.values, width=0.4, color="#eb6834", label="After cleaning (clean)")
ax.text(0.7, gb.loc[0], f"grid = 0: {gb.loc[0]:,} → {ga.loc[0]:,} rows", color="#52514e", fontsize=9, va="top")
ax.set_title("Results rows per starting grid slot, before vs after cleaning")
ax.set_xlabel("Grid slot (0 = no grid slot)")
ax.set_ylabel("Results rows")
ax.legend(frameon=False, loc="upper right")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

assert ga.loc[0] == 87   # only the pit-lane starters are left (Decision 8)
print(f"grid = 0: {gb.loc[0]:,} rows ({gb.loc[0] / grid_before.sum():.1%} of raw) -> {ga.loc[0]} rows "
      f"({ga.loc[0] / grid_after.sum():.2%} of clean). Slots 1-20: before {gb.loc[1:20].sum():,}, after {ga.loc[1:20].sum():,}.")

**What the chart shows:** the `grid = 0` spike shrinks from 1,638 rows (6.1% of `raw`) to **87 rows** (0.35% of `clean`): these are the pit-lane starters that Decision 8 keeps, and the assert in the cell checks the number. Slots 1–20 keep almost all of their rows (22,168 → 21,687), so the shape of the real grid is unchanged; the small drop comes mostly from the Indy 500 rows (Decision 6), the non-starters that still had a grid slot (Decision 8) and the shared-drive rows (Decision 5). The remaining `grid = 0` rows are the only ones for which Section 8 needs a separate flag.

### 6.4 Drivers per race before vs after
The same chart as 4.4. Before cleaning, entries (every row of `results`) exceeded starters by many drivers per race; after cleaning every row should be a driver who started. The cell also checks, race by race, that the only rows lost since the raw starters are those of Decision 5.

In [ ]:
# Drivers per race, per season: raw entries and raw starters (from 4.4) vs rows in clean
clean_rows = clean["results"].merge(clean["races"][["raceId", "year"]], on="raceId")
field_after = clean_rows.groupby(["year", "raceId"]).size().groupby("year").mean().round(1)

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(field_size.index, field_size["all_entries"], color="#2a78d6", linewidth=2, label="Before: all entries in raw results")
ax.plot(field_size.index, field_size["starters"], color="#52514e", linewidth=1.2, linestyle="--",
        label="Before: starters only (reference, from 4.4)")
ax.plot(field_after.index, field_after.values, color="#eb6834", linewidth=2, label="After: rows in clean results")
ax.set_title("Average number of drivers per race, per season, before vs after cleaning")
ax.set_xlabel("Season")
ax.set_ylabel("Drivers per race (average)")
ax.set_ylim(0, field_size["all_entries"].max() + 3)
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

# Race by race: raw starters (Indy 500 excluded, Decision 6) vs rows in clean. Only Decision 5 should explain the difference.
starter_rows = (raw["results"][~raw["results"]["positionText"].isin(["F", "W"])]
                .merge(clean["races"][["raceId", "year", "name"]], on="raceId"))   # clean["races"] has no Indy 500
race_check = (starter_rows.groupby(["raceId", "year", "name"]).size().rename("raw_starters").reset_index()
              .merge(clean["results"].groupby("raceId").size().rename("clean_rows").reset_index(), on="raceId"))
race_check["rows_lost"] = race_check["raw_starters"] - race_check["clean_rows"]
assert race_check["rows_lost"].min() >= 0 and race_check["rows_lost"].sum() == 125   # = Decision 5 (5.4)

gap_before = field_size["all_entries"] - field_size["starters"]
print(f"Before: entries exceed starters by up to {gap_before.max():.1f} drivers per race ({gap_before.idxmax()}). "
      f"After: every row is a starter; the only rows lost since are Decision 5's {race_check['rows_lost'].sum()} rows "
      f"in {(race_check['rows_lost'] > 0).sum()} races (1950-1964).")
print(f"Races in clean: {len(race_check):,}; fewest drivers in a race: {race_check['clean_rows'].min()}")
race_check.sort_values("rows_lost", ascending=False).head(5)

**What the chart shows:** before cleaning, entries exceeded starters by up to 12.7 drivers per race (1989: 38.8 entries vs 26.1 starters). After cleaning (orange) the line follows the starters, so the gap is gone. The orange line sits below the dashed starters line in the 1950s for two reasons: the 4.4 averages include the Indy 500 races (33–55 starters each, Decision 6), and Decision 5 removed shared-drive rows. The race-by-race check in the cell shows that, apart from Indy, the only rows lost are those 125 rows in 36 races (1950–1964).

**Side effect of Decision 5 worth knowing:** because it removes every row of a repeated pair, the **1955 Argentine Grand Prix drops from 35 starter rows to 12** (23 rows lost), as many drivers swapped cars in that race. The smallest field in `clean` is 6 drivers (2005 United States Grand Prix, where most teams withdrew after the formation lap). None of this reaches the model (seasons from 2003), but it affects early-season counts in Q1 and Q2.

### 6.5 Class balance before vs after
The same idea as 4.9: the share of rows that finish on the podium, per season, for all raw rows and for the clean rows.

In [ ]:
# Class balance before vs after: share of podium rows per season (compare with 4.9)
def podium_share_by_year(results_df, races_df):
    d = results_df.merge(races_df[["raceId", "year"]], on="raceId")
    return (d["positionOrder"] <= 3).groupby(d["year"]).mean() * 100

podium_before = podium_share_by_year(raw["results"], raw["races"])       # all raw rows (non-starters count as "no")
podium_after = podium_share_by_year(clean["results"], clean["races"])
share_raw_all = (raw["results"]["positionOrder"] <= 3).mean() * 100
share_clean_all = (clean["results"]["positionOrder"] <= 3).mean() * 100
clean_modeling = clean["results"].merge(clean["races"][["raceId", "year"]], on="raceId")
clean_modeling = clean_modeling[clean_modeling["year"] >= TRAIN_START_YEAR]
share_clean_modeling = (clean_modeling["positionOrder"] <= 3).mean() * 100

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(podium_before.index, podium_before.values, color="#2a78d6", linewidth=2, label="Before: all raw rows")
ax.plot(podium_after.index, podium_after.values, color="#eb6834", linewidth=2, label="After: clean rows (starters only)")
ax.axvline(TRAIN_START_YEAR, color="#52514e", linestyle=":", linewidth=1)
ax.text(TRAIN_START_YEAR + 0.5, 3, f"TRAIN_START_YEAR = {TRAIN_START_YEAR}", color="#52514e", fontsize=9)
ax.set_title("Share of rows that are podium finishes, per season, before vs after cleaning")
ax.set_xlabel("Season")
ax.set_ylabel("Podium (% of rows)")
ax.set_ylim(0, 25)
ax.legend(frameon=False, loc="upper right")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

assert len(clean_modeling) == 8910   # Decision 4: 2003-2024 = 8,910 driver-races
print(f"Podium share of rows: raw {share_raw_all:.1f}% -> clean {share_clean_all:.1f}% (all seasons); "
      f"clean {TRAIN_START_YEAR}+: {share_clean_modeling:.1f}% of {len(clean_modeling):,} rows. "
      f"Largest per-season change: {(podium_after - podium_before).abs().max():.1f} points ({(podium_after - podium_before).abs().idxmax()}).")

**What the chart shows:** the podium share of rows rises from **12.7%** (raw, where non-starters count as "no") to **13.6%** (clean, all seasons), which is exactly the "starters" figure of 4.9. For the modeling seasons (2003 and later) it is **14.4% of 8,910 rows**, the same as in Decision 4. The two lines differ most in the crowded 1980s fields (largest gap 3.7 points, 1989) and coincide from the late 1990s on, because there are almost no non-starters left to remove. The class balance that motivates PR-AUC as the main metric (Decision 3) is therefore unchanged by cleaning: about 6 in 7 rows are still "no podium".

### 6.6 What cleaning changed: summary
One table with the measures from Sections 3–5, before and after.

In [ ]:
# Summary: what cleaning changed (one row per measure)
rr = raw["results"]; cr = clean["results"]
def races_over_3_podium_rows(df):          # shared cars: two drivers with the same positionOrder
    return int(((df["positionOrder"] <= 3).groupby(df["raceId"]).sum() > 3).sum())

def n_repeated_pairs(df):
    return int(df.loc[df.duplicated(grain_key, keep=False), grain_key].drop_duplicates().shape[0])

cleaning_summary = pd.DataFrame([
    ("Rows in results",                         len(rr), len(cr)),
    ("Races",                                    len(raw["races"]), len(clean["races"])),
    ("Non-starters in results (F / W)",          int(rr["positionText"].isin(["F", "W"]).sum()), int(cr["positionText"].isin(["F", "W"]).sum())),
    ("Rows with grid = 0",                       int((rr["grid"] == 0).sum()), int((cr["grid"] == 0).sum())),
    ("Repeated (raceId, driverId) pairs",        n_repeated_pairs(rr), n_repeated_pairs(cr)),
    ("Missing markers as text (\\N or empty), all tables", int(missing_check["markers_before"].sum()), int(missing_check["markers_after"].sum())),
    ("Missing cells as NaN, all tables",         0, sum(int(d.isna().sum().sum()) for d in clean.values())),
    ("Races with more than 3 podium rows (shared cars)", races_over_3_podium_rows(rr), races_over_3_podium_rows(cr)),
    ("Podium share of results rows (%)",         round(share_raw_all, 1), round(share_clean_all, 1)),
], columns=["measure", "before (raw)", "after (clean)"], dtype=object)
cleaning_summary

**What the table shows:** the grain problems are solved (no non-starters, no repeated driver-race pairs, 87 pit-lane starters instead of 1,638 `grid = 0` rows) and no missing marker is left as text. It also shows a **correction to an earlier note**: the 18 races in which more than 3 rows have `positionOrder <= 3` (shared cars, 1950–1960) exist only in `raw`; in `clean` there are none, because Decisions 6, 5 and 8 removed them. So Q1 and Q2 are not inflated by shared-car podiums.

**What cleaning deliberately did not do:** it did not fill any missing value (that belongs in the Pipeline, fitted on train only) and it did not `dropna()` any row (that would remove every driver who did not finish). The after-race columns stay in `clean` because the target and Q3 need related information; leakage is prevented later by the feature list.

## 7. Data-Engineering Questions (Q1, Q2, Q3)
**Goal:** Answer the three questions of the brief with multi-hop joins, a chart and a short paragraph each on grain, missing data and the pre-race cutoff.
**Uses:** `clean`, `grain_key`, `TRAIN_START_YEAR`
**Creates:** `analysis` (joined table), `conv` (Q1), `nationality_to_countries`, `home_table` (Q2), `status_class()`, `franchise_map`, `team_rates` (Q3)
**Constants defined here:** `HYBRID_START_YEAR = 2014`, `GROUND_EFFECT_START_YEAR = 2022` (the period boundaries named in the questions)
**Notes for the whole section:** Q1–Q3 are analyses of what happened, not features. `grid`, driver nationality and circuit country are known before the race and can be reused by Section 8; the status of a race is known only after it and is never a feature. All wording is "linked to", not "caused by".

### 7.1 One joined table for Q1–Q3
`results` is the base (one row per driver per race, Decisions 5, 6, 8) and is joined to `races`, `circuits`, `drivers`, `constructors` and `status`. Each join is declared many-to-one, so it cannot multiply rows, and the cell asserts that the row count and the grain did not change. We work on a copy; `clean` is not touched.

In [ ]:
# One joined table for Q1-Q3 (a copy; clean is not changed). Grain: one row per (raceId, driverId) = clean["results"].
HYBRID_START_YEAR = 2014           # Q1 and Q3: start of the hybrid era
GROUND_EFFECT_START_YEAR = 2022    # Q3: start of the ground-effect era
LAST_SEASON = int(clean["races"]["year"].max())

q_races = clean["races"][["raceId", "year", "round", "circuitId", "name"]].rename(columns={"name": "race_name"})
q_circuits = clean["circuits"][["circuitId", "name", "location", "country"]].rename(columns={"name": "circuit_name", "location": "circuit_location", "country": "circuit_country"})
q_drivers = clean["drivers"][["driverId", "nationality"]]
q_teams = clean["constructors"][["constructorId", "name"]].rename(columns={"name": "constructor_name"})

analysis = (clean["results"][["raceId", "driverId", "constructorId", "grid", "positionOrder", "statusId"]]
            .merge(q_races, on="raceId", how="left", validate="many_to_one")
            .merge(q_circuits, on="circuitId", how="left", validate="many_to_one")
            .merge(q_drivers, on="driverId", how="left", validate="many_to_one")
            .merge(q_teams, on="constructorId", how="left", validate="many_to_one")
            .merge(clean["status"], on="statusId", how="left", validate="many_to_one"))
analysis["podium"] = analysis["positionOrder"] <= 3
analysis["nationality"] = analysis["nationality"].str.strip()   # one raw value has a trailing space ("Argentinian "); clean is not changed

# Joins must neither add nor lose rows, and nothing may be left unmatched
assert len(analysis) == len(clean["results"]) == 24525
assert not analysis.duplicated(grain_key).any()
assert analysis[["year", "circuit_country", "nationality", "constructor_name", "status", "grid", "positionOrder"]].notna().all().all()
print(f"analysis: {len(analysis):,} rows (one per driver per race), seasons {analysis['year'].min()}-{analysis['year'].max()}, "
      f"{analysis['raceId'].nunique():,} races, {analysis['circuitId'].nunique()} circuits")

**What the check shows:** 24,525 rows, still one per (raceId, driverId), across 1,114 races at 77 circuits (1950–2024), and none of the joined columns has a missing value (`grid` and `positionOrder` included). One raw value has a trailing space (`drivers.nationality` = "Argentinian "); we strip it in this copy only. It was the only cell with stray whitespace in all text columns of `clean`.

### Q1. Which circuits convert a front-row start into a podium most often, and does that ranking change after the 2014 hybrid era?
**TA guidance:** what matters in Q1 is the **grid starting position, not qualifying**; the brief's note is a reminder that the two differ because of penalties. So a **front-row start = `grid` of 1 or 2**, and the answer uses the grid only.
**Grain:** one row per driver per race; the rate of a circuit is podiums ÷ front-row starts (2 per race). **Missing data:** `grid` has no missing values and exists for every season (qualifying only from 1994). **Pre-race cutoff:** the grid is fixed before the lights go out; the podium is the outcome we measure, not an input.

### 7.2 Check: how often does the grid differ from qualifying?
One short check, on 1994 and later (the only seasons with qualifying). Pit-lane starts have no slot to compare, so they are left out of the first measure.

In [ ]:
# Q1 check: how often does the grid slot differ from the qualifying position? (qualifying exists from 1994 only)
quali_pos = clean["qualifying"][["raceId", "driverId", "position"]].rename(columns={"position": "quali_position"})
assert not quali_pos.duplicated(grain_key).any()
gq_all = (analysis[analysis["year"] >= 1994]
          .merge(quali_pos, on=grain_key, how="inner", validate="one_to_one"))
gq_all["era"] = pd.cut(gq_all["year"], [1993, 2003, HYBRID_START_YEAR - 1, LAST_SEASON],
                       labels=["1994-2003", f"2004-{HYBRID_START_YEAR - 1}", f"{HYBRID_START_YEAR}-{LAST_SEASON}"])

gq = gq_all[gq_all["grid"] > 0].copy()           # pit-lane starts (grid = 0) have no grid slot to compare
gq["grid_differs"] = gq["grid"] != gq["quali_position"]
qfr = gq_all[gq_all["quali_position"].isin([1, 2])].copy()   # drivers who qualified on the front row
qfr["lost_front_row"] = ~qfr["grid"].isin([1, 2])

grid_vs_quali = pd.concat([
    gq.groupby("era", observed=True).agg(rows=("grid", "size"), grid_differs_pct=("grid_differs", "mean")),
    qfr.groupby("era", observed=True).agg(quali_front_row=("grid", "size"), lost_front_row_pct=("lost_front_row", "mean"))], axis=1)
grid_vs_quali[["grid_differs_pct", "lost_front_row_pct"]] = (grid_vs_quali[["grid_differs_pct", "lost_front_row_pct"]] * 100).round(1)

by_year = gq.groupby("year")["grid_differs"].mean() * 100
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.bar(by_year.index, by_year.values, color="#2a78d6", width=0.8)
ax.set_title("How often does the starting grid slot differ from the qualifying position?")
ax.set_xlabel("Season")
ax.set_ylabel("Rows where grid ≠ qualifying position (%)")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

print(f"{len(gq):,} driver-races from 1994 on with a grid slot and a qualifying row: grid differs from the qualifying position in "
      f"{gq['grid_differs'].mean():.1%}. Of {len(qfr):,} drivers who qualified on the front row, {qfr['lost_front_row'].mean():.1%} did not start there.")
grid_vs_quali

**What the chart and table show:** from 1994 on, 10,346 driver-races have both a grid slot and a qualifying row, and the grid differs from the qualifying position in **23.3%** of them. It differs rarely before 2004 (1.5%) and often since (25.4% in 2004–2013, 30.4% from 2014). Of the 984 drivers who qualified on the front row, **3.8%** did not start there (0.0%, 2.5% and 6.2% in the three eras). The near-zero values before 2004 suggest that the dataset's grid column simply copies qualifying in those years, but we cannot confirm that from the data. This is why Q1 uses the grid.

**Decision 15 — Q1 sample rule:** only circuits with at least **10 front-row starts in each period** are ranked (before 2014, and 2014 onward); every rate gets a 95% Wilson interval and the change in ranking gets a bootstrap interval.
**Why:** a circuit has only 2 front-row starts per race, so after 2014 even a circuit with a race every year has at most 26. Rates from a handful of starts would produce rankings that mostly reflect luck, so the minimum removes the noisiest circuits and the intervals show how noisy the rest are. 10 is the highest minimum that still leaves a useful number of circuits.
**Evidence:** cell 7.3 prints the number of circuits kept for each minimum: 21 circuits at 5, **16 at 10**, 13 at 15, 6 at 20 and none at 30; only 24 of the 77 circuits have front-row starts in both periods.

### 7.3 Front-row conversion per circuit and period
For every circuit: front-row starts and podiums before and from 2014, the rate with its Wilson interval, and the rank of each circuit among the circuits kept.

In [ ]:
# Q1: front-row (grid 1 or 2) -> podium conversion per circuit, before vs after the hybrid era starts
MIN_FRONT_ROW_STARTS = 10     # per circuit AND per period; below this a rate is too noisy to rank

def wilson_ci(successes, n, z=1.96):
    """95% Wilson score interval for a share (works for small n and for rates near 0 or 1)."""
    k = np.asarray(successes, dtype=float); n = np.asarray(n, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = k / n
        centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
        half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half

def front_row_conversion(first_year=int(analysis["year"].min())):
    """One row per circuit: front-row starts and podiums before / from HYBRID_START_YEAR, rates, Wilson CI, ranks."""
    fr = analysis[analysis["grid"].isin([1, 2]) & (analysis["year"] >= first_year)].copy()
    fr["period"] = np.where(fr["year"] < HYBRID_START_YEAR, "before", "hybrid")
    t = (fr.groupby(["circuitId", "circuit_location", "circuit_name", "period"])
           .agg(starts=("podium", "size"), podiums=("podium", "sum")).unstack("period", fill_value=0))
    t.columns = [f"{a}_{b}" for a, b in t.columns]
    t = t.reset_index()
    for p in ["before", "hybrid"]:
        t[f"rate_{p}"] = t[f"podiums_{p}"] / t[f"starts_{p}"] * 100
        lo, hi = wilson_ci(t[f"podiums_{p}"], t[f"starts_{p}"])
        t[f"ci_low_{p}"], t[f"ci_high_{p}"] = lo * 100, hi * 100
    t["enough_data"] = (t["starts_before"] >= MIN_FRONT_ROW_STARTS) & (t["starts_hybrid"] >= MIN_FRONT_ROW_STARTS)
    ranked = t[t["enough_data"]].copy()
    ranked["rank_before"] = ranked["rate_before"].rank(ascending=False, method="average")
    ranked["rank_hybrid"] = ranked["rate_hybrid"].rank(ascending=False, method="average")
    return t, ranked

conv_all, conv = front_row_conversion()
conv = conv.sort_values("rate_hybrid", ascending=False).reset_index(drop=True)

# All circuits together, for context
fr_all = analysis[analysis["grid"].isin([1, 2])]
for label, part in [("before", fr_all[fr_all["year"] < HYBRID_START_YEAR]), ("hybrid", fr_all[fr_all["year"] >= HYBRID_START_YEAR])]:
    lo, hi = wilson_ci(part["podium"].sum(), len(part))
    print(f"All circuits, {label}: {part['podium'].sum():,} podiums from {len(part):,} front-row starts = {part['podium'].mean():.1%} (95% CI {lo:.1%}-{hi:.1%})")
n_total = len(conv_all)
n_both = int(((conv_all["starts_before"] > 0) & (conv_all["starts_hybrid"] > 0)).sum())
by_min = {k: int(((conv_all["starts_before"] >= k) & (conv_all["starts_hybrid"] >= k)).sum()) for k in [5, 10, 15, 20, 30]}
print("Circuits with at least k front-row starts in BOTH periods:", by_min)
print(f"{n_total} circuits have front-row starts; {n_both} in both periods; {len(conv)} with at least {MIN_FRONT_ROW_STARTS} in each period (kept).")

conv_table = conv[["circuit_location", "starts_before", "rate_before", "starts_hybrid", "rate_hybrid", "rank_before", "rank_hybrid"]].round(1)
conv_table

**What the table shows:** over all circuits, **56.7%** of front-row starts ended on the podium before 2014 (992 of 1,751) and **76.2%** from 2014 on (345 of 453); the intervals (54.3–59.0% and 72.0–79.9%) do not overlap. Of 77 circuits with front-row starts, 24 have them in both periods and **16** meet the minimum. From 2014 the highest rates are Monaco (89.5%, 19 starts), Suzuka (88.9%, 18) and Abu Dhabi (86.4%, 22); the lowest are Budapest (59.1%, 22), Mexico City (61.1%, 18) and Sakhir (66.7%, 24). Before 2014 the highest were Suzuka (70.0%, 50 starts) and Budapest (67.9%, 56), and the lowest Monza (48.8%, 123) and Monaco (49.6%, 119).

### 7.4 Chart: ranking before vs after 2014
Each circuit is one row: blue = before 2014, orange = 2014 and after, with 95% intervals. The numbers in brackets are the front-row starts in each period.

In [ ]:
# Q1 chart: front-row -> podium rate per circuit, before vs from 2014, with 95% Wilson intervals
plot_df = conv.sort_values("rate_hybrid").reset_index(drop=True)
ypos = np.arange(len(plot_df))
labels = [f"{r.circuit_location} ({r.starts_before} / {r.starts_hybrid})" for r in plot_df.itertuples()]

fig, ax = plt.subplots(figsize=FIGSIZE)
for i, r in plot_df.iterrows():
    ax.plot([r["rate_before"], r["rate_hybrid"]], [i, i], color="#52514e", linewidth=0.6, alpha=0.5, zorder=1)
ax.errorbar(plot_df["rate_before"], ypos, xerr=[plot_df["rate_before"] - plot_df["ci_low_before"], plot_df["ci_high_before"] - plot_df["rate_before"]],
            fmt="o", color="#2a78d6", markersize=5, elinewidth=1, capsize=2, label=f"Before {HYBRID_START_YEAR}", zorder=2)
ax.errorbar(plot_df["rate_hybrid"], ypos, xerr=[plot_df["rate_hybrid"] - plot_df["ci_low_hybrid"], plot_df["ci_high_hybrid"] - plot_df["rate_hybrid"]],
            fmt="o", color="#eb6834", markersize=5, elinewidth=1, capsize=2, label=f"{HYBRID_START_YEAR} and after", zorder=3)
ax.set_yticks(ypos)
ax.set_yticklabels(labels, fontsize=8)
ax.set_title("Front-row starts that ended on the podium, by circuit")
ax.set_xlabel("Podium rate of front-row starts (%), with 95% interval")
ax.set_ylabel("Circuit (front-row starts before / from 2014)")
ax.set_xlim(0, 100)
ax.legend(frameon=False, loc="upper left")
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

**What the chart shows:** the orange dots sit to the right of the blue ones for almost every circuit: the podium rate of a front-row start is higher from 2014 at 14 of 16 circuits, equal at Sakhir (66.7% in both) and lower only at Budapest. This fits the much lower retirement rates of modern cars (4.8). But the intervals are wide (several are 30 points or more because there are only 10–30 starts), and they separate the two periods at **only one circuit, Monaco** (49.6% → 89.5%).

### 7.5 Did the ranking change? Spearman correlation and robustness
The Spearman rank correlation compares the two rankings (1 = same order, 0 = unrelated). A bootstrap that redraws every circuit's podium count from its observed rate shows how much of the result is sampling noise. As a robustness check we repeat the comparison with only 2003–2013 as the "before" period.

In [ ]:
# Q1: did the ranking of circuits change? Spearman rank correlation + how much of it is just small-sample noise
rho = conv["rate_before"].corr(conv["rate_hybrid"], method="spearman")

# Parametric bootstrap: redraw every circuit's podium count from its observed rate, recompute rho (sampling noise only)
rng = np.random.default_rng(SEED)
boot = []
for _ in range(2000):
    rate_b = pd.Series(rng.binomial(conv["starts_before"], conv["podiums_before"] / conv["starts_before"]) / conv["starts_before"])
    rate_h = pd.Series(rng.binomial(conv["starts_hybrid"], conv["podiums_hybrid"] / conv["starts_hybrid"]) / conv["starts_hybrid"])
    boot.append(rate_b.corr(rate_h, method="spearman"))
rho_low, rho_high = np.nanpercentile(boot, [2.5, 97.5])
print(f"Spearman rank correlation of the circuit rankings (before vs {HYBRID_START_YEAR}+): {rho:.2f} "
      f"(95% bootstrap interval {rho_low:.2f} to {rho_high:.2f}, {len(conv)} circuits)")

# Biggest movers
conv["rank_change"] = conv["rank_before"] - conv["rank_hybrid"]       # positive = moved up
movers = conv.sort_values("rank_change", ascending=False)[["circuit_location", "rank_before", "rank_hybrid", "rank_change"]]
print("Biggest climbers:", ", ".join(f"{r.circuit_location} ({r.rank_before:.0f}→{r.rank_hybrid:.0f})" for r in movers.head(3).itertuples()))
print("Biggest fallers: ", ", ".join(f"{r.circuit_location} ({r.rank_before:.0f}→{r.rank_hybrid:.0f})" for r in movers.tail(3).iloc[::-1].itertuples()))

# Do the intervals separate the periods? (circuits whose before / after 95% intervals do not overlap)
sep = conv[(conv["ci_low_hybrid"] > conv["ci_high_before"]) | (conv["ci_low_before"] > conv["ci_high_hybrid"])]
print(f"{len(sep)} of {len(conv)} circuits have non-overlapping intervals: " + ", ".join(sep["circuit_location"]))

# Robustness: compare 2003-2013 (modern qualifying era) with 2014+ instead of 1950-2013
conv_2003 = front_row_conversion(first_year=TRAIN_START_YEAR)[1]
rho_2003 = conv_2003["rate_before"].corr(conv_2003["rate_hybrid"], method="spearman")
print(f"Robustness, 'before' = {TRAIN_START_YEAR}-{HYBRID_START_YEAR - 1} only: {len(conv_2003)} circuits with at least {MIN_FRONT_ROW_STARTS} starts per period, Spearman {rho_2003:.2f}")
movers.round(1)

**What the output shows:** the rank correlation between the two periods is **−0.08** (95% bootstrap interval −0.50 to 0.38, 16 circuits), so the order of circuits before 2014 says almost nothing about the order after. The biggest climbers are Monaco (15th → 1st), Spa (12th → 4th) and Montreal (13th → 8th); the biggest fallers are Budapest (2nd → 16th), Sakhir (tied 3rd–4th → 14th) and Mexico City (tied 6th–7th → 15th). With only 2003–2013 as the "before" period the correlation is **0.36** (14 circuits), still weak.

### Q1 answer
Over all circuits a front-row start became a podium far more often from 2014 (76.2%) than before (56.7%). **The ranking of circuits changes a lot** (rank correlation −0.08; 0.36 when comparing with 2003–2013 only): Monaco moves from near the bottom to the top and Budapest from near the top to the bottom. Because every circuit has only 10–26 starts after 2014, **only Monaco's jump is clearly more than chance**; for the other circuits we can say the ranking is unstable, not which circuit is truly best. Early seasons are not inflated by shared cars: `clean` has no race with more than 3 podium rows (6.6). **Limitations:** small samples per circuit; the "before" period mixes eras with very different reliability (the 2003–2013 check addresses this); circuits that appear in only one period are not ranked.

### Q2. Do drivers podium more often at a circuit in their home country than elsewhere, after controlling for starting position?
**Grain:** one row per driver per race, with the driver's nationality (`drivers`) and the circuit's country (`races` → `circuits`). **Missing data:** none in the joined columns (7.1); the only gap is that some nationalities have no country that ever hosted a race. **Pre-race cutoff:** nationality, circuit and grid are all known before the race, so the home flag could also serve as a feature in Section 8; the podium is the outcome. The Indianapolis 500 is already excluded (Decision 6), because 99.8% of its drivers were American and it would inflate the home effect. In `clean`, no race has more than 3 podium rows (6.6).

**Decision 12 — Nationality → country mapping:** an explicit dictionary `nationality_to_countries` (a list of home countries per nationality), with circuit countries spelled the same way (`"United States"` → `"USA"`). Dual nationalities count as home in either country; "Rhodesian" and "East German" have no home country.
**Why:** nationalities are demonyms ("British", "Dutch") and circuit countries are place names ("UK", "Netherlands"), so string matching cannot work. A dictionary can be checked completely: the cell asserts that every nationality in `drivers` is mapped and that the dictionary has no typo.
**Evidence:** cell 7.6: 43 nationalities, all mapped. The checks found three data issues: one nationality with a trailing space ("Argentinian "), "Argentine" and "Argentinian" as two spellings of one nationality, and the Las Vegas circuit using "United States" while all other US circuits use "USA" (without the fix, American drivers at Las Vegas would count as "away"). Two typo detectors in the cell print the host countries no nationality maps to (Azerbaijan, Bahrain, Korea, Morocco, Qatar, Saudi Arabia, Singapore, Turkey, UAE: they never had a home driver) and the mapped countries that never host a race (e.g. Finland, Poland, New Zealand).

### 7.6 Mapping and home flag

In [ ]:
# Decision 12: nationality -> home country, as an explicit dictionary (values = list of countries, as spelled in circuits.country)
country_fix = {"United States": "USA"}           # the Las Vegas circuit uses a different spelling than the other US circuits
analysis["circuit_country"] = analysis["circuit_country"].replace(country_fix)

nationality_to_countries = {
    "American": ["USA"], "Argentine": ["Argentina"], "Argentinian": ["Argentina"], "Australian": ["Australia"],
    "Austrian": ["Austria"], "Belgian": ["Belgium"], "Brazilian": ["Brazil"], "British": ["UK"],
    "Canadian": ["Canada"], "Chilean": ["Chile"], "Chinese": ["China"], "Colombian": ["Colombia"],
    "Czech": ["Czech Republic"], "Danish": ["Denmark"], "Dutch": ["Netherlands"], "Finnish": ["Finland"],
    "French": ["France"], "German": ["Germany"], "Hungarian": ["Hungary"], "Indian": ["India"],
    "Indonesian": ["Indonesia"], "Irish": ["Ireland"], "Italian": ["Italy"], "Japanese": ["Japan"],
    "Liechtensteiner": ["Liechtenstein"], "Malaysian": ["Malaysia"], "Mexican": ["Mexico"], "Monegasque": ["Monaco"],
    "New Zealander": ["New Zealand"], "Polish": ["Poland"], "Portuguese": ["Portugal"], "Russian": ["Russia"],
    "South African": ["South Africa"], "Spanish": ["Spain"], "Swedish": ["Sweden"], "Swiss": ["Switzerland"],
    "Thai": ["Thailand"], "Uruguayan": ["Uruguay"], "Venezuelan": ["Venezuela"],
    # Judgement calls: dual nationalities count as home in either country; no country exists today for these two
    "Argentine-Italian": ["Argentina", "Italy"], "American-Italian": ["USA", "Italy"],
    "Rhodesian": [], "East German": [],
}

# Checks: every nationality in the drivers table is mapped, and the dictionary has no typo or extra keys
all_nationalities = set(clean["drivers"]["nationality"].str.strip())   # same whitespace fix as in 7.1
unmapped = sorted(all_nationalities - set(nationality_to_countries))
extra_keys = sorted(set(nationality_to_countries) - all_nationalities)
assert not unmapped and not extra_keys, f"unmapped: {unmapped}, extra: {extra_keys}"

host_countries = set(analysis["circuit_country"])
mapped_countries = {c for v in nationality_to_countries.values() for c in v}
print(f"{len(all_nationalities)} nationalities, all mapped (unmapped: {unmapped}).")
print("Host countries that no driver nationality maps to (should be countries that never had a driver):", sorted(host_countries - mapped_countries))
print("Mapped countries that never host a race in clean (expected, e.g. Finland):", sorted(mapped_countries - host_countries))

# Home flag: the circuit's country is one of the driver's home countries
analysis["home_race"] = [c in nationality_to_countries[n] for n, c in zip(analysis["nationality"], analysis["circuit_country"])]
home_by_nat = (analysis.groupby("nationality")
               .agg(rows=("home_race", "size"), home_rows=("home_race", "sum"))
               .assign(has_home_circuit=lambda d: d["home_rows"] > 0).sort_values("home_rows", ascending=False))
print(f"home_race = True in {analysis['home_race'].sum():,} of {len(analysis):,} rows ({analysis['home_race'].mean():.1%}); "
      f"{int(home_by_nat['has_home_circuit'].sum())} of {len(home_by_nat)} nationalities ever race at home.")
home_by_nat.head(8)

**What the output shows:** all 43 nationalities are mapped; `home_race` is true in **1,914 of 24,525 rows (7.8%)**, and 27 of the 43 nationalities have raced at home at least once. Most home starts belong to British (441), Italian (396), French (205), German (199), Brazilian (114) and American (111) drivers.

**Decision 16 — Q2 design:** compare home and away drivers **inside four grid buckets** (1–3, 4–6, 7–10, 11 and higher); leave out the 87 pit-lane starts (`grid = 0`); summarise with one pooled number in which away drivers are re-weighted to the grid mix of the home drivers; the interval comes from a bootstrap that resamples whole races.
**Why:** home drivers may simply qualify better, so a raw comparison would mix two effects. Inside a bucket both groups start from similar places. Pit-lane starters have no slot and never reached the podium (4.7), so they would only add noise. Resampling races (not rows) respects that drivers in one race share the same conditions.
**Evidence:** cell 7.7 (rates per bucket), 7.9 (raw gap vs pooled estimate). The buckets are coarse (podium rate is 63.5% from pole but 46.3% from 3rd, cell 4.7), so some difference in starting position remains inside "grid 1–3"; this is a limitation, not removed by the method.

### 7.7 Home vs away inside grid buckets

In [ ]:
 # Q2: home vs away podium rate inside grid buckets (so the comparison is between drivers who started from similar slots)
GRID_BUCKETS = [0, 3, 6, 10, 100]
BUCKET_LABELS = ["grid 1-3", "grid 4-6", "grid 7-10", "grid 11+"]

q2 = analysis[analysis["grid"] > 0].copy()          # pit-lane starters (grid = 0, 87 rows, no podiums) have no slot to compare
q2["grid_bucket"] = pd.cut(q2["grid"], GRID_BUCKETS, labels=BUCKET_LABELS)
print(f"Rows used: {len(q2):,} (left out: {len(analysis) - len(q2)} pit-lane starts with grid = 0).")

def home_vs_away(df):
    t = (df.groupby(["grid_bucket", "home_race"], observed=False)["podium"].agg(starts="size", podiums="sum")
           .unstack("home_race", fill_value=0))
    t.columns = [f"{a}_{'home' if b else 'away'}" for a, b in t.columns]
    for side in ["home", "away"]:
        t[f"rate_{side}"] = t[f"podiums_{side}"] / t[f"starts_{side}"] * 100
        lo, hi = wilson_ci(t[f"podiums_{side}"], t[f"starts_{side}"])
        t[f"ci_low_{side}"], t[f"ci_high_{side}"] = lo * 100, hi * 100
    t["difference_pts"] = t["rate_home"] - t["rate_away"]
    return t

home_table = home_vs_away(q2)
home_table[["starts_home", "rate_home", "starts_away", "rate_away", "difference_pts"]].round(1)

**What the table shows:** 24,438 rows are used (the 87 pit-lane starts are left out). The podium rates of home and away drivers are very close in every bucket: **grid 1–3:** 57.1% at home (224 starts) vs 55.8% away (3,087), +1.3 points; **grid 4–6:** 26.9% (223) vs 26.1% (3,095), +0.8; **grid 7–10:** 7.2% (334) vs 9.3% (4,087), −2.1; **grid 11+:** 1.1% (1,129) vs 1.7% (12,259), −0.6.

### 7.8 Chart: home vs away podium rate per grid bucket

In [ ]:
# Q2 chart: podium rate of home vs away drivers, inside each grid bucket (95% Wilson intervals)
xpos = np.arange(len(home_table))
fig, ax = plt.subplots(figsize=FIGSIZE)
for offset, side, color, label in [(-0.2, "away", "#2a78d6", "Away (racing outside their home country)"),
                                   (0.2, "home", "#eb6834", "Home race")]:
    rate = home_table[f"rate_{side}"]
    err = [rate - home_table[f"ci_low_{side}"], home_table[f"ci_high_{side}"] - rate]
    ax.bar(xpos + offset, rate, width=0.38, color=color, label=label, yerr=err, capsize=3, error_kw={"elinewidth": 1})
ax.set_xticks(xpos)
ax.set_xticklabels([f"{b}\nhome n={int(r.starts_home):,}\naway n={int(r.starts_away):,}" for b, r in home_table.iterrows()])
ax.set_title("Podium rate at home vs away, among drivers who started from similar grid slots")
ax.set_xlabel("Starting grid bucket")
ax.set_ylabel("Finished on the podium (%), with 95% interval")
ax.legend(frameon=False, loc="upper right")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

**What the chart shows:** within each bucket the orange (home) and blue (away) bars are about the same height, and the intervals overlap everywhere. The home groups are small (223–334 starts in the three upper buckets), which is why their intervals are wide. The sample sizes are in the axis labels.

### 7.9 One pooled number, with a bootstrap interval
The pooled difference weights each bucket by the number of home starts, so it answers: "for the grid positions home drivers actually started from, how does their podium rate compare with away drivers from the same positions?" It is shown next to the raw gap, and again for 2003 onward only.

In [ ]:
# Q2: one pooled number. Compare home drivers with away drivers who started from the same grid buckets
# (direct standardisation: away rates are re-weighted to the grid mix of the home drivers).
def pooled_home_effect(df, n_boot=2000):
    race_ids = np.sort(df["raceId"].unique())
    def per_race(side, stat):
        sub = df[df["home_race"] == (side == "home")]
        t = sub.groupby(["raceId", "grid_bucket"], observed=False)["podium"].agg(stat).unstack("grid_bucket", fill_value=0)
        return t.reindex(race_ids, fill_value=0).to_numpy(dtype=float)
    Nh, Kh, Na, Ka = per_race("home", "size"), per_race("home", "sum"), per_race("away", "size"), per_race("away", "sum")

    def diff(idx):
        nh, kh, na, ka = Nh[idx].sum(0), Kh[idx].sum(0), Na[idx].sum(0), Ka[idx].sum(0)
        w = nh / nh.sum()
        with np.errstate(divide="ignore", invalid="ignore"):
            d = kh / nh - ka / na
        ok = np.isfinite(d)
        return float((w[ok] * d[ok]).sum() / w[ok].sum() * 100)

    rng = np.random.default_rng(SEED)
    point = diff(np.arange(len(race_ids)))
    boot = [diff(rng.integers(0, len(race_ids), len(race_ids))) for _ in range(n_boot)]   # resample whole races
    low, high = np.percentile(boot, [2.5, 97.5])
    raw_gap = (df.loc[df["home_race"], "podium"].mean() - df.loc[~df["home_race"], "podium"].mean()) * 100
    return point, low, high, raw_gap

for label, part in [("all seasons", q2), (f"{TRAIN_START_YEAR}-{LAST_SEASON} only", q2[q2["year"] >= TRAIN_START_YEAR])]:
    est, lo, hi, raw_gap = pooled_home_effect(part)
    print(f"{label}: {int(part['home_race'].sum()):,} home rows. Raw gap home - away: {raw_gap:+.1f} points. "
          f"Same grid buckets: {est:+.1f} points (95% interval {lo:+.1f} to {hi:+.1f}).")

**What the output shows:** with all seasons, home drivers finished on the podium **2.1 points less** often than away drivers in the raw comparison (1,910 home rows), and **0.5 points less** after controlling for the grid bucket (95% interval −1.7 to +0.8). For 2003–2024 only (363 home rows) the figures are −0.2 raw and −1.4 adjusted (interval −4.1 to +1.5).

### Q2 answer
**No: we find no sign that drivers podium more often at home once starting position is controlled.** The adjusted difference is about half a point (−0.5, interval −1.7 to +0.8), and the home and away intervals overlap in every bucket; the 2003–2024 interval also includes zero. Home drivers are not better off even before the control (−2.1 points raw), which may reflect weaker cars (not tested here). This is a statement about association in this data, not about cause. **Limitations:** the four buckets are coarse; the home groups are small (363 home rows from 2003 on); dual nationalities are treated as home in both countries; car quality is only captured through the grid slot.

### Q3. Which constructors have the highest mechanical-retirement rate in 2014–2021 vs 2022–2024?
**Grain:** one row per driver per race; the rate of a team is mechanical retirements ÷ **starts** (every row of `clean` is a start, Decision 8). **Missing data:** `status` has no missing value for these rows. A retirement is defined by `status`, not by `positionText`, because a classified driver can still have retired. **Pre-race cutoff:** the reason a race ended is known only after the race, so status is used here to answer the question and is **never a feature**. Drivers who were disqualified, excluded, ill or withdrew after starting count as starts but not as mechanical.

**Decision 11 — Status → mechanical mapping:** every status becomes `finished`, `mechanical`, `borderline` or `not mechanical`.
- **Fixed by the TA:** Puncture, Tyre, Tyre puncture, Fire and Safety belt are mechanical; Refuelling is not ("for simplicity, all mechanical except refuelling").
- **Our choice, mechanical:** the failure of a named car component (Engine, Brakes, Gearbox, Power Unit, Suspension, Electrical, Hydraulics, and the other 30 or so listed in cell 7.10), following the brief's definition.
- **Our choice, not mechanical:** crashes and driver errors (Collision, Accident, Collision damage, Spun off) and non-technical stops (Disqualified, Excluded, Illness, Withdrew), which the brief excludes.
- **Borderline (77 starts):** Retired (cause unknown), Wheel, Wheel nut, Rear wing, Front wing, Undertray, Damage, Debris, Out of fuel. The **main answer counts them as not mechanical** (cause unknown, or usually contact, or a pit-stop or fuel-planning error like Refuelling), and a **broad version** counts them as mechanical to test the choice.
**Why:** the brief defines a mechanical retirement as a component failure, not a crash, the driver or a regulation decision; a status that does not say which of these happened should not be counted as a component failure in the main answer.
**Evidence:** cell 7.10 lists every status with its class and counts (65 statuses, all mapped). Cell 7.14 shows that the answer hardly depends on the borderline group: the rankings of the strict and broad versions have a rank correlation of 0.91 (2014–2021) and 0.95 (2022–2024), and they share the same top-3 teams.

### 7.10 Starts in both periods and the status mapping

In [ ]:
# Q3 data: starters in the two periods + Decision 11 (status -> finished / mechanical / borderline / not mechanical)
PERIOD_1 = f"{HYBRID_START_YEAR}-{GROUND_EFFECT_START_YEAR - 1}"
PERIOD_2 = f"{GROUND_EFFECT_START_YEAR}-{LAST_SEASON}"
q3 = analysis[analysis["year"] >= HYBRID_START_YEAR].copy()      # every row is a starter (Decision 8)
q3["period"] = np.where(q3["year"] < GROUND_EFFECT_START_YEAR, PERIOD_1, PERIOD_2)

# Fixed by the TA (Decision 11): all mechanical except Refuelling
TA_MECHANICAL = ["Puncture", "Tyre", "Tyre puncture", "Fire", "Safety belt"]
TA_NOT_MECHANICAL = ["Refuelling"]
# Our choice: a failure of a named car component (brief's definition)
OUR_MECHANICAL = [
    "Engine", "Brakes", "Gearbox", "Power Unit", "Suspension", "Electrical", "Hydraulics", "Power loss", "Oil leak",
    "Overheating", "Mechanical", "Water pressure", "Turbo", "Fuel pressure", "Battery", "Water leak", "Electronics",
    "Exhaust", "ERS", "Transmission", "Radiator", "Throttle", "Clutch", "Oil pressure", "Steering", "Fuel pump",
    "Fuel leak", "Driveshaft", "Differential", "Cooling system", "Brake duct", "Drivetrain", "Spark plugs", "Water pump",
    "Technical", "Vibrations", "Seat"]
# Our choice: cause unclear or probably contact / procedure -> not counted as mechanical in the main answer, but counted in the "broad" check
BORDERLINE = ["Retired", "Wheel", "Wheel nut", "Rear wing", "Front wing", "Undertray", "Damage", "Debris", "Out of fuel"]
# Our choice: crash, driver, illness or a regulation decision (excluded by the brief's definition)
OUR_NOT_MECHANICAL = ["Collision", "Accident", "Collision damage", "Spun off", "Disqualified", "Excluded", "Illness", "Withdrew"]

def status_class(status):
    if status == "Finished" or (status.startswith("+") and status.endswith(("Lap", "Laps"))):   # "+1 Lap", "+2 Laps", ...
        return "finished"
    if status in TA_MECHANICAL + OUR_MECHANICAL:
        return "mechanical"
    if status in BORDERLINE:
        return "borderline"
    if status in TA_NOT_MECHANICAL + OUR_NOT_MECHANICAL:
        return "not mechanical"
    return "UNMAPPED"

q3["status_class"] = q3["status"].map(status_class)
assert (q3["status_class"] != "UNMAPPED").all(), sorted(q3.loc[q3["status_class"] == "UNMAPPED", "status"].unique())
assert set(TA_MECHANICAL + TA_NOT_MECHANICAL) <= set(clean["status"]["status"])      # the TA's statuses exist in the status table
q3["mech_strict"] = q3["status_class"] == "mechanical"
q3["mech_broad"] = q3["status_class"].isin(["mechanical", "borderline"])

status_table = (q3[q3["status_class"] != "finished"].groupby(["status_class", "status", "period"]).size()
                .unstack("period", fill_value=0).assign(total=lambda d: d.sum(axis=1))
                .sort_values(["status_class", "total"], ascending=[True, False]))
class_counts = q3.groupby(["status_class", "period"]).size().unstack(fill_value=0)
print(f"{len(q3):,} starts in {HYBRID_START_YEAR}-{LAST_SEASON} ({(q3['period'] == PERIOD_1).sum():,} in {PERIOD_1}, {(q3['period'] == PERIOD_2).sum():,} in {PERIOD_2}); "
      f"{q3['status'].nunique()} distinct statuses, all mapped.")
class_counts

**What the output shows:** there are **4,598 starts** in 2014–2024 (3,246 in 2014–2021 and 1,352 in 2022–2024), with 65 distinct statuses, all mapped. By class (2014–2021 / 2022–2024): finished 2,660 / 1,165; **mechanical 300 / 74**; borderline 59 / 18; not mechanical 227 / 95. Over all teams the mechanical-retirement rate fell from 9.2% to 5.5% of starts. Of the TA's statuses only Puncture (7) and Tyre (4) occur in these years; Tyre puncture, Fire, Safety belt and Refuelling have no starts here. The largest mechanical statuses are Engine (79), Brakes (48), Gearbox (40) and Power Unit (39); Brakes fell from 44 to 4 between the periods.

**Decision 7 — Team renames:** link the renamed teams into one lineage (`franchise_map`) for Q3 only, and keep the original `constructorId` everywhere else in `clean`. Lineages: Force India → Racing Point → Aston Martin; Toro Rosso → AlphaTauri → RB; Sauber → Alfa Romeo → Sauber; Lotus F1 → Renault → Alpine; Marussia → Manor Marussia. Caterham stays separate.
**Why:** each rename gets a new `constructorId`, so without linking a team's history is cut in two and cannot be compared across the two periods. Ergast has no lineage column, so the mapping rests on public team history (each lineage is the same team, with the same base, renamed or re-owned); this is a judgement, which is why the original ids are kept and the comparison is repeated without linking.
**Evidence:** cell 7.11 (lineage table: e.g. Aston Martin has 44 starts in 2014–2021 under its own id but 317 as a lineage; Alpine 44 vs 318). Without linking, only **6 teams** can be compared across the periods (cell 7.14); with linking, **10**. The check "season before vs season after each rename" (cell 7.11) shows no sign of a break: in all 9 renames inside 2014–2024 the 95% intervals of the mechanical rate overlap. This is weak evidence, as each season has only 31–48 starts; it shows we cannot see a break, not that none exists.

### 7.11 Lineages and the rename check

In [ ]:
# Decision 7: link team renames into one lineage (franchise), used in Q3 only. Original constructorIds stay in clean.
franchise_map = {
    10: "Aston Martin (ex Force India, Racing Point)", 211: "Aston Martin (ex Force India, Racing Point)", 117: "Aston Martin (ex Force India, Racing Point)",
    5: "RB (ex Toro Rosso, AlphaTauri)", 213: "RB (ex Toro Rosso, AlphaTauri)", 215: "RB (ex Toro Rosso, AlphaTauri)",
    15: "Sauber (incl. Alfa Romeo)", 51: "Sauber (incl. Alfa Romeo)",
    208: "Alpine (ex Lotus F1, Renault)", 4: "Alpine (ex Lotus F1, Renault)", 214: "Alpine (ex Lotus F1, Renault)",
    206: "Marussia / Manor", 209: "Marussia / Manor",
}
q3["franchise"] = q3["constructorId"].map(franchise_map).fillna(q3["constructor_name"])

lineage = (q3[q3["constructorId"].isin(franchise_map)]
           .groupby(["franchise", "constructorId", "constructor_name"])
           .agg(first_year=("year", "min"), last_year=("year", "max"),
                starts_period_1=("period", lambda s: (s == PERIOD_1).sum()), starts_period_2=("period", lambda s: (s == PERIOD_2).sum()))
           .reset_index().sort_values(["franchise", "first_year"]))
display(lineage)

# Evidence for the link: strict mechanical rate in the season before vs the season after each rename (same franchise, new constructorId)
season_team = (q3.groupby(["franchise", "year", "constructorId"])
               .agg(starts=("mech_strict", "size"), mech=("mech_strict", "sum")).reset_index())
main_id = season_team.sort_values("starts").groupby(["franchise", "year"]).tail(1).sort_values(["franchise", "year"])
events = []
for fr_name, g in main_id.groupby("franchise"):
    g = g.reset_index(drop=True)
    for i in range(1, len(g)):
        if g.loc[i, "constructorId"] != g.loc[i - 1, "constructorId"] and g.loc[i, "year"] == g.loc[i - 1, "year"] + 1:
            events.append({"franchise": fr_name, "rename": f"{g.loc[i - 1, 'year']} -> {g.loc[i, 'year']}",
                           "starts_before": g.loc[i - 1, "starts"], "mech_before": g.loc[i - 1, "mech"],
                           "starts_after": g.loc[i, "starts"], "mech_after": g.loc[i, "mech"]})
rename_check = pd.DataFrame(events)
for s in ["before", "after"]:
    rename_check[f"rate_{s}"] = rename_check[f"mech_{s}"] / rename_check[f"starts_{s}"] * 100
    lo, hi = wilson_ci(rename_check[f"mech_{s}"], rename_check[f"starts_{s}"])
    rename_check[f"lo_{s}"], rename_check[f"hi_{s}"] = lo * 100, hi * 100
rename_check["intervals_overlap"] = ~((rename_check["lo_after"] > rename_check["hi_before"]) | (rename_check["lo_before"] > rename_check["hi_after"]))
print(f"{len(rename_check)} renames inside {HYBRID_START_YEAR}-{LAST_SEASON}; the mechanical-retirement rate stays within sampling noise in "
      f"{int(rename_check['intervals_overlap'].sum())} of them (95% intervals overlap).")
rename_check[["franchise", "rename", "starts_before", "rate_before", "starts_after", "rate_after", "intervals_overlap"]].round(1)

**What the output shows:** 13 constructor ids fall into 5 lineages. The mechanical-retirement rate in the season before and after a rename stays within sampling noise in all 9 renames (for example Force India → Racing Point 2018 → 2019, 42 starts each, and Racing Point → Aston Martin 2020 → 2021).

### 7.12 Mechanical-retirement rate per team and period
Rate = mechanical retirements ÷ starts, per lineage, with 95% Wilson intervals. We show teams with at least 100 starts in each period; this minimum is not binding here, because every team shown has 134 or more starts in 2022–2024 and the two teams left out (Caterham, Marussia / Manor) have none.

In [ ]:
# Q3: mechanical-retirement rate = mechanical retirements / starts, per franchise and period
MIN_STARTS_Q3 = 100        # per period; below this a rate rests on fewer than about 50 race weekends of one team

def mech_rates(df, group_col, flag):
    t = (df.groupby([group_col, "period"]).agg(starts=(flag, "size"), mech=(flag, "sum"))
           .unstack("period", fill_value=0))
    t.columns = [f"{a}_{b}" for a, b in t.columns]
    for p in [PERIOD_1, PERIOD_2]:
        t[f"rate_{p}"] = t[f"mech_{p}"] / t[f"starts_{p}"] * 100
        lo, hi = wilson_ci(t[f"mech_{p}"], t[f"starts_{p}"])
        t[f"ci_low_{p}"], t[f"ci_high_{p}"] = lo * 100, hi * 100
    t["enough_data"] = (t[f"starts_{PERIOD_1}"] >= MIN_STARTS_Q3) & (t[f"starts_{PERIOD_2}"] >= MIN_STARTS_Q3)
    t["change_pts"] = t[f"rate_{PERIOD_2}"] - t[f"rate_{PERIOD_1}"]
    return t

team_all = mech_rates(q3, "franchise", "mech_strict")
team_rates = team_all[team_all["enough_data"]].sort_values(f"rate_{PERIOD_2}", ascending=False)

tot = q3.groupby("period")["mech_strict"].agg(["sum", "size"])
print("All teams together: " + ", ".join(f"{p}: {int(tot.loc[p, 'sum'])} of {int(tot.loc[p, 'size']):,} starts = {tot.loc[p, 'sum'] / tot.loc[p, 'size']:.1%}" for p in [PERIOD_1, PERIOD_2]))
print(f"{len(team_all)} franchises raced in {HYBRID_START_YEAR}-{LAST_SEASON}; {len(team_rates)} have at least {MIN_STARTS_Q3} starts in both periods (shown). "
      f"Left out: {', '.join(team_all.index[~team_all['enough_data']])}.")
team_table = team_rates[[f"starts_{PERIOD_1}", f"mech_{PERIOD_1}", f"rate_{PERIOD_1}", f"starts_{PERIOD_2}", f"mech_{PERIOD_2}", f"rate_{PERIOD_2}", "change_pts"]].round(1)
team_table

**What the table shows:** 12 lineages raced in 2014–2024, and 10 of them in both periods. **2014–2021:** the highest rates are RB / Toro Rosso (14.7%, 47 of 319 starts), Alpine / Renault (13.5%, 43 of 318) and McLaren (13.0%, 41 of 316); the lowest are Mercedes (4.1%, 13 of 320) and Ferrari (5.3%, 17 of 318). **2022–2024:** Alpine is highest (10.4%, 14 of 135), then Sauber / Alfa Romeo (8.8%, 12 of 136) and RB (5.9%, 8 of 136); Aston Martin, McLaren and Red Bull are lowest, each 3.7% (5 of 134–136 starts).

### 7.13 Chart: mechanical-retirement rate per team, 2014–2021 vs 2022–2024

In [ ]:
# Q3 chart: mechanical-retirement rate per franchise, 2014-2021 vs 2022-2024 (95% Wilson intervals)
plot_t = team_rates.copy()
xpos = np.arange(len(plot_t))
fig, ax = plt.subplots(figsize=FIGSIZE)
for offset, p, color in [(-0.2, PERIOD_1, "#2a78d6"), (0.2, PERIOD_2, "#eb6834")]:
    rate = plot_t[f"rate_{p}"]
    err = [rate - plot_t[f"ci_low_{p}"], plot_t[f"ci_high_{p}"] - rate]
    ax.bar(xpos + offset, rate, width=0.38, color=color, label=p, yerr=err, capsize=2, error_kw={"elinewidth": 1})
ax.set_xticks(xpos)
ax.set_xticklabels([n.split(" (")[0] for n in plot_t.index], rotation=30, ha="right", fontsize=9)
ax.set_title("Mechanical-retirement rate per team (retirements per 100 starts)")
ax.set_xlabel("Team (renamed teams linked, Decision 7), sorted by 2022-2024 rate")
ax.set_ylabel("Mechanical retirements (% of starts), with 95% interval")
ax.legend(title="Period", frameon=False, loc="upper right")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.show()

**What the chart shows:** 8 of 10 teams have a lower rate in 2022–2024 (the exceptions are Sauber, +0.7 points, and Mercedes, +0.3, both within noise). The biggest drops are McLaren (13.0% → 3.7%), RB (14.7% → 5.9%) and Red Bull (11.3% → 3.7%). Most intervals overlap, so single-team differences should be read with care.

### 7.14 Do our choices change the answer? (Decisions 7 and 11)
The same ranking with (1) the main strict mapping, (2) the broad mapping that also counts the borderline statuses, and (3) the strict mapping without linking renamed teams.

In [ ]:
# Q3 checks: does the answer depend on our choices? (Decision 11: strict vs broad status mapping; Decision 7: linked vs separate names)
def top_three(t, p):
    return ", ".join(f"{n.split(' (')[0]} {r:.1f}%" for n, r in t[f"rate_{p}"].sort_values(ascending=False).head(3).items())

broad = mech_rates(q3, "franchise", "mech_broad").loc[team_rates.index]
separate_all = mech_rates(q3, "constructor_name", "mech_strict")
separate = separate_all[separate_all["enough_data"]]

check_rows = []
for label, t in [("Main: strict mapping, linked names", team_rates), ("Broad mapping (adds borderline statuses), linked names", broad),
                 ("Strict mapping, separate names (no linking)", separate)]:
    row = {"version": label, "teams_compared": len(t)}
    for p in [PERIOD_1, PERIOD_2]:
        row[f"top 3 in {p}"] = top_three(t, p)
    check_rows.append(row)
robustness = pd.DataFrame(check_rows)
for r in robustness.itertuples(index=False):
    print(f"{r.version}  ({r.teams_compared} teams)\n   {PERIOD_1}: {r[2]}\n   {PERIOD_2}: {r[3]}")

# Rank agreement of main vs broad, per period (same teams)
for p in [PERIOD_1, PERIOD_2]:
    rho_b = team_rates[f"rate_{p}"].corr(broad[f"rate_{p}"], method="spearman")
    print(f"Spearman rank correlation, strict vs broad, {p}: {rho_b:.2f}")
n_down = int((team_rates["change_pts"] < 0).sum())
separated = team_rates[(team_rates[f"ci_high_{PERIOD_2}"] < team_rates[f"ci_low_{PERIOD_1}"]) | (team_rates[f"ci_low_{PERIOD_2}"] > team_rates[f"ci_high_{PERIOD_1}"])]
print(f"{n_down} of {len(team_rates)} teams have a lower mechanical-retirement rate in {PERIOD_2}; "
      f"the 95% intervals of the two periods are apart for {len(separated)}: {', '.join(n.split(' (')[0] for n in separated.index)}.")
robustness

**What the output shows:** the strict and broad versions rank the teams almost identically (Spearman 0.91 and 0.95) and name the same top three in both periods (2014–2021: RB, Alpine, McLaren in the strict version and Alpine, RB, McLaren in the broad one; 2022–2024: Alpine, Sauber, RB in both). Without linking renamed teams, only 6 teams can be compared; RB, Alpine, Sauber and Aston Martin cannot, so the highest-rate teams would not even appear. The 95% intervals of the two periods are apart for only two teams: McLaren (clearly) and RB (barely).

### Q3 answer
**Highest mechanical-retirement rate, 2014–2021:** RB / Toro Rosso (14.7% of starts), Alpine / Renault (13.5%) and McLaren (13.0%). **2022–2024:** Alpine (10.4%), Sauber / Alfa Romeo (8.8%) and RB (5.9%). Overall, mechanical retirements fell from 9.2% to 5.5% of starts, and 8 of 10 teams improved, most of all McLaren (−9.3 points), RB (−8.9) and Red Bull (−7.6). Alpine is the only team in the top two in both periods. The answer does not depend on how the borderline statuses are classified (Decision 11). **Limitations:** 2022–2024 has only 3 seasons, so each team has about 135 starts and most intervals overlap; the TA's rule counts tyre, puncture, fire and safety-belt failures as mechanical "for simplicity"; renamed teams are linked by our own judgement (Decision 7); the data have no engine-supplier column, so a team's rate also reflects the reliability of the engine it used, which we cannot separate.

## 8. Feature Engineering
**Goal:** build the one modeling table `model_df`: one row per (raceId, driverId) from `TRAIN_START_YEAR` (2003, Decision 4), the target `podium`, and only features that are known **after qualifying and before the race starts**.

**Uses:** `clean` (never `raw`), `grain_key`, `analysis["home_race"]` (7.6, Decision 12)

**Creates:** `hist`, `model_df`, `FORM_WINDOW`, `FEATURE_GROUPS`, `FEATURES`, `feature_notes`

**Feature groups (prefix → group):** `grid_` grid · `quali_` qualifying · `drv_` driver form · `con_` constructor form · `stand_` standings · `ctx_` context.

**Rules followed here:** missing values stay NaN (filling happens in the Pipeline, fitted on train only, Section 10); no `dropna()`; form features use `.shift(1)`; standings come from the **previous round**. Charts and trials in this section use only seasons up to `VAL_END_YEAR` (train + validation, Decision 10), so the test seasons are never looked at.

### 8.1 Base table and target
`hist` holds **every season** in `clean` (sorted by race date). Form and standings features are computed on it, so the first 2003 races already have history from 2002 and earlier. Each feature only looks at **earlier** races, so using older seasons for history is not leakage. `model_df` then keeps the seasons from `TRAIN_START_YEAR` only.

In [ ]:
# 8.1 History table (all seasons) + base modeling table (from TRAIN_START_YEAR) with the target
race_order = (clean["races"][["raceId", "year", "round", "date", "circuitId"]]
              .sort_values(["year", "round"]).reset_index(drop=True))

hist = (clean["results"][["raceId", "driverId", "constructorId", "grid", "positionOrder", "positionText", "points"]]
        .merge(race_order, on="raceId", how="left", validate="many_to_one")
        .sort_values(["date", "raceId", "driverId"]).reset_index(drop=True))
hist["podium"] = (hist["positionOrder"] <= 3).astype(int)
hist["dnf"] = (~hist["positionText"].astype(str).str.isdigit()).astype(int)   # not classified: R, D, E, N, ...

model_keys = ["raceId", "driverId", "constructorId", "year", "round", "date"]
model_df = hist.loc[hist["year"] >= TRAIN_START_YEAR, model_keys + ["podium"]].reset_index(drop=True)
model_df = model_df.rename(columns={"podium": TARGET})

# Fingerprint of clean at the start of Section 8: 9.5 proves that Sections 8-9 did not change clean
clean_fingerprint_8 = {name: pd.util.hash_pandas_object(df, index=True).sum() for name, df in clean.items()}

def add_features(df, new, cols):
    """Merge feature columns (keyed on grain_key) into model_df; the number of rows must not change."""
    out = df.merge(new[grain_key + cols], on=grain_key, how="left", validate="one_to_one")
    assert len(out) == len(df)
    return out

assert len(hist) == len(clean["results"]) and not hist.duplicated(grain_key).any()
assert not model_df.duplicated(grain_key).any()
print(f"hist: {len(hist):,} rows ({hist['year'].min()}-{hist['year'].max()}), used only for history features")
print(f"model_df: {len(model_df):,} rows, {model_df['raceId'].nunique()} races, seasons {model_df['year'].min()}-{model_df['year'].max()}, "
      f"podium rate {model_df[TARGET].mean():.1%}")

**What the output shows:** `model_df` starts with **8,910 rows in 428 races (2003–2024)** and a podium rate of **14.4%**, the same numbers as Decision 4 and cell 6.5, so nothing was lost or added. `hist` keeps all 24,525 rows of `clean["results"]` for the history features only.

### 8.2 Grid features
The grid slot is the strongest single signal (cell 4.7: 63.5% podium from pole, under 6% from 10th). `grid = 0` means a **pit-lane start** (87 rows kept by Decision 8): such a car starts **behind** everybody, so the raw value 0 would wrongly sit "in front of" pole.

In [ ]:
# 8.2 Grid features (grid is set before the start: qualifying + penalties)
grid_feats = hist[grain_key + ["grid"]].copy()
last_slot = grid_feats.groupby("raceId")["grid"].transform("max")
grid_feats["grid_pitlane_start"] = (grid_feats["grid"] == 0).astype(int)
grid_feats["grid_position"] = grid_feats["grid"].where(grid_feats["grid"] > 0, last_slot + 1)      # pit lane = behind the last grid slot (Decision 17)
grid_feats["grid_front_row"] = grid_feats["grid"].between(1, 2).astype(int)

grid_cols = ["grid_position", "grid_front_row", "grid_pitlane_start"]
model_df = add_features(model_df, grid_feats, grid_cols)

pit = model_df[model_df["grid_pitlane_start"] == 1]
print(f"Pit-lane starts in model_df: {len(pit)} rows, podiums: {pit[TARGET].sum()}; "
      f"their grid_position = last slot + 1 (range {pit['grid_position'].min()}-{pit['grid_position'].max()})")
print(f"Front-row starts: {model_df['grid_front_row'].sum():,} rows; missing grid values: {model_df['grid_position'].isna().sum()}")

**Decision 17 — Pit-lane starters:** `grid_position` = **last grid slot of that race + 1** for the pit-lane starters (`grid = 0`), plus a flag `grid_pitlane_start` = 1.

**Why:** a pit-lane starter starts behind the whole grid, so its position is "after the last slot". Leaving 0 would put it in front of pole; making it NaN would let the Pipeline fill in a median (around 10th), which is also wrong. The flag keeps the information that it was a pit-lane start. This is a recode of a known value, not a filled-in guess.

**Evidence:** cell 4.7 / 6.3: 0 podiums from 87 pit-lane starts (0%), like the back of the grid; output above: the pit-lane rows in `model_df` and their new positions.

### 8.3 Qualifying features
`grid` is **not** the qualifying result: penalties move drivers back (cell 7.2: grid differs from qualifying in 25–30% of rows from 2004). Qualifying also measures **pace**: how far a driver was from the fastest car.

- `quali_position`: qualifying position.
- `quali_q1_gap_pct`: % slower than the fastest **Q1** time of that race. Q1 is used because every driver runs it; Q2 / Q3 do not exist before 2005–06 (single-lap formats) and only the faster drivers reach them.
- `quali_grid_penalty`: `grid_position − quali_position` (positive = starts further back than qualified, e.g. a penalty).

The known q1 outlier of about 1,000 s is handled with a rule: a Q1 time more than 1.5 × the fastest Q1 of that race is not a real flying lap → its gap stays NaN.

In [ ]:
# 8.3 Qualifying features (qualifying ends before the race; q1 is already in seconds in clean)
QUALI_OUTLIER_FACTOR = 1.5       # a Q1 time above 1.5 x the race's fastest Q1 is not a real flying lap

quali_feats = clean["qualifying"][["raceId", "driverId", "position", "q1"]].copy()
fastest_q1 = quali_feats.groupby("raceId")["q1"].transform("min")
not_real_lap = quali_feats["q1"] > QUALI_OUTLIER_FACTOR * fastest_q1
quali_feats["quali_position"] = quali_feats["position"]
quali_feats["quali_q1_gap_pct"] = ((quali_feats["q1"] / fastest_q1 - 1) * 100).where(~not_real_lap)

outliers = quali_feats[not_real_lap].merge(race_order[["raceId", "year"]], on="raceId")
print("Q1 times flagged as 'not a real lap' (all seasons):")
print(outliers[["raceId", "driverId", "year", "q1"]].assign(fastest_q1=fastest_q1[not_real_lap].values).to_string(index=False))

model_df = add_features(model_df, quali_feats, ["quali_position", "quali_q1_gap_pct"])
model_df["quali_grid_penalty"] = model_df["grid_position"] - model_df["quali_position"]
assert (outliers["year"] < TRAIN_START_YEAR).all()   # the outlier lap is outside the modeling seasons

quali_cols = ["quali_position", "quali_q1_gap_pct", "quali_grid_penalty"]
print(f"\nMissing in model_df ({len(model_df):,} rows):")
print(model_df[quali_cols].isna().sum().to_string())

**What the output shows:** the only Q1 time above 1.5 × the fastest is the **1,002.6 s lap from 1995** (raceId 255, fastest Q1 98.0 s), so it is **outside the modeling seasons** (assert) and does not touch `model_df`; the rule stays in the code as a guard. In `model_df`, **21 rows (0.24%)** have no qualifying row at all (so no `quali_position` and no penalty), and `quali_q1_gap_pct` is missing in **144 rows (1.6%)**: the 21 plus drivers with a qualifying position but no Q1 time (most in 2003–2005: 48 rows). All stay NaN (the Pipeline fills them, train only).

### 8.4 Sprint races (Decision 9)
Sprint races run on Saturday, before the Sunday race. Two formats exist in the data:
- **2021–2022:** Friday qualifying set the **sprint** grid, and the sprint result set the **Sunday** grid.
- **2023–2024:** a separate "sprint shootout" sets the sprint grid; Friday qualifying sets the Sunday grid again.

In [ ]:
# 8.4 Which races have sprints, and what does the Sunday grid follow on those weekends?
sprint_races = set(clean["sprint_results"]["raceId"])
chk = (model_df[["raceId", "year", "grid_position", "quali_position"]]
       .assign(sprint_weekend=model_df["raceId"].isin(sprint_races),
               grid_equals_quali=lambda d: d["grid_position"] == d["quali_position"]))
sprint_table = (chk[chk["year"] >= clean["races"].loc[clean["races"]["raceId"].isin(sprint_races), "year"].min()]
                .groupby(["year", "sprint_weekend"])
                .agg(races=("raceId", "nunique"), rows=("raceId", "size"), grid_equals_quali=("grid_equals_quali", "mean"))
                .round(2).unstack("sprint_weekend"))
print(f"Sprint weekends: {len(sprint_races)} races, {model_df['raceId'].isin(sprint_races).sum()} rows of model_df "
      f"({model_df['raceId'].isin(sprint_races).mean():.1%})")
sprint_table

**What the table shows:** 18 sprint weekends (3 in 2021, 3 in 2022, 6 in 2023, 6 in 2024), about 4% of `model_df`, **all from 2021 on**. On 2021–22 sprint weekends the Sunday grid equals the qualifying position in only **12–18%** of rows (the sprint reshuffled the grid), against **63–75%** on 2023–24 sprint weekends and on normal weekends of the same years.

**Decision 9 — Sprint races: use partly.** Keep the 18 sprint-weekend races. Use the Sunday `grid` as it is, and **no features built from `sprint_results`**. Driver and constructor form count Grand Prix results only.

**Why:** our cutoff is "after qualifying, before the race starts". The sprint is finished before the Sunday start, so the Sunday grid (which in 2021–22 is the sprint result) is known at prediction time → **not leakage**. Dropping the weekends would remove rows from the test era for no reason. Separate sprint features would be NaN in 96% of rows and every sprint weekend falls in validation or test (Decision 10), so the model could not learn them. Standings use the previous round, so this weekend's sprint points are never included.

**Evidence:** table above (grid vs qualifying on sprint weekends); 8.9 shows that standings after a sprint weekend include the sprint points. **Limitation:** in 2021–22 sprint weekends `quali_position` describes the sprint grid, not the Sunday grid (`quali_grid_penalty` then measures the sprint gain or loss).

### 8.5 Form features: which window? (trial)
Form = a driver's (or team's) results in the **previous N races** (`.shift(1)` removes the current race). The window rolls across season boundaries (the last races of the previous season are the most recent form), **but it restarts after a break of more than a year**: otherwise a comeback would carry very old results (e.g. Mercedes' first race in 2010 would get its 1955 results as "form").

**Trial:** N = 3, 5, 10, 15, 20 for the driver and the team podium rate. Score = ROC-AUC of the feature alone, on seasons up to `VAL_END_YEAR` only (train + validation, no test seasons).

In [ ]:
# 8.5 Trial (kept as evidence): which form window N? + effect of restarting form after a long break
FORM_MAX_GAP_DAYS = 365      # a break longer than this (no race for a year) restarts the form history

def rolling_past(df, key, col, n, max_gap_days=FORM_MAX_GAP_DAYS):
    """Mean of `col` over the previous n races of each `key` (current race excluded by shift(1)).
    History restarts after a break longer than max_gap_days. NaN if there is no earlier race. df sorted by date."""
    gap = df.groupby(key)["date"].diff().dt.days
    stint = (gap.isna() | (gap > max_gap_days)).groupby(df[key]).cumsum()
    return df.groupby([df[key], stint], sort=False)[col].transform(lambda s: s.shift(1).rolling(n, min_periods=1).mean())

# one row per team per race (average of its cars) -> team form
team_race = (hist.groupby(["raceId", "constructorId"], sort=False)
             .agg(date=("date", "first"), year=("year", "first"),
                  podium=("podium", "mean"), positionOrder=("positionOrder", "mean"), dnf=("dnf", "mean"))
             .reset_index().sort_values(["date", "raceId", "constructorId"]).reset_index(drop=True))

eda_years = model_df["year"] <= VAL_END_YEAR
def auc_alone(values, on, keys):
    """ROC-AUC of one feature alone on train + validation seasons (rows with a value only)."""
    m = model_df.loc[eda_years, grain_key + ["constructorId", TARGET]].merge(values, on=keys, how="left")
    ok = m["value"].notna()
    return roc_auc_score(m.loc[ok, TARGET], m.loc[ok, "value"]), int((~ok).sum())

trial = []
for n in [3, 5, 10, 15, 20]:
    drv_auc, drv_nan = auc_alone(hist[grain_key].assign(value=rolling_past(hist, "driverId", "podium", n)), "drv", grain_key)
    con_auc, con_nan = auc_alone(team_race[["raceId", "constructorId"]].assign(value=rolling_past(team_race, "constructorId", "podium", n)),
                                 "con", ["raceId", "constructorId"])
    trial.append({"window_N": n, "driver_podium_rate_auc": round(drv_auc, 4), "team_podium_rate_auc": round(con_auc, 4),
                  "driver_rows_no_history": drv_nan, "team_rows_no_history": con_nan})
form_trial = pd.DataFrame(trial)

# Restart after a long break: what it changes (N = 10)
no_reset = rolling_past(team_race, "constructorId", "podium", 10, max_gap_days=10**6)
merc_2010 = team_race.index[(team_race["constructorId"] == 131) & (team_race["year"] == 2010)][0]
print(f"Mercedes (constructorId 131), first race of 2010: team form without restart = {no_reset[merc_2010]:.2f} "
      f"(from its 1954-55 races), with restart = {rolling_past(team_race, 'constructorId', 'podium', 10)[merc_2010]}")
form_trial

**What the table shows:** the podium rate over a longer window separates podium and non-podium rows better: the AUC rises from N = 3 to a peak around N = 15 and falls again for 20. N = 10 is within about 0.007 of the best for both the driver and the team. Almost every row has some history (the "no history" counts are drivers' and teams' first races, and races after a break of more than a year). Restarting after a long break changes the AUC by less than 0.001 but removes nonsense values like Mercedes' 1955 results as its 2010 form.

**Decision 18 — Form window and missing history:** form features use the **last 10 races** (`FORM_WINDOW = 10`), rolling across season boundaries, **restarting after a break of more than a year** (`FORM_MAX_GAP_DAYS = 365`), always with `.shift(1)`. No earlier race → **NaN** (filled later in the Pipeline, train only), not 0: a 0 would say "never on the podium", which is not known for a new driver or team. `ctx_driver_races_before` tells the model how much history a driver has.

**Why:** 10 races (about half a season) scores within 0.01 AUC of the best window while still reacting to a new car or a team switch within the season; 3 and 5 are clearly noisier (lower AUC). Chosen on train + validation seasons only.

**Evidence:** trial table above (seasons up to `VAL_END_YEAR`), Mercedes 2010 example.

In [ ]:
# 8.6 Driver form features (last FORM_WINDOW starts, current race excluded)
FORM_WINDOW = 10    # Decision 18

drv_feats = hist[grain_key].copy()
drv_feats[f"drv_podium_rate_last{FORM_WINDOW}"] = rolling_past(hist, "driverId", "podium", FORM_WINDOW)
drv_feats[f"drv_avg_finish_last{FORM_WINDOW}"] = rolling_past(hist, "driverId", "positionOrder", FORM_WINDOW)
drv_feats[f"drv_dnf_rate_last{FORM_WINDOW}"] = rolling_past(hist, "driverId", "dnf", FORM_WINDOW)
drv_cols = [c for c in drv_feats.columns if c.startswith("drv_")]
model_df = add_features(model_df, drv_feats, drv_cols)
print(model_df[drv_cols].describe().round(2).to_string())
print("\nMissing (driver's first start, or first race after a break of more than a year):", model_df[drv_cols[0]].isna().sum(), "rows")

**What the output shows:** three driver-form features over the last 10 races. **128 rows (1.4%)** have no history: a driver's first start, or his first race after a break of more than a year. The podium rate moves in steps of 0.1 (0 to 10 podiums in the last 10 races).

### 8.7 Constructor form: follow the team across renames? (trial)
Constructor form = the team's results (both cars) in its **previous 10 races** (Decision 18). A renamed team gets a new `constructorId`, so its form restarts. Decision 7 linked renames with `franchise_map` **for Q3 only**. Should form follow `franchise_map`?

**Trial:** compute the team podium rate both ways and compare on seasons up to `VAL_END_YEAR`.

In [ ]:
# 8.7 Trial (kept as evidence): constructor form keyed on constructorId vs on franchise_map (Decision 7 lineages)
team_race["franchise"] = team_race["constructorId"].map(franchise_map).fillna(team_race["constructorId"].astype(str))

trial_con = {}
for key in ["constructorId", "franchise"]:
    tr = team_race[["raceId", "constructorId"]].assign(rate=rolling_past(team_race, key, "podium", FORM_WINDOW))
    m = model_df[eda_years].merge(tr, on=["raceId", "constructorId"], how="left", validate="many_to_one")
    ok = m["rate"].notna()
    trial_con[key] = m["rate"]
    print(f"key = {key:13s}: rows with no team history {(~ok).sum():3d}, "
          f"ROC-AUC alone {roc_auc_score(m.loc[ok, TARGET], m.loc[ok, 'rate']):.4f}")
diff = ~np.isclose(trial_con["constructorId"], trial_con["franchise"], equal_nan=True)   # NaN = NaN counts as "same"
print(f"Rows where the two versions differ: {diff.sum()} of {eda_years.sum():,} ({diff.mean():.1%})")

# Which teams start fresh (first season) inside the modeling years? -> renames that franchise_map does NOT link
first_season = hist.groupby("constructorId")["year"].min()
new_ids = first_season[(first_season > TRAIN_START_YEAR) & (first_season <= VAL_END_YEAR)]
new_teams = (clean["constructors"].set_index("constructorId").loc[new_ids.index, ["name"]]
             .assign(first_season=new_ids, in_franchise_map=lambda t: t.index.isin(list(franchise_map))))
new_teams.sort_values("first_season")

**What the output shows:** the two versions differ in only **80 of 7,558 rows (1.1%)**: a rename only matters for the team's first races under the new name, because after 10 races the window holds only races under the new name anyway. The AUC is practically the same (**0.8817** with `constructorId`, **0.8820** with `franchise_map`). The list of teams that first appear inside the modeling years shows that `franchise_map` links only the renames from 2014 on (it was built for Q3); earlier renames inside 2003–2013 (e.g. Minardi → Toro Rosso, Jordan → Midland → Spyker → Force India, BAR → Honda → Brawn → Mercedes, Jaguar → Red Bull) are **not** in it.

**Decision 19 — Constructor form follows `constructorId`** (no linking of renames in the features). Decision 7 stays as it is (Q3 only).

**Why:** using `franchise_map` would link the 2014+ renames but not the 2003–2013 ones, so the same situation would be treated two ways. Extending the map to all renames would add more judgement for almost no gain: the two versions differ only in the first races after a rename and score the same. With `constructorId` a renamed team simply has a short history at first (NaN for its first race), which the Pipeline handles.

**Evidence:** trial output above (rows that differ, AUC per version, list of new constructor ids and whether `franchise_map` covers them); Decision 7 / cell 7.11 (rename check shows no break, weak evidence).

In [ ]:
# 8.8 Constructor form features (team = constructorId, Decision 19; previous FORM_WINDOW races, current race excluded)
con_feats = team_race[["raceId", "constructorId"]].copy()
con_feats[f"con_podium_rate_last{FORM_WINDOW}"] = rolling_past(team_race, "constructorId", "podium", FORM_WINDOW)
con_feats[f"con_avg_finish_last{FORM_WINDOW}"] = rolling_past(team_race, "constructorId", "positionOrder", FORM_WINDOW)
con_feats[f"con_dnf_rate_last{FORM_WINDOW}"] = rolling_past(team_race, "constructorId", "dnf", FORM_WINDOW)
con_cols = [c for c in con_feats.columns if c.startswith("con_")]

n_before = len(model_df)
model_df = model_df.merge(con_feats, on=["raceId", "constructorId"], how="left", validate="many_to_one")
assert len(model_df) == n_before and not model_df.duplicated(grain_key).any()
print(model_df[con_cols].describe().round(2).to_string())
print("\nMissing (team's first race under this constructorId, or after a break of more than a year):", model_df[con_cols[0]].isna().sum(), "rows")

**What the output shows:** three constructor-form features; a team-race value is the average of its cars in that race (e.g. one of two cars on the podium = 0.5). **55 rows (0.6%)** have no team history: a team's first race under a new `constructorId` (e.g. Red Bull 2005, Brawn 2009, Haas 2016), or after a break of more than a year.

### 8.9 Standings before the race (previous round)
`driver_standings` / `constructor_standings` for race X **already include race X** (cell 2.7). So the features use the standings after the **previous round of the same season**. Before the first round of a season everybody has **0 points** (a fact, not a guess) and **no rank** yet (NaN). A driver who is not in the previous round's standings (e.g. a mid-season debut) also has 0 points so far.

In [ ]:
# 8.9 Standings from the previous round of the same season
prev_round = race_order.assign(prev_raceId=race_order.groupby("year")["raceId"].shift(1))[["raceId", "prev_raceId"]]

ds = clean["driver_standings"][["raceId", "driverId", "points", "position"]].rename(
    columns={"raceId": "prev_raceId", "points": "stand_driver_points_before", "position": "stand_driver_rank_before"})
cs = clean["constructor_standings"][["raceId", "constructorId", "points", "position"]].rename(
    columns={"raceId": "prev_raceId", "points": "stand_con_points_before", "position": "stand_con_rank_before"})

stand_feats = (hist[grain_key + ["constructorId"]]
     .merge(prev_round, on="raceId", how="left", validate="many_to_one")
     .merge(ds, on=["prev_raceId", "driverId"], how="left", validate="many_to_one")
     .merge(cs, on=["prev_raceId", "constructorId"], how="left", validate="many_to_one"))
for col in ["stand_driver_points_before", "stand_con_points_before"]:
    stand_feats[col] = stand_feats[col].where(stand_feats[col].notna(), 0.0)      # no standings row before this race = 0 points so far (a fact)
stand_cols = ["stand_driver_points_before", "stand_driver_rank_before", "stand_con_points_before", "stand_con_rank_before"]
model_df = add_features(model_df, stand_feats, stand_cols)

# Check 1: first round of every season -> 0 points, no rank
first = model_df["round"] == 1
assert (model_df.loc[first, ["stand_driver_points_before", "stand_con_points_before"]] == 0).all().all()
assert model_df.loc[first, ["stand_driver_rank_before", "stand_con_rank_before"]].isna().all().all()

# Check 2 (proof that standings for race X include race X): points before + points scored in X = standings after X
after = clean["driver_standings"][["raceId", "driverId", "points"]].rename(columns={"points": "standings_after"})
proof = (model_df[grain_key + ["year", "stand_driver_points_before"]]
         .merge(hist[grain_key + ["points"]], on=grain_key, validate="one_to_one")
         .merge(after, on=grain_key, how="inner", validate="one_to_one"))
proof["adds_up"] = np.isclose(proof["stand_driver_points_before"] + proof["points"], proof["standings_after"])
proof["sprint_weekend"] = proof["raceId"].isin(sprint_races)
print("Share of driver-races where points_before + race points = standings after the race:")
print(proof.groupby("sprint_weekend")["adds_up"].agg(["size", "mean"]).round(3).to_string())

# By hand: one driver, first races of one season
hand = (model_df.loc[(model_df["driverId"] == 1) & (model_df["year"] == 2008),
                     ["year", "round", "stand_driver_points_before", "stand_driver_rank_before"]]
        .merge(proof.loc[proof["driverId"] == 1, ["year", "raceId", "points", "standings_after"]]
               .merge(race_order[["raceId", "round"]], on="raceId"), on=["year", "round"]).head(5))
print("\nDriver 1 (Lewis Hamilton), 2008, first 5 rounds:")
print(hand[["round", "stand_driver_points_before", "points", "standings_after", "stand_driver_rank_before"]].to_string(index=False))

**What the output shows:** on normal weekends, "points before + points scored in the race = standings after the race" holds in **100% of 8,476 rows**: the standings table for race X **includes** race X, so taking the previous round is necessary. On sprint weekends it holds in only **64% of 355 rows**, because the standings after the race also include **the sprint points** of that weekend — another reason the previous round is the safe choice (Decision 9). The hand-printed rows show it for one driver: each round's "points before" is the previous round's "standings after". Asserts: every first round has 0 points and no rank.

### 8.10 Context features
- `ctx_home_race`: the circuit is in the driver's home country (Decision 12, reused from 7.6 via `analysis`).
- `ctx_field_size`: number of starters in the race (more cars = harder to reach the top 3).
- `ctx_driver_age`: age in years on race day (date of birth is fixed).
- `ctx_driver_races_before`: number of earlier starts in `clean` (experience; also tells the model when the form features have little history).

In [ ]:
# 8.10 Context features
ctx_feats = hist[grain_key + ["date"]].copy()
ctx_feats["ctx_field_size"] = ctx_feats.groupby("raceId")["driverId"].transform("size")
ctx_feats["ctx_driver_races_before"] = hist.groupby("driverId").cumcount()          # hist is sorted by date -> earlier starts only
dob = clean["drivers"].set_index("driverId")["dob"]
ctx_feats["ctx_driver_age"] = (ctx_feats["date"] - ctx_feats["driverId"].map(dob)).dt.days / 365.25
ctx_feats = ctx_feats.merge(analysis[grain_key + ["home_race"]], on=grain_key, how="left", validate="one_to_one")
ctx_feats["ctx_home_race"] = ctx_feats["home_race"].astype(int)

ctx_cols = ["ctx_home_race", "ctx_field_size", "ctx_driver_age", "ctx_driver_races_before"]
model_df = add_features(model_df, ctx_feats, ctx_cols)
print(model_df[ctx_cols].describe().round(2).to_string())

# Year / era as a feature? Podium rate per season (train + validation seasons only) and the seasons each split would contain
per_year = model_df[eda_years].groupby("year")[TARGET].mean()
print(f"\nPodium rate per season {per_year.index.min()}-{per_year.index.max()}: min {per_year.min():.1%}, max {per_year.max():.1%}")
print(f"Seasons after VAL_END_YEAR (test) never appear in training: {sorted(set(model_df['year']) - set(model_df.loc[eda_years, 'year']))}")

**What the output shows:** the four context features are complete (no missing values); 4% of rows are home races, field sizes range from 6 (2005 US GP) to 24 starters. The podium rate per season stays in a narrow band (**12.6–15.8%** in 2003–2021), because every race has exactly 3 podium places.

**Decision 20 — No year / era feature.** `year` stays in `model_df` (needed for the split) but is not a feature.

**Why:** with a time-based split (Decision 10) every test season is a value the model has never seen, so a year feature can only extrapolate; a "2022 rules era" flag would be 0 in every training row, so the model could learn nothing from it. The base rate does not drift (podium share per season is flat), and era changes in car performance are already captured by the form, standings and qualifying-gap features, which are relative to the other cars in the same season.

**Evidence:** output above: podium rate per season 12.6–15.8%; the test seasons 2022–2024 never appear in training.

### 8.11 Feature groups and the final feature list
`FEATURE_GROUPS` is built from the prefixes (code conventions 3.3), so the ablation in Section 11 can remove one group at a time automatically. `feature_notes` gives one line per feature: what it is and why it is known **before the race starts**.

In [ ]:
# 8.11 FEATURE_GROUPS / FEATURES from the prefixes + one line per feature (what it is, why it is safe before the start)
PREFIXES = ["grid_", "quali_", "drv_", "con_", "stand_", "ctx_"]
FEATURE_GROUPS = {p.rstrip("_"): [c for c in model_df.columns if c.startswith(p)] for p in PREFIXES}
FEATURES = [c for cols in FEATURE_GROUPS.values() for c in cols]

W = FORM_WINDOW
why_safe = {
    "grid_position": ("starting slot (pit lane = last slot + 1)", "the grid is fixed before the start"),
    "grid_front_row": ("1 if grid slot 1 or 2", "from the grid"),
    "grid_pitlane_start": ("1 if starting from the pit lane (grid = 0)", "known at the start"),
    "quali_position": ("qualifying position", "qualifying ends before the race"),
    "quali_q1_gap_pct": ("% slower than the fastest Q1 time of the race", "qualifying ends before the race"),
    "quali_grid_penalty": ("grid slot minus qualifying position (> 0 = moved back)", "both known before the start"),
    f"drv_podium_rate_last{W}": (f"driver's podium share in his previous {W} races", ".shift(1): earlier races only"),
    f"drv_avg_finish_last{W}": (f"driver's average finishing place in his previous {W} races", ".shift(1): earlier races only"),
    f"drv_dnf_rate_last{W}": (f"share of the driver's previous {W} races not classified", ".shift(1): earlier races only"),
    f"con_podium_rate_last{W}": (f"team's podium share (per car) in its previous {W} races", ".shift(1): earlier races only"),
    f"con_avg_finish_last{W}": (f"team's average finishing place in its previous {W} races", ".shift(1): earlier races only"),
    f"con_dnf_rate_last{W}": (f"share of the team's cars not classified in its previous {W} races", ".shift(1): earlier races only"),
    "stand_driver_points_before": ("driver's championship points before this race", "standings of the previous round"),
    "stand_driver_rank_before": ("driver's championship position before this race", "standings of the previous round"),
    "stand_con_points_before": ("team's championship points before this race", "standings of the previous round"),
    "stand_con_rank_before": ("team's championship position before this race", "standings of the previous round"),
    "ctx_home_race": ("1 if the circuit is in the driver's home country (Decision 12)", "nationality and circuit are fixed"),
    "ctx_field_size": ("number of starters in the race", "known when the grid forms"),
    "ctx_driver_age": ("driver's age on race day (years)", "date of birth and race date are fixed"),
    "ctx_driver_races_before": ("number of earlier starts (experience)", "counts earlier races only"),
}
assert set(why_safe) == set(FEATURES), set(why_safe) ^ set(FEATURES)
feature_notes = pd.DataFrame(
    [{"group": grp, "feature": f, "what it is": why_safe[f][0], "why safe before the race": why_safe[f][1],
      "missing (rows)": int(model_df[f].isna().sum())} for grp, cols in FEATURE_GROUPS.items() for f in cols])
print(f"{len(FEATURES)} features in {len(FEATURE_GROUPS)} groups: " + ", ".join(f"{g} {len(c)}" for g, c in FEATURE_GROUPS.items()))
feature_notes

**What the table shows:** **20 features in 6 groups** (grid 3, qualifying 3, driver form 3, constructor form 3, standings 4, context 4). Missing values come only from: no qualifying row / no Q1 time (21 / 144 rows), the first races of a driver (128) or team (55) or after a long break, and rounds where a driver or team has no championship position yet (594 / 484 rows, of which 453 are first rounds of a season). They all stay NaN for the Pipeline.

### 8.12 Leakage checks
1. **Recompute form by brute force** for 300 random rows: for each, take the driver's (team's) races **strictly before** that race date, keep the last `FORM_WINDOW` (after any break of more than a year), and compare with the feature.
2. **By hand:** one driver's first races of a season — the podium in race k must not appear in his form until race k + 1.
3. **No post-race column** in `FEATURES`, every feature has a group prefix, one row per (raceId, driverId).

In [ ]:
# 8.12 Leakage checks
rng = np.random.default_rng(SEED)
sample = model_df.iloc[rng.choice(len(model_df), 300, replace=False)]
drv_rate, con_rate = f"drv_podium_rate_last{FORM_WINDOW}", f"con_podium_rate_last{FORM_WINDOW}"

def brute_force(past):
    """Mean podium of the last FORM_WINDOW earlier races, after the last break longer than FORM_MAX_GAP_DAYS."""
    if past.empty:
        return np.nan
    dates = past["date"].reset_index(drop=True)
    breaks = dates.diff().dt.days > FORM_MAX_GAP_DAYS
    start = breaks[breaks].index.max() if breaks.any() else 0
    return past["podium"].iloc[start:].tail(FORM_WINDOW).mean()

for _, row in sample.iterrows():
    gap_now = lambda past: past if past.empty or (row["date"] - past["date"].iloc[-1]).days <= FORM_MAX_GAP_DAYS else past.iloc[0:0]
    d_past = gap_now(hist[(hist["driverId"] == row["driverId"]) & (hist["date"] < row["date"])])
    t_past = gap_now(team_race[(team_race["constructorId"] == row["constructorId"]) & (team_race["date"] < row["date"])])
    for feat, expected in [(drv_rate, brute_force(d_past)), (con_rate, brute_force(t_past))]:
        assert (np.isnan(expected) and np.isnan(row[feat])) or np.isclose(expected, row[feat]), (row[grain_key].tolist(), feat)
print(f"Check 1 passed: {drv_rate} and {con_rate} equal a brute-force recomputation from earlier races only (300 rows).")

# Check 2: by hand, driver 1 (Lewis Hamilton) in the first races of his career (2007)
by_hand = model_df.loc[(model_df["driverId"] == 1) & (model_df["year"] == 2007),
                       ["round", TARGET, drv_rate, "stand_driver_points_before"]].head(6)
print("\nCheck 2 (by hand): a result in round k only enters the form from round k + 1")
print(by_hand.to_string(index=False))

# Check 3: no post-race information among the features
post_race = {"position", "positionText", "positionOrder", "points", "laps", "time", "milliseconds", "fastestLap",
             "rank", "fastestLapTime", "fastestLapSpeed", "statusId", "status", "podium", "dnf", "stop", "lap", "duration"}
assert not post_race & set(FEATURES), post_race & set(FEATURES)
assert all(f.startswith(tuple(PREFIXES)) for f in FEATURES) and TARGET not in FEATURES
assert not model_df.duplicated(grain_key).any() and len(model_df) == 8910
print(f"\nCheck 3 passed: {len(FEATURES)} features, none of them a post-race column; model_df still {len(model_df):,} unique driver-races.")

**What the output shows:** all three checks pass. The brute-force recomputation (which only ever looks at races with an **earlier date**) gives exactly the same driver and team form for 300 random rows. In the hand-printed rows (Lewis Hamilton's first races, 2007), his podium in round 1 is **not** in his round-1 form (NaN: no earlier race) and appears only from round 2; "points before" is always the previous round's total.

### 8.13 Why each feature group helps: one chart per group
Each chart shows the podium rate of the rows in train + validation seasons (up to `VAL_END_YEAR`), grouped by one feature of the group. Grey dashed line = overall podium rate. Orange = a special value of the feature (pit-lane start, no history, first round).

In [ ]:
# 8.13a Grid group: podium rate by grid_position (pit-lane starts highlighted)
eda = model_df[eda_years].copy()
BASE_RATE = eda[TARGET].mean()

def podium_rate_bars(groups, title, xlabel, highlight=()):
    """Bar chart of the podium rate per group (index = labels), with the overall rate as a reference line."""
    stats = groups[TARGET].agg(["mean", "size"])
    fig, ax = plt.subplots(figsize=FIGSIZE)
    colors = ["#eb6834" if lab in highlight else "#2a78d6" for lab in stats.index]
    bars = ax.bar([str(i) for i in stats.index], stats["mean"] * 100, color=colors)
    for bar, rate in zip(bars, stats["mean"]):
        if rate == 0:                                   # make a 0% bar visible
            ax.annotate("0%", (bar.get_x() + bar.get_width() / 2, 0), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)
    ax.axhline(BASE_RATE * 100, color="#52514e", linestyle="--", linewidth=1, label=f"overall {BASE_RATE:.1%}")
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel("Podium rate (%)")
    ax.grid(axis="y", alpha=0.3); ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False)
    plt.show()
    return stats

grid_bucket = eda["grid_position"].clip(upper=20).astype(int).astype(str).replace({"20": "20+"})
grid_bucket[eda["grid_pitlane_start"] == 1] = "pit"
order = [str(i) for i in range(1, 20)] + ["20+", "pit"]
grid_stats = podium_rate_bars(eda.groupby(pd.Categorical(grid_bucket, categories=order, ordered=True), observed=True),
                              f"Podium rate by grid position, {TRAIN_START_YEAR}-{VAL_END_YEAR}", "Grid position (pit = pit-lane start)", highlight=("pit",))

**What the chart shows:** on the modeling seasons the grid slot is even stronger than over all years (cell 4.7): **79.9% podium from pole**, 64.0% from 2nd, 52.5% from 3rd, 20.3% from 5th, 5% or less from 8th on. The 36 pit-lane starts in 2003–2021 (orange) never reached the podium (0%), like the back of the grid → supports `grid_position` with pit-lane starts placed behind the last slot (Decision 17).

In [ ]:
# 8.13b Qualifying group: podium rate by the gap to the fastest Q1 time
gap_bins = [0, 0.25, 0.5, 0.75, 1, 1.5, 2, 3, np.inf]
gap_labels = ["fastest-0.25", "0.25-0.5", "0.5-0.75", "0.75-1", "1-1.5", "1.5-2", "2-3", "3+"]
gap_bucket = pd.cut(eda["quali_q1_gap_pct"], gap_bins, labels=gap_labels, include_lowest=True)
quali_stats = podium_rate_bars(eda.groupby(gap_bucket, observed=True),
                               f"Podium rate by Q1 gap to the fastest car, {TRAIN_START_YEAR}-{VAL_END_YEAR}",
                               "Q1 time, % slower than the fastest Q1 of the race")

**What the chart shows:** drivers within 0.25% of the fastest Q1 time finish on the podium in **55.2%** of races, 0.25–0.5% behind in 32.6%; from 1.5% behind the rate is under 5% (1.4% from 3% behind). The overall rate is 14.3%. The gap measures **pace**, which the grid slot alone does not show (e.g. a fast car starting further back after a penalty) → supports the qualifying group.

In [ ]:
# 8.13c Driver form group: podium rate by the driver's podium share in his previous races
drv_bucket = pd.cut(eda[drv_rate], [-0.01, 0, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0],
                    labels=["0", "0-0.1", "0.1-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1"]).astype(str).replace({"nan": "no history"})
order = ["no history", "0", "0-0.1", "0.1-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1"]
drv_stats = podium_rate_bars(eda.groupby(pd.Categorical(drv_bucket, categories=order, ordered=True), observed=True),
                             f"Podium rate by driver podium share in his last {FORM_WINDOW} races, {TRAIN_START_YEAR}-{VAL_END_YEAR}",
                             f"Driver's podium share in his previous {FORM_WINDOW} races", highlight=("no history",))

**What the chart shows:** the more often a driver reached the podium in his previous 10 races, the more often he does so again: **2.2%** for drivers with no podium in their last 10 races (half of all rows), **21.2%** with 1–2 podiums, **71.2%** with 9–10. Drivers with no history (orange, 116 rows) reach it in 2.6% → supports the driver-form group (Decision 18).

In [ ]:
# 8.13d Constructor form group: podium rate by the team's podium share (per car) in its previous races
con_bucket = pd.cut(eda[con_rate], [-0.01, 0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0],
                    labels=["0", "0-0.1", "0.1-0.2", "0.2-0.3", "0.3-0.4", "0.4-0.5", "0.5-1"]).astype(str).replace({"nan": "no history"})
order = ["no history", "0", "0-0.1", "0.1-0.2", "0.2-0.3", "0.3-0.4", "0.4-0.5", "0.5-1"]
con_stats = podium_rate_bars(eda.groupby(pd.Categorical(con_bucket, categories=order, ordered=True), observed=True),
                             f"Podium rate by team podium share in its last {FORM_WINDOW} races, {TRAIN_START_YEAR}-{VAL_END_YEAR}",
                             f"Team's podium share per car in its previous {FORM_WINDOW} races", highlight=("no history",))

**What the chart shows:** the same pattern for teams: cars of teams with no podium in their last 10 races reach it in only **1.4%** of races, cars of the leading teams (share above 0.5 per car) in **59.7%**. The car matters as much as the driver in F1, so this group adds information the driver form does not have (e.g. a driver moving to a top team).

In [ ]:
# 8.13e Standings group: podium rate by the driver's championship position before the race
rank = eda["stand_driver_rank_before"]
rank_bucket = pd.cut(rank, [0, 1, 2, 3, 5, 8, 12, np.inf], labels=["1", "2", "3", "4-5", "6-8", "9-12", "13+"]).astype(str)
rank_bucket[rank.isna()] = "first round"
order = ["first round", "1", "2", "3", "4-5", "6-8", "9-12", "13+"]
stand_stats = podium_rate_bars(eda.groupby(pd.Categorical(rank_bucket, categories=order, ordered=True), observed=True),
                               f"Podium rate by championship position before the race, {TRAIN_START_YEAR}-{VAL_END_YEAR}",
                               "Driver's championship position after the previous round", highlight=("first round",))

**What the chart shows:** the championship leader before a race reaches the podium in **69.0%** of races, 2nd in 59.1%, 3rd in 41.1%; from 9th place on the rate is under 4%. At the first round (orange) nobody has a position yet and the rate (12.0%) is close to the overall rate → that is why the rank stays NaN there: "no position yet" is not the same as "last".

In [ ]:
# 8.13f Context group: podium rate by experience (number of earlier starts)
exp_bucket = pd.cut(eda["ctx_driver_races_before"], [-1, 0, 10, 25, 50, 100, 150, 200, np.inf],
                    labels=["0", "1-10", "11-25", "26-50", "51-100", "101-150", "151-200", "200+"])
ctx_stats = podium_rate_bars(eda.groupby(exp_bucket, observed=True),
                             f"Podium rate by driver experience, {TRAIN_START_YEAR}-{VAL_END_YEAR}",
                             "Number of earlier starts")

**What the chart shows:** drivers with up to 25 earlier starts reach the podium in under 3% of races; the rate rises to **29.9%** at 151–200 starts and falls back to 20.0% beyond 200 (the relationship is not a straight line, which tree models can capture). Experienced drivers tend to be in the stronger cars, so this is **linked to**, not caused by, experience. The other context features have weaker signals: the home flag showed no advantage once the grid is controlled (Q2, cell 7.9), and field size changes the "3 out of N" base rate. They are kept, and the ablation (Section 11) will show whether the group helps.

### 8.14 Correlation between the features
Spearman correlation (rank-based, so it works for counts, rates and flags) between all features, train + validation seasons.

In [ ]:
# 8.14 Correlation heatmap (Spearman) between all features, seasons up to VAL_END_YEAR
corr = eda[FEATURES].corr(method="spearman")
fig, ax = plt.subplots(figsize=FIGSIZE)
sns.heatmap(corr, cmap="RdBu_r", vmin=-1, vmax=1, center=0, ax=ax, square=False,
            cbar_kws={"label": "Spearman correlation"}, xticklabels=True, yticklabels=True)
ax.set_title(f"Correlation between features, {TRAIN_START_YEAR}-{VAL_END_YEAR}")
ax.set_xlabel("Feature"); ax.set_ylabel("Feature")
ax.tick_params(labelsize=7)
plt.show()

pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
         .rename("spearman").reset_index().assign(abs_corr=lambda d: d["spearman"].abs())
         .sort_values("abs_corr", ascending=False).head(8))
pairs.round(2)

**What the heatmap shows:** features of the same group are strongly correlated (grid and qualifying position **0.97**; driver and team points **0.95**), and the form and standings groups are also correlated with each other (driver vs team average finish 0.91, driver average finish vs championship rank 0.90), because they all measure "how strong is this car and driver". This is expected and is not leakage. It matters for two later steps: logistic regression coefficients of correlated features are hard to read (use the regularised model and the XAI section), and in the ablation (Section 11) removing one group may cost little because another group carries the same information.

## 9. Train / Validation / Test Split
**Goal:** split `model_df` into train / validation / test **by season**, compare 2–3 boundary options inside 2003–2024 (Decision 4) and choose one (Decision 10).

**Uses:** `model_df`, `FEATURES`, `TARGET`, `TRAIN_END_YEAR`, `VAL_END_YEAR` (setup cell)

**Creates:** `split_of`, `X_train`, `y_train`, `X_val`, `y_val`, `X_test`, `y_test`

**The test set is locked:** this section shows only row counts and class balance per split. No model is scored here, and no feature is chosen by looking at the test seasons.

### 9.1 Why not a random split?
We predict **future** races from **past** ones. A random split mixes seasons: the model would train on races that happen **after** the races it is tested on (e.g. learn from 2023 that a team is strong, then "predict" its 2022 podiums), and rows of the **same race** would sit in both train and test (the 3 podium places of a race are shared out between its drivers, so knowing who finished on the podium in train tells you about the test drivers of that race). Both make the test score look better than it will be on a new race. The cell below measures how much of this a random 70/15/15 split would cause.

In [ ]:
# 9.1 What a random row split would do (illustration only; we do NOT use it)
rng = np.random.default_rng(SEED)
random_split = pd.Series(rng.choice(["train", "val", "test"], size=len(model_df), p=[0.70, 0.15, 0.15]), index=model_df.index)
races_in = model_df.groupby("raceId").apply(lambda d: set(random_split[d.index]), include_groups=False)
test_races = races_in[races_in.apply(lambda s: "test" in s)]
shared = test_races.apply(lambda s: "train" in s).mean()
latest_train = model_df.loc[random_split == "train", "date"].max()
future_share = (model_df.loc[random_split == "train", "date"] > model_df.loc[random_split == "test", "date"].min()).mean()
print(f"Random split: {shared:.1%} of the races in the test part also have rows in train (same race on both sides).")
print(f"Random split: {future_share:.1%} of the training rows are from races later than the first test race "
      f"(the latest training race is {latest_train.date()}).")

**What the output shows:** with a random split **100%** of the test races also have drivers in the training data, and **99.8%** of the training rows come from races **later** than the earliest test race: the model would "see the future". A split **by season** avoids both: all races of a season are on one side, and validation and test always come **after** training. This matches the TA advice ("watch leakage") and how the model would be used (predict the next race from past races).

### 9.2 Three split options
All options use seasons 2003–2024 (Decision 4), split by season, in time order:

| Option | Train | Validation | Test | Idea |
|-|-|-|-|-|
| A | 2003–2019 | 2020–2021 | 2022–2024 | the brief's example (TA: "just a naive example") |
| B | 2003–2018 | 2019–2021 | 2022–2024 | 3 validation seasons, so 2020 is not the main validation season |
| C | 2003–2019 | 2020–2022 | 2023–2024 | validation already includes the first season of the 2022 rules |

In [ ]:
# 9.2 Rows, races and podium share per split for each option (counts and class balance only)
SPLIT_OPTIONS = {"A": (2019, 2021), "B": (2018, 2021), "C": (2019, 2022)}   # (last train season, last validation season)

def split_of(years, train_end, val_end):
    """Label each season as train / val / test for given boundaries."""
    return pd.Series(np.select([years <= train_end, years <= val_end], ["train", "val"], "test"), index=years.index)

rows = []
for name, (tr_end, va_end) in SPLIT_OPTIONS.items():
    lab = split_of(model_df["year"], tr_end, va_end)
    for part in ["train", "val", "test"]:
        part_df = model_df[lab == part]
        rows.append({"option": name, "split": part, "seasons": f"{part_df['year'].min()}-{part_df['year'].max()}",
                     "races": part_df["raceId"].nunique(), "rows": len(part_df), "share_of_rows": f"{len(part_df) / len(model_df):.1%}",
                     "podium_%": round(part_df[TARGET].mean() * 100, 1)})
split_options = pd.DataFrame(rows)
split_options

**What the table shows:** the podium share is about the same in every split of every option (**14.1–15.1%**), because every race has exactly 3 podium places → class balance does not decide between the options (and no stratification is needed). The options differ in **size and era**: A's validation is only 2 seasons (775 rows) and half of it is 2020; B's validation has 3 seasons (1,195 rows, 13% of rows) and keeps the whole 2022–2024 rules era as test; C moves 2022 into validation (1,214 rows) but leaves only 2 test seasons (913 rows).

### 9.3 Is 2020 a good validation season?
2020 was the COVID season: a short, unusual calendar (cell 4.3). If it is the main validation season, the threshold and the model choice would be tuned on an untypical year.

In [ ]:
# 9.3 How unusual was 2020? Races, rows and repeated circuits per season (2018-2021)
season_check = (model_df[model_df["year"].between(2018, VAL_END_YEAR)]
                .merge(race_order[["raceId", "circuitId"]], on="raceId", validate="many_to_one")
                .groupby("year")
                .agg(races=("raceId", "nunique"), rows=("raceId", "size"), circuits=("circuitId", "nunique"),
                     podium_pct=(TARGET, "mean")))
season_check["races_at_a_repeated_circuit"] = season_check["races"] - season_check["circuits"]
season_check["podium_pct"] = (season_check["podium_pct"] * 100).round(1)
season_check

**What the table shows:** 2020 had only **17 races at 14 circuits** (vs 21 in 2018–2019 and 22 in 2021): **3 races were repeats at a circuit already used that season** (two races each in Austria, at Silverstone and in Bahrain), so it covers fewer different circuits than a normal season. Its podium share is normal (every race has 3 podium places), so it is not a "wrong" season, but it is a small and unusual one. With option A it is 44% of the validation rows (338 of 775); with option B 28% (338 of 1,195), next to the normal 2019 and 2021 seasons.

**Decision 10 — Split boundaries (option B):** split by season: **train 2003–2018, validation 2019–2021, test 2022–2024** (`TRAIN_END_YEAR = 2018`, `VAL_END_YEAR = 2021`, set in the setup cell).

**Why:**
- **By time, not random:** the model must predict future races from past ones; a random split puts later races and rows of the same race into training (9.1).
- **Validation 2019–2021** (3 seasons, 1,195 rows) is large enough to choose models and the threshold, and the short COVID season 2020 is 338 of its 1,195 rows (28%, 9.3), instead of half of a 2-season validation set (option A).
- **Test 2022–2024 = the whole new rules era** (ground-effect cars from 2022): 3 complete seasons, 1,352 rows, 68 races, untouched until the final evaluation. This is a hard but honest test: the model is trained only on earlier rule sets and must work after a big rule change, which is exactly the situation of a real forecast. Option C would tune on 2022 but leave only 2 test seasons.
- Class balance is the same in all splits (about 14–15% podiums), so no stratification is needed.

**Limitation:** the test seasons come from a new rule era, so test scores may be lower than validation scores; this is a property of the problem, not a bug, and it goes into the report.

**Evidence:** 9.1 (random split: shared races and future rows), 9.2 table (rows / races / podium % per option), 9.3 (2020: 17 races at 14 circuits), cell 4.3 (races per season).

In [ ]:
# 9.4 Chart: rows per season and which split each season belongs to (Decision 10)
assert TRAIN_END_YEAR is not None and VAL_END_YEAR is not None and TRAIN_START_YEAR <= TRAIN_END_YEAR < VAL_END_YEAR < model_df["year"].max()
model_df["split"] = split_of(model_df["year"], TRAIN_END_YEAR, VAL_END_YEAR)     # bookkeeping column, not a feature

per_season = model_df.groupby(["year", "split"]).size().unstack(fill_value=0)
split_colors = {"train": "#2a78d6", "val": "#eb6834", "test": "#52514e"}
fig, ax = plt.subplots(figsize=FIGSIZE)
for part in ["train", "val", "test"]:
    ax.bar(per_season.index, per_season[part], color=split_colors[part], label=f"{part} ({per_season[part].sum():,} rows)")
ax.set_title(f"Rows (driver-races) per season and split: train {TRAIN_START_YEAR}-{TRAIN_END_YEAR}, "
             f"validation {TRAIN_END_YEAR + 1}-{VAL_END_YEAR}, test {VAL_END_YEAR + 1}-{model_df['year'].max()}")
ax.set_xlabel("Season"); ax.set_ylabel("Rows (driver-races)")
ax.set_xticks(per_season.index); ax.tick_params(axis="x", rotation=90)
ax.grid(axis="y", alpha=0.3); ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, loc="upper left")
plt.show()

**What the chart shows:** each season belongs to exactly one split, in time order: 16 training seasons (blue), 3 validation seasons (orange, with the small 2020 in the middle) and the 3 seasons of the 2022 rules era as test (grey). The number of rows per season grows with the calendar (16 races in 2003, 24 in 2024).

### 9.5 The split: `X_train` … `y_test`
The feature matrices hold only `FEATURES` (no ids, no year, no target), with NaN left in place for the Pipeline (Section 10). `model_df` keeps the ids and the `split` column, so any row can be traced back to its race and driver.

In [ ]:
# 9.5 X / y per split (missing values stay NaN: the Pipeline fills them, fitted on train only)
train_mask, val_mask, test_mask = (model_df["split"] == s for s in ["train", "val", "test"])
X_train, y_train = model_df.loc[train_mask, FEATURES], model_df.loc[train_mask, TARGET]
X_val, y_val = model_df.loc[val_mask, FEATURES], model_df.loc[val_mask, TARGET]
X_test, y_test = model_df.loc[test_mask, FEATURES], model_df.loc[test_mask, TARGET]

# Checks: every row in exactly one split, splits in time order, no race in two splits, same columns
assert len(X_train) + len(X_val) + len(X_test) == len(model_df)
assert model_df.loc[train_mask, "date"].max() < model_df.loc[val_mask, "date"].min()
assert model_df.loc[val_mask, "date"].max() < model_df.loc[test_mask, "date"].min()
assert model_df.groupby("raceId")["split"].nunique().max() == 1
assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns) == FEATURES
# Sections 8-9 read clean and raw only: both unchanged
assert all(pd.util.hash_pandas_object(clean[n], index=True).sum() == clean_fingerprint_8[n] for n in clean)
assert all(pd.util.hash_pandas_object(raw[n], index=True).sum() == raw_fingerprint[n] for n in raw)
print("Checks passed: one split per row and per race, splits in time order, clean and raw unchanged.")

split_summary = pd.DataFrame({
    "seasons": [f"{TRAIN_START_YEAR}-{TRAIN_END_YEAR}", f"{TRAIN_END_YEAR + 1}-{VAL_END_YEAR}", f"{VAL_END_YEAR + 1}-{model_df['year'].max()}"],
    "races": [model_df.loc[m, "raceId"].nunique() for m in (train_mask, val_mask, test_mask)],
    "rows": [len(y_train), len(y_val), len(y_test)],
    "podiums": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "podium_%": [round(y.mean() * 100, 1) for y in (y_train, y_val, y_test)],
    "rows_with_any_missing_feature": [int(X.isna().any(axis=1).sum()) for X in (X_train, X_val, X_test)],
}, index=["train", "val", "test"])
split_summary

**What the table shows:** train has **6,363 rows (300 races)**, validation **1,195 (60 races)** and test **1,352 (68 races)**; the podium share is **14.1%, 15.1% and 15.1%**. Every check passes: each row is in exactly one split, no race is split between two parts, train < validation < test in time, and `clean` / `raw` are unchanged. 567 / 83 / 81 rows have at least one missing feature (62% of them are first rounds of a season, where there is no championship rank yet); they are kept and filled in the Pipeline.

**Hand-off to Section 10 (Models):** use `X_train, y_train` to fit, `X_val, y_val` to compare models and choose the threshold, and `X_test, y_test` **once**, at the very end. Feature groups for the ablation: `FEATURE_GROUPS`.

## 10. Models
**Goal:** train the required models on the same rows and features, improve each one in labelled attempts, choose the final model and its probability threshold **on validation**, and score the **test set once** at the end.

**Uses:** `X_train, y_train, X_val, y_val, X_test, y_test`, `FEATURES`, `evaluate()`, `results_table` (setup cell)

**Creates:** `fitted_models`, `val_probs`, `val_thresholds`, `best_name`, `best_pipeline`, `THRESHOLD`, `comparison`

**Models:** "podium if grid ≤ 3" rule (to beat) → logistic regression (official baseline, TA) → random forest → XGBoost → shallow neural network (Keras). Every model is a sklearn `Pipeline` with the step names `"imputer"`, `"scaler"`, `"model"` (code conventions 3.3), so all preprocessing is fitted **on train only**.

**Metrics (Decision 3):** PR-AUC is the main metric (podiums are only about 14% of rows), ROC-AUC secondary, plus F1, precision and recall at a threshold. **Threshold rule:** for every model the threshold with the best F1 **on validation** is used, and the same number is applied unchanged to test.

### 10.1 Helpers
One function fits a pipeline on train, picks its threshold on validation and adds its train and validation scores to `results_table` through the shared `evaluate()`. The imputer fills missing values with the **training** median and adds a 0/1 "was missing" column for each feature that has gaps (e.g. "no form history", "no qualifying time"), so the model can still see that a value was unknown.

In [ ]:
# 10.1 Helpers used by every model in Sections 10-13
fitted_models, val_probs, val_thresholds = {}, {}, {}

def make_preprocess(scale=True):
    """Pipeline steps: median imputer with missing-value indicators (+ standard scaler). Fitted on train only."""
    steps = [("imputer", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    return steps

def best_f1_threshold(y_true, y_prob):
    """Probability threshold with the highest F1 on validation."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return float(thresholds[np.argmax(f1[:-1])])

def fit_and_score(name, pipeline, threshold=None, verbose=True):
    """Fit on train; score train + validation with a validation-chosen threshold (or a fixed one); keep the model."""
    pipeline.fit(X_train, y_train)
    p_train = pipeline.predict_proba(X_train)[:, 1]
    p_val = pipeline.predict_proba(X_val)[:, 1]
    thr = best_f1_threshold(y_val, p_val) if threshold is None else threshold
    evaluate(name, y_train, p_train, "train", thr)
    row = evaluate(name, y_val, p_val, "val", thr)
    fitted_models[name], val_probs[name], val_thresholds[name] = pipeline, p_val, thr
    if verbose:
        print(f"{name:20s} val PR-AUC {row['pr_auc']:.4f} | ROC-AUC {row['roc_auc']:.4f} | F1 {row['f1']:.3f} "
              f"(threshold {thr:.3f}) | train PR-AUC {average_precision_score(y_train, p_train):.4f}")
    return row

def scores(split="val", names=None):
    """Results of one split (best PR-AUC first), optionally only some models."""
    t = pd.DataFrame(results_table).query("split == @split")
    if names is not None:
        t = t[t["model"].isin(names)]
    return t.drop_duplicates("model", keep="last").sort_values("pr_auc", ascending=False).reset_index(drop=True).round(4)

def val_pr_auc(pipeline, X_tr=None, y_tr=None, cols=None):
    """Fit a copy of `pipeline` (default: on train) and return its validation PR-AUC; used for tuning grids and experiments."""
    cols = FEATURES if cols is None else cols
    X_tr = X_train if X_tr is None else X_tr
    y_tr = y_train if y_tr is None else y_tr
    model = clone(pipeline).fit(X_tr[cols], y_tr)
    return average_precision_score(y_val, model.predict_proba(X_val[cols])[:, 1])

print(f"Train {X_train.shape}, validation {X_val.shape}; podium share {y_train.mean():.1%} / {y_val.mean():.1%}")

### 10.1b What the preprocessing does to the data
The Pipeline's preprocessing has two steps (fitted on the **training seasons only**): (1) the **imputer** fills each missing value with that feature's training median and adds a 0/1 "was missing" column for every feature that has gaps; (2) the **scaler** (for logistic regression and the neural network) shifts and stretches each column to mean 0 and standard deviation 1, so a feature counted in hundreds (experience) does not outweigh a 0/1 flag. Tree models (random forest) use only the imputer; XGBoost uses neither. The cell shows the effect on the training rows.

In [ ]:
# 10.1b Effect of the preprocessing steps on the training data (missing values, columns, ranges)
prep = Pipeline(make_preprocess(scale=True)).fit(X_train)
X_imp = prep.named_steps["imputer"].transform(X_train)
X_out = prep.transform(X_train)
n_ind = X_out.shape[1] - X_train.shape[1]
print(f"Before: {X_train.shape[1]} columns, {int(X_train.isna().sum().sum()):,} missing cells in {int(X_train.isna().any(axis=1).sum())} rows")
print(f"After imputer: {X_imp.shape[1]} columns ({n_ind} 'was missing' indicators), {int(np.isnan(X_imp).sum())} missing cells")
before = X_train.agg(["mean", "std", "min", "max"]).T
after = pd.DataFrame(X_out[:, :X_train.shape[1]], columns=FEATURES).agg(["mean", "std", "min", "max"]).T
effect = before.round(2).join(after.round(2), lsuffix="_before", rsuffix="_after")

fig, ax = plt.subplots(figsize=FIGSIZE)
order = before["std"].sort_values().index
y = np.arange(len(order))
ax.barh(y - 0.2, before.loc[order, "std"], height=0.4, color="#2a78d6", label="before scaling")
ax.barh(y + 0.2, after.loc[order, "std"], height=0.4, color="#eb6834", label="after imputer + scaler")
ax.set_xscale("log"); ax.set_yticks(y); ax.set_yticklabels(order); ax.tick_params(axis="y", labelsize=7)
ax.set_title("Spread of each feature before vs after preprocessing (training rows)")
ax.set_xlabel("Standard deviation (log scale)"); ax.set_ylabel("Feature")
ax.grid(axis="x", alpha=0.3); ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False)
plt.show()
effect.loc[["grid_position", "quali_q1_gap_pct", "drv_podium_rate_last10", "stand_con_points_before", "ctx_driver_races_before", "ctx_home_race"]]

**What the output shows:** before preprocessing the training rows have **1,400 missing cells in 567 rows**, spread over 11 features (no qualifying time, no form history, no championship rank yet); afterwards none are missing, and **11 "was missing" columns** are added (20 → 31 columns) so the models can still tell a filled value from a real one. The spreads differ by a factor of about **3,000** before scaling (constructor points before the race: standard deviation 105; pit-lane flag: 0.035); after scaling every feature has mean 0 and standard deviation 1 (orange bars), so no feature dominates a linear model or a neural network just because of its unit. The ranks, orders and shapes inside each feature do not change (scaling is a straight-line transform), which is why tree models do not need it.

### 10.2 Rule to beat: "podium if grid ≤ 3"
Cell 4.7 showed why this rule is strong (79.9% podium from pole in 2003–2021, chart 8.13a). As a "model", its score is `1 / grid_position` (pole = 1, 2nd = 0.5, 3rd = 0.33, …), so ROC-AUC and PR-AUC measure how well the **grid order alone** ranks the drivers, and the fixed threshold 1/3 means exactly "grid 1, 2 or 3".

In [ ]:
# 10.2 Rule baseline: podium if grid <= 3 (score = 1 / grid_position; fixed threshold 1/3)
class GridTop3Rule(ClassifierMixin, BaseEstimator):
    """'Podium if grid <= 3' with the sklearn interface (nothing is learned)."""
    def fit(self, X, y):
        self.classes_ = np.array([0, 1])
        return self
    def predict_proba(self, X):
        score = 1 / np.asarray(X["grid_position"], dtype=float)
        return np.column_stack([1 - score, score])

fit_and_score("Rule_grid_top3", Pipeline([("model", GridTop3Rule())]), threshold=1 / 3);

**What the output shows:** the grid order alone already reaches a validation **PR-AUC of 0.683** and ROC-AUC 0.917; the fixed rule "grid 1–3" gets F1 0.719 (precision 0.721, recall 0.717). Every model below must beat these numbers to be worth it.

### 10.3 Logistic regression (official baseline): 3 attempts
**How it works:** it gives each (scaled) feature a weight, adds them up into a score, and turns the score into a probability with the sigmoid curve (score 0 → 50%). The weights are learned by making the predicted probabilities as close as possible to the real 0/1 podiums (log-loss), with a penalty on large weights (regularisation, strength `1/C`). It is the simplest reasonable model for a yes/no target, which is why the TA asked for it as the baseline.

- **Attempt 1 – `LR_baseline`:** median imputer + missing indicators, standard scaler, default regularisation (C = 1).
- **Attempt 2 – `LR_v2_balanced`:** same, with `class_weight="balanced"` (podium rows weigh about 6× more).
- **Attempt 3 – `LR_v3_tuned`:** regularisation strength C chosen on validation.

In [ ]:
# 10.3 Logistic regression attempts
def make_lr(C=1.0, class_weight=None):
    return Pipeline(make_preprocess(scale=True) +
                    [("model", LogisticRegression(C=C, class_weight=class_weight, max_iter=2000, random_state=SEED))])

fit_and_score("LR_baseline", make_lr())                                  # Attempt 1
fit_and_score("LR_v2_balanced", make_lr(class_weight="balanced"))        # Attempt 2

# Attempt 3: tune C on validation (PR-AUC)
lr_grid = pd.DataFrame([{"C": C, "val_pr_auc": val_pr_auc(make_lr(C=C))} for C in [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10]])
best_C = float(lr_grid.loc[lr_grid["val_pr_auc"].idxmax(), "C"])
print("\nAttempt 3 grid (validation PR-AUC):\n" + lr_grid.round(4).to_string(index=False))
fit_and_score("LR_v3_tuned", make_lr(C=best_C));

**What the output shows:** logistic regression beats the grid rule clearly (validation PR-AUC **0.726** for the baseline vs 0.683). Class weights do **not** help PR-AUC (0.718; they shift the probabilities up but hardly change the ranking; the threshold chosen on validation already handles the imbalance). Regularisation strength matters little: the grid is flat from C = 0.3 upwards (0.726–0.727), so the tuned model (0.727) is practically the baseline. Train PR-AUC (0.690) is even a bit lower than validation → no overfitting (a linear model is simple).

**Limitation:** a linear model on scaled features cannot represent the non-linear patterns seen in Section 8 (e.g. podium rate falling steeply over the first grid slots, experience rising then falling, chart 8.13f), and its coefficients are hard to read because the features are strongly correlated (heatmap 8.14).

### 10.4 Random forest: 3 attempts
**How it works:** it grows many decision trees (here 300); each tree asks a series of yes/no questions about the features (e.g. "grid ≤ 3?", "team podium share > 0.3?") and ends in a leaf with a podium share. Every tree sees a different random sample of rows and only a random subset of features at each split, so the trees make different mistakes; the forest's probability is the average over all trees. It can learn non-linear patterns and interactions without scaling.

- **Attempt 1 – `RF_v1`:** 300 trees, sklearn defaults (trees grown until every leaf is pure).
- **Attempt 2 – `RF_v2_tuned`:** minimum leaf size and features per split chosen on validation (limits overfitting).
- **Attempt 3 – `RF_v3_balanced`:** the tuned forest with `class_weight="balanced_subsample"`.

Trees do not need scaling, so the pipeline has only the imputer.

In [ ]:
# 10.4 Random forest attempts
def make_rf(min_samples_leaf=1, max_features="sqrt", class_weight=None, n_estimators=300):
    return Pipeline(make_preprocess(scale=False) +
                    [("model", RandomForestClassifier(n_estimators=n_estimators, min_samples_leaf=min_samples_leaf,
                                                      max_features=max_features, class_weight=class_weight,
                                                      random_state=SEED, n_jobs=-1))])

fit_and_score("RF_v1", make_rf())                                        # Attempt 1

# Attempt 2: tune leaf size and features per split on validation (PR-AUC)
rf_grid = pd.DataFrame([{"min_samples_leaf": leaf, "max_features": mf, "val_pr_auc": val_pr_auc(make_rf(leaf, mf))}
                        for leaf in [1, 5, 10, 20, 50] for mf in ["sqrt", 0.5]])
best_rf = rf_grid.loc[rf_grid["val_pr_auc"].idxmax()]
rf_params = {"min_samples_leaf": int(best_rf["min_samples_leaf"]), "max_features": best_rf["max_features"]}
print("\nAttempt 2 grid (validation PR-AUC):\n" + rf_grid.round(4).to_string(index=False))
fit_and_score("RF_v2_tuned", make_rf(**rf_params))
fit_and_score("RF_v3_balanced", make_rf(**rf_params, class_weight="balanced_subsample"));   # Attempt 3

**What the output shows:** the default forest **overfits** (train PR-AUC 1.000, validation 0.719, below logistic regression). Forcing bigger leaves fixes this: with at least 20 rows per leaf and √(features) per split the tuned forest reaches validation **PR-AUC 0.748** (train 0.766, a small gap), the best scikit-learn model. Class weights again do not help PR-AUC (0.742).

**Limitation:** a forest averages many trees, so single predictions are hard to explain without XAI (Section 12); probabilities from forests with large leaves are smooth but not perfectly calibrated (fine for ranking and a validation-chosen threshold).

### 10.5 XGBoost: 3 attempts
**How it works:** gradient boosting builds small trees **one after another**; each new tree is fitted to the errors the trees before it still make, and its answer is added with a small weight (the learning rate). After a few hundred rounds the sum of all trees gives the log-odds of a podium. Depth, learning rate and minimum child weight control how much each tree can memorise.

XGBoost handles missing values itself (it learns which branch NaN should follow), which is why the team chose it (code conventions 3.1); so its pipeline has only the `"model"` step (no imputer, no scaler).
- **Attempt 1 – `XGB_v1`:** 300 trees, depth 4, learning rate 0.05.
- **Attempt 2 – `XGB_v2_tuned`:** depth, minimum child weight and learning rate chosen on validation.
- **Attempt 3 – `XGB_v3_weighted`:** the tuned model with `scale_pos_weight` = non-podium / podium rows in train (class weight).

In [ ]:
# 10.5 XGBoost attempts (missing values handled natively -> no imputer)
def make_xgb(max_depth=4, learning_rate=0.05, min_child_weight=1, scale_pos_weight=1.0, n_estimators=300):
    return Pipeline([("model", XGBClassifier(n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate,
                                             min_child_weight=min_child_weight, subsample=0.8, colsample_bytree=0.8,
                                             scale_pos_weight=scale_pos_weight, eval_metric="aucpr",
                                             random_state=SEED, n_jobs=-1))])

fit_and_score("XGB_v1", make_xgb())                                      # Attempt 1

# Attempt 2: tune on validation (PR-AUC)
xgb_grid = pd.DataFrame([{"max_depth": d, "min_child_weight": w, "learning_rate": lr,
                          "val_pr_auc": val_pr_auc(make_xgb(d, lr, w))}
                         for d in [2, 3, 4, 6] for w in [1, 10] for lr in [0.03, 0.1]])
best_xgb = xgb_grid.loc[xgb_grid["val_pr_auc"].idxmax()]
xgb_params = {"max_depth": int(best_xgb["max_depth"]), "min_child_weight": int(best_xgb["min_child_weight"]),
              "learning_rate": float(best_xgb["learning_rate"])}
print("\nAttempt 2 grid (validation PR-AUC), best 5:\n" + xgb_grid.sort_values("val_pr_auc", ascending=False).head().round(4).to_string(index=False))
fit_and_score("XGB_v2_tuned", make_xgb(**xgb_params))

neg_pos = float((y_train == 0).sum() / (y_train == 1).sum())
fit_and_score("XGB_v3_weighted", make_xgb(**xgb_params, scale_pos_weight=neg_pos));   # Attempt 3

**What the output shows:** the first XGBoost **overfits** (train PR-AUC 0.894, validation 0.706, below logistic regression). Tuning on validation picks very **shallow trees** (depth 2, minimum child weight 1, learning rate 0.03): validation PR-AUC **0.734** with train 0.729, so no gap any more. The class weight (`scale_pos_weight` = non-podium / podium rows) does not help (0.732), as for the other models. The tuned XGBoost is close to, but below, the tuned random forest (0.748).

**Limitation:** boosted trees can overfit small data if the trees are deep or the learning rate high (seen in `XGB_v1`); like the forest, single predictions need XAI to explain.

### 10.6 Shallow neural network (Keras): 3 attempts
**How it works:** a feed-forward network with 1–2 hidden layers: each hidden unit takes a weighted sum of the scaled inputs and passes it through a ReLU (negative → 0); the output unit turns the last layer into a probability with a sigmoid. The weights are learned with the Adam optimiser by lowering the log-loss on mini-batches of 64 rows, epoch after epoch. **Dropout** randomly switches off a share of hidden units during training (so the net cannot rely on single units), and **early stopping** keeps the weights from the epoch with the lowest held-out loss.

A small wrapper (`KerasPodiumNet`) gives the Keras model the sklearn interface (`fit` / `predict_proba`), so it sits in the same `Pipeline` (imputer + scaler + model), is scored with the same helpers and can be explained in Section 12. **Early stopping** watches the loss on the **last 15% of the training rows** (the latest training races, because the rows are in date order), so the validation seasons stay untouched for comparing models. Seeds are fixed (`keras.utils.set_random_seed(SEED)`).
- **Attempt 1 – `NN_v1`:** 1 hidden layer of 16 units, no dropout, learning rate 0.001.
- **Attempt 2 – `NN_v2_dropout`:** 2 hidden layers (64, 32) with 30% dropout.
- **Attempt 3 – `NN_v3_tuned`:** layer sizes, dropout and learning rate chosen on validation (8 combinations).

In [ ]:
# 10.6 Shallow neural network (Keras) with the sklearn interface
class KerasPodiumNet(ClassifierMixin, BaseEstimator):
    """Feed-forward net with 1-2 hidden layers, dropout and early stopping, usable inside a sklearn Pipeline."""
    def __init__(self, hidden=(32,), dropout=0.0, learning_rate=1e-3, batch_size=64, epochs=200, patience=15):
        self.hidden, self.dropout, self.learning_rate = hidden, dropout, learning_rate
        self.batch_size, self.epochs, self.patience = batch_size, epochs, patience

    def fit(self, X, y):
        keras.utils.set_random_seed(SEED)
        X, y = np.asarray(X, dtype="float32"), np.asarray(y, dtype="float32")
        layers = [keras.Input(shape=(X.shape[1],))]
        for units in self.hidden:
            layers.append(keras.layers.Dense(units, activation="relu"))
            if self.dropout > 0:
                layers.append(keras.layers.Dropout(self.dropout))
        layers.append(keras.layers.Dense(1, activation="sigmoid"))
        self.model_ = keras.Sequential(layers)
        self.model_.compile(optimizer=keras.optimizers.Adam(learning_rate=self.learning_rate), loss="binary_crossentropy")
        stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=self.patience, restore_best_weights=True)
        # validation_split takes the LAST 15% of the rows (= the latest training races); validation seasons stay unused
        self.history_ = self.model_.fit(X, y, validation_split=0.15, epochs=self.epochs, batch_size=self.batch_size,
                                        callbacks=[stop], verbose=0)
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        p = self.model_.predict(np.asarray(X, dtype="float32"), verbose=0).ravel()
        return np.column_stack([1 - p, p])

def make_nn(hidden=(32,), dropout=0.0, learning_rate=1e-3):
    return Pipeline(make_preprocess(scale=True) + [("model", KerasPodiumNet(hidden, dropout, learning_rate))])

fit_and_score("NN_v1", make_nn(hidden=(16,)))                            # Attempt 1
fit_and_score("NN_v2_dropout", make_nn(hidden=(64, 32), dropout=0.3))    # Attempt 2

# Attempt 3: tune on validation (PR-AUC)
nn_grid = pd.DataFrame([{"hidden": h, "dropout": dr, "learning_rate": lr, "val_pr_auc": val_pr_auc(make_nn(h, dr, lr))}
                        for h in [(32,), (64, 32)] for dr in [0.2, 0.4] for lr in [1e-3, 3e-4]])
best_nn = nn_grid.loc[nn_grid["val_pr_auc"].idxmax()]
nn_params = {"hidden": best_nn["hidden"], "dropout": float(best_nn["dropout"]), "learning_rate": float(best_nn["learning_rate"])}
print("\nAttempt 3 grid (validation PR-AUC):\n" + nn_grid.round(4).to_string(index=False))
fit_and_score("NN_v3_tuned", make_nn(**nn_params));

**What the output shows:** the network improves from 1 small layer (`NN_v1`, validation PR-AUC 0.692) to 2 layers with dropout (`NN_v2_dropout`, 0.708). The grid picks **1 hidden layer of 32 units, 20% dropout, learning rate 0.001** (`NN_v3_tuned`, 0.715); train and validation scores are close, so dropout and early stopping keep it from overfitting. It stays **below** logistic regression (0.726) and the tuned forest (0.748): with this little data and hand-made features, a neural network has nothing extra to learn. (The red "CUDA error" line on Kaggle only says that no GPU is attached; the network trains on the CPU.)

**Limitation:** with about 5,400 training rows after the early-stopping split and 20 mostly hand-made features, a neural network has little room to beat simpler models; results move a little with the seed, which is why the seed is fixed.

In [ ]:
# 10.7 Training vs early-stopping loss of the tuned network (Keras history)
history = fitted_models["NN_v3_tuned"].named_steps["model"].history_.history
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(range(1, len(history["loss"]) + 1), history["loss"], color="#2a78d6", linewidth=2, label="training loss")
ax.plot(range(1, len(history["val_loss"]) + 1), history["val_loss"], color="#eb6834", linewidth=2,
        label="early-stopping loss (last 15% of training rows)")
best_epoch = int(np.argmin(history["val_loss"])) + 1
ax.axvline(best_epoch, color="#52514e", linestyle="--", linewidth=1, label=f"best epoch ({best_epoch}), weights restored")
ax.set_title("NN_v3_tuned: loss per epoch")
ax.set_xlabel("Epoch"); ax.set_ylabel("Binary cross-entropy")
ax.grid(axis="y", alpha=0.3); ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False)
plt.show()

**What the chart shows:** both losses fall fast in the first 5 epochs and are almost flat after about 15. The early-stopping loss is lowest at **epoch 35**; training ran 15 more epochs without improvement (patience 15) and then stopped at epoch 50, restoring the epoch-35 weights. The early-stopping loss is **lower** than the training loss because dropout is switched on while the training loss is measured (it makes the training task harder) and off when the held-out rows are scored; there is no sign of overfitting (the held-out loss does not rise).

### 10.8 Choosing the final model and the threshold (validation only)
All attempts are compared on **validation PR-AUC** (Decision 3). The best one becomes `best_pipeline`; its validation-chosen threshold becomes `THRESHOLD` (used unchanged on test and in `predict_podium`). As an alternative decision rule we also check "**the 3 highest probabilities in each race** = podium" on validation, because every race has exactly 3 podium places.

In [ ]:
# 10.8 Validation comparison of every attempt -> final model and threshold
val_table = scores("val")
best_name = val_table.loc[0, "model"]
best_pipeline = fitted_models[best_name]
THRESHOLD = val_thresholds[best_name]

# Alternative rule: top 3 probabilities per race = podium (validation only)
top3 = (model_df.loc[X_val.index, ["raceId"]].assign(p=val_probs[best_name], y=y_val.values)
        .assign(rank=lambda d: d.groupby("raceId")["p"].rank(ascending=False, method="first")))
top3_pred = (top3["rank"] <= 3).astype(int)
thr_pred = (val_probs[best_name] >= THRESHOLD).astype(int)
rule_compare = pd.DataFrame({
    "rule": [f"probability >= THRESHOLD ({THRESHOLD:.3f})", "top 3 probabilities per race"],
    "f1": [f1_score(y_val, thr_pred), f1_score(y_val, top3_pred)],
    "precision": [precision_score(y_val, thr_pred), precision_score(y_val, top3_pred)],
    "recall": [recall_score(y_val, thr_pred), recall_score(y_val, top3_pred)],
    "predicted_podiums": [int(thr_pred.sum()), int(top3_pred.sum())]}).round(3)

print(f"Final model (best validation PR-AUC): {best_name}; THRESHOLD = {THRESHOLD:.3f}")
print(rule_compare.to_string(index=False))
val_table[["model", "pr_auc", "roc_auc", "f1", "precision", "recall", "threshold"]]

**What the output shows:** the best attempt on validation is **`RF_v2_tuned`** (PR-AUC **0.748**), ahead of `RF_v3_balanced` (0.742), `XGB_v2_tuned` (0.734), the logistic regressions (0.726–0.727) and `NN_v3_tuned` (0.715); every model beats the grid rule (0.683). `THRESHOLD` = **0.374**: on validation it predicts 198 podiums (180 real), F1 0.735, precision 0.702, recall 0.772; "top 3 per race" gets F1 0.717.

**Decision 21 — Final model and threshold:** the final model is **`RF_v2_tuned`**, the attempt with the **highest validation PR-AUC**; the threshold is the one with the best validation F1 for that model (**`THRESHOLD` = 0.374**), used unchanged on test and in `predict_podium`. "Top 3 per race" is shown as an alternative rule but not used, because the inference function scores **one driver at a time** (a single record has no race to rank within).

**Why:** PR-AUC is the agreed main metric (Decision 3) and is measured on validation only; the threshold must also be fixed before the test set is opened. The forest is also the simplest model to retrain and explain among the top three.

**Evidence:** the validation table above (all 13 attempts) and the threshold comparison above.

In [ ]:
# 10.9 Precision-recall curves on validation: best attempt of each model family + the grid rule
families = {"Rule": "Rule_grid_top3", "LR": "LR_", "RF": "RF_", "XGB": "XGB_", "NN": "NN_"}
best_of_family = {fam: scores("val", [m for m in fitted_models if m.startswith(prefix)]).loc[0, "model"]
                  for fam, prefix in families.items()}
family_colors = {"Rule": "#52514e", "LR": "#2a78d6", "RF": "#eb6834", "XGB": "#1b9e77", "NN": "#9467bd"}

fig, ax = plt.subplots(figsize=FIGSIZE)
for fam, name in best_of_family.items():
    precision, recall, _ = precision_recall_curve(y_val, val_probs[name])
    ax.plot(recall, precision, color=family_colors[fam], linewidth=2,
            label=f"{name} (PR-AUC {average_precision_score(y_val, val_probs[name]):.3f})")
ax.axhline(y_val.mean(), color="#52514e", linestyle=":", linewidth=1, label=f"no skill ({y_val.mean():.1%} podiums)")
ax.set_title(f"Precision-recall curves, validation {TRAIN_END_YEAR + 1}-{VAL_END_YEAR}")
ax.set_xlabel("Recall (share of real podiums found)"); ax.set_ylabel("Precision (share of predicted podiums that were real)")
ax.grid(axis="y", alpha=0.3); ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, loc="lower left")
plt.show()

**What the chart shows:** every model's curve lies above the grid rule over most of the recall range: the models keep a higher precision when they try to find more of the real podiums. The curves of the learned models are close to each other, so the choice between them is a matter of small differences (see the table in 10.8). The dotted line is a model with no skill (precision = podium share).

In [ ]:
# 10.10 ROC curves on validation (same models)
fig, ax = plt.subplots(figsize=FIGSIZE)
for fam, name in best_of_family.items():
    fpr, tpr, _ = roc_curve(y_val, val_probs[name])
    ax.plot(fpr, tpr, color=family_colors[fam], linewidth=2, label=f"{name} (ROC-AUC {roc_auc_score(y_val, val_probs[name]):.3f})")
ax.plot([0, 1], [0, 1], color="#52514e", linestyle=":", linewidth=1, label="no skill")
ax.set_title(f"ROC curves, validation {TRAIN_END_YEAR + 1}-{VAL_END_YEAR}")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate (recall)")
ax.grid(axis="y", alpha=0.3); ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, loc="lower right")
plt.show()

**What the chart shows:** all ROC-AUCs are high (above 0.9), including the grid rule, because most non-podium rows (the back of the grid) are easy to rule out. This is why ROC-AUC is only the secondary metric: it hides how hard it is to pick the 3 podium drivers among the front runners, which PR-AUC shows (Decision 3).

### 10.11 Final evaluation on the test set (opened once)
Everything above was decided on train and validation. Now every fitted attempt is scored **once** on the test seasons (2022–2024), each with its own validation threshold. No model, feature or threshold is changed after this cell. The table shows all five metrics on **train, validation and test** (train = how well the model fits the seasons it learned from; a big train–validation gap means overfitting).

In [ ]:
# 10.11 Test set: scored once, thresholds from validation
test_probs = {}
for name, pipeline in fitted_models.items():
    test_probs[name] = pipeline.predict_proba(X_test)[:, 1]
    evaluate(name, y_test, test_probs[name], "test", val_thresholds[name])

comparison = (pd.DataFrame(results_table).drop_duplicates(["model", "split"], keep="last")
              .pivot(index="model", columns="split", values=["pr_auc", "roc_auc", "f1", "precision", "recall"]))
comparison = comparison.reindex(columns=["train", "val", "test"], level=1).sort_values(("pr_auc", "val"), ascending=False).round(3)
print(f"Final model: {best_name} (THRESHOLD {THRESHOLD:.3f}) -> test PR-AUC {average_precision_score(y_test, test_probs[best_name]):.3f}")
comparison

**What the table shows:** the final model `RF_v2_tuned` reaches a test **PR-AUC of 0.719** (train 0.766, validation 0.748), ROC-AUC 0.939, F1 0.682 at `THRESHOLD`. Logistic regression is just as good on test (0.719), XGBoost and the network a bit lower (`XGB_v2_tuned` 0.701, `NN_v3_tuned` 0.698), and the grid rule lowest (**0.647**, validation 0.683). So the learned models still beat the rule by about **0.07** PR-AUC on seasons they never saw. The train column shows the overfitting of the untuned trees (`RF_v1` train PR-AUC 1.000, `XGB_v1` 0.894) and the small gaps of the tuned models.

**How to read it:** the test seasons are the new 2022 rules era (Decision 10), so a drop from validation to test is expected (0.748 → 0.719 for the final model); the ranking of the models stays about the same, and the final model still beats the grid rule.

In [ ]:
# 10.12 Confusion matrices of the final model at THRESHOLD: validation and test
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)
for ax, (split, y_true, p) in zip(axes, [("validation", y_val, val_probs[best_name]), ("test", y_test, test_probs[best_name])]):
    cm = confusion_matrix(y_true, (p >= THRESHOLD).astype(int))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["no podium", "podium"], yticklabels=["no podium", "podium"])
    ax.set_title(f"{split} ({len(y_true):,} rows)"); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
fig.suptitle(f"Confusion matrices: {best_name}, threshold {THRESHOLD:.3f}")
plt.tight_layout(); plt.show()

**What the chart shows:** how the final model's yes/no answers split into correct podiums (bottom right), missed podiums (bottom left), false alarms (top right) and correct "no podium" (top left), on validation and on test. For `RF_v2_tuned`: validation 139 of 180 podiums found with 59 false alarms; test 146 of 204 found with 78 false alarms. Because every race has exactly 3 podiums, the model predicts roughly as many podiums as there are, and most errors are swaps among the front runners (a driver predicted on the podium who finished 4th–5th, and the other way round).

## 11. Experiments (Ablation + Preprocessing Order)
**Goal:** (1) **ablation** — remove one feature group at a time, retrain, and measure the drop; (2) two **preprocessing-order experiments** — what changes if a preprocessing step is done **before** the split instead of after it; (3) the **"train both ways"** experiment that backs up Decision 4 (2003+ vs 1994+).

**Uses:** `fitted_models`, `FEATURE_GROUPS`, `X_train … y_val`, `make_lr()`, `val_pr_auc()`, the Section 8 feature tables (`grid_feats`, `quali_feats`, …)

**Creates:** `EXP_MODEL`, `ablation`, `exp_preprocess`, `exp_resample`, `exp_years`

**Rules:** every experiment is scored on **validation** (PR-AUC); the test set is not used here. The experiments use **`EXP_MODEL` = the best scikit-learn model on validation** (`RF_v2_tuned` in our run): each experiment retrains many times, and the aim is to compare settings with one fixed, fast model, not to re-score the final model.

In [ ]:
# 11.0 The model used for all experiments: best scikit-learn attempt (LR or RF) on validation
sklearn_attempts = [m for m in fitted_models if m.startswith(("LR_", "RF_"))]
EXP_MODEL = scores("val", sklearn_attempts).loc[0, "model"]
exp_pipeline = fitted_models[EXP_MODEL]
EXP_REFERENCE = average_precision_score(y_val, val_probs[EXP_MODEL])
print(f"Experiments use {EXP_MODEL}: validation PR-AUC with all features = {EXP_REFERENCE:.4f}")

### 11.1 Ablation: remove one feature group at a time
For each of the 6 groups (prefixes, Section 8.11), `EXP_MODEL` is retrained on train without that group and scored on validation. A big drop = the group carries information the other groups do not have.

In [ ]:
# 11.1 Ablation: retrain without one feature group at a time (validation PR-AUC)
rows = [{"removed_group": "none (all features)", "n_features": len(FEATURES), "val_pr_auc": EXP_REFERENCE}]
for group, cols in FEATURE_GROUPS.items():
    kept = [f for f in FEATURES if f not in cols]
    rows.append({"removed_group": group, "n_features": len(kept), "val_pr_auc": val_pr_auc(exp_pipeline, cols=kept)})
ablation = pd.DataFrame(rows)
ablation["drop_vs_all"] = EXP_REFERENCE - ablation["val_pr_auc"]
ablation.round(4)

In [ ]:
# 11.2 Chart: drop in validation PR-AUC when a group is removed
abl = ablation.iloc[1:].sort_values("drop_vs_all", ascending=False)
fig, ax = plt.subplots(figsize=FIGSIZE)
colors = ["#2a78d6" if d >= 0 else "#eb6834" for d in abl["drop_vs_all"]]
ax.bar(abl["removed_group"], abl["drop_vs_all"], color=colors)
ax.axhline(0, color="#52514e", linewidth=1)
ax.set_title(f"Ablation: drop in validation PR-AUC when one feature group is removed ({EXP_MODEL})")
ax.set_xlabel("Feature group removed"); ax.set_ylabel("PR-AUC drop (all features minus without group)")
ax.grid(axis="y", alpha=0.3); ax.spines[["top", "right"]].set_visible(False)
plt.show()

**What the table and chart show:** removing the **grid** group costs by far the most (PR-AUC 0.748 → 0.716, a drop of **0.033**), even though qualifying position is 0.97 correlated with grid position (8.14): the grid slot after penalties is the information closest to the race start. Driver form (−0.009), constructor form (−0.008), qualifying (−0.005) and standings (−0.005) each cost little on their own, because they overlap (they all describe "how strong are this car and driver", heatmap 8.14): when one is removed, the others carry most of the same information. Without the **context** group the model is slightly **better** (+0.004, orange): within noise for one validation set of 1,195 rows (180 podiums), and in line with Q2 (no home advantage) and the weak context chart 8.13f. We keep it (it is cheap and the difference is not reliable), but it is the first candidate to drop in a later milestone.

### 11.3 Preprocessing-order experiment 1: fill missing values and scale **before** vs **after** the split
- **After the split (our Pipeline):** the imputer's medians and the scaler's means / standard deviations are learned from the **training seasons only**.
- **Before the split (wrong):** they are learned from **all** of `model_df` (train + validation + test), then the model is trained on train and scored on validation. The validation and test seasons have then shaped the preprocessing: information from the future leaks into training.

Model: logistic regression (`make_lr()`), because it depends on scaling, so the effect is easiest to see.

In [ ]:
# 11.3 Experiment 1: imputer + scaler fitted after the split (train only) vs before the split (all seasons)
X_all = model_df[FEATURES]
prep_after = Pipeline(make_preprocess(scale=True)).fit(X_train)            # correct: train only
prep_before = Pipeline(make_preprocess(scale=True)).fit(X_all)             # wrong: train + validation + test

results = []
for label, prep in [("after the split (train only)", prep_after), ("before the split (all seasons)", prep_before)]:
    lr = LogisticRegression(max_iter=2000, random_state=SEED).fit(prep.transform(X_train), y_train)
    p = lr.predict_proba(prep.transform(X_val))[:, 1]
    results.append({"preprocessing fitted": label, "val_pr_auc": average_precision_score(y_val, p), "val_roc_auc": roc_auc_score(y_val, p)})
exp_preprocess = pd.DataFrame(results)

# What changed in the learned statistics
stats = pd.DataFrame({"median_train": prep_after.named_steps["imputer"].statistics_,
                      "median_all": prep_before.named_steps["imputer"].statistics_}, index=FEATURES)
stats = stats[stats["median_train"] != stats["median_all"]]
print(f"Imputer medians that change when the validation and test seasons are included: {len(stats)} of {len(FEATURES)}")
print(stats.round(2).to_string())
exp_preprocess.round(4)

**What the output shows:** fitting the imputer and scaler on all seasons changes **9 of the 20** learned medians (e.g. constructor points before the race 21 → 26, experience 68 → 82 starts, field size 22 → 20) and every scaler mean, so the validation and test seasons influence how the training data is transformed. Here the score barely moves (validation PR-AUC 0.7262 after vs 0.7264 before): the imputer only fills the 1–2% missing values, and a logistic regression can absorb a shifted scaling with its coefficients. The size of the effect is small **in this dataset**, but the order still matters: with a "before" pipeline the validation score is no longer a clean estimate, and the leak gets bigger when the later seasons differ more (e.g. after a rule change). This is why all preprocessing sits inside the Pipeline and is fitted on train only (Section 10).

### 11.4 Preprocessing-order experiment 2: oversampling **before** vs **after** the split
Oversampling copies podium rows until both classes are equally frequent (a simple form of SMOTE-style resampling, done with numpy, no extra library).
- **No resampling:** the reference.
- **After the split (correct):** only the **training** rows are oversampled; validation stays as it is.
- **Before the split (wrong):** the train + validation seasons are oversampled **first** and then split by season, so the validation set is oversampled too (about 50% podiums instead of 15%, with copied rows).

In [ ]:
# 11.4 Experiment 2: random oversampling of podium rows, after vs before the split
def oversample(X, y, seed=SEED):
    """Copy random podium rows until both classes have the same count."""
    rng = np.random.default_rng(seed)
    pos, neg = np.flatnonzero(y.values == 1), np.flatnonzero(y.values == 0)
    extra = rng.choice(pos, size=len(neg) - len(pos), replace=True)
    idx = np.concatenate([np.arange(len(y)), extra])
    return X.iloc[idx], y.iloc[idx]

rows = [{"resampling": "none", "val_rows": len(y_val), "val_podium_share": y_val.mean(), "val_pr_auc": EXP_REFERENCE}]

X_os, y_os = oversample(X_train, y_train)                                      # after the split: train only
rows.append({"resampling": "after the split (train only)", "val_rows": len(y_val), "val_podium_share": y_val.mean(),
             "val_pr_auc": val_pr_auc(exp_pipeline, X_os, y_os)})

pool = model_df[model_df["year"] <= VAL_END_YEAR].reset_index(drop=True)      # before the split: train + validation seasons
X_pool, y_pool = oversample(pool[FEATURES], pool[TARGET])
pool_years = pool["year"].loc[X_pool.index].values                             # each copy keeps the season of its original row
tr, va = pool_years <= TRAIN_END_YEAR, pool_years > TRAIN_END_YEAR
model_wrong = clone(exp_pipeline).fit(X_pool[tr], y_pool[tr])
p_wrong = model_wrong.predict_proba(X_pool[va])[:, 1]
rows.append({"resampling": "before the split (train + validation)", "val_rows": int(va.sum()), "val_podium_share": y_pool[va].mean(),
             "val_pr_auc": average_precision_score(y_pool[va], p_wrong)})
exp_resample = pd.DataFrame(rows)
exp_resample.round(4)

**What the table shows:** oversampling **only the training rows** does not help (validation PR-AUC 0.739 vs 0.748 without resampling; like the class weights in Section 10 it changes the probabilities more than the ranking). Oversampling **before** the split makes the validation PR-AUC jump to **0.935**: not because the model is better, but because the "validation" set now has **52% podiums** in 2,104 rows instead of 15% in 1,195 (PR-AUC depends on the share of positives) and contains copies of the same rows. This number would be a false promise: on real races only about 15% of drivers finish on the podium. Resampling, like every preprocessing step, must happen **after** the split, on train only — and here it brings no gain, so the final models do not use it.

### 11.5 "Train both ways" (backs up Decision 4): train from 2003 vs from the first qualifying season
Decision 4 starts `model_df` in 2003 because qualifying data is complete only from 2003 (cell 4.1). Here we test it: the same features are built for the seasons from the first year with qualifying data (1994) using the **same Section 8 feature tables**, and `EXP_MODEL` and logistic regression are trained on **1994–2018** vs **2003–2018**; both are scored on the same validation seasons (2019–2021).

In [ ]:
# 11.5 Train both ways: training seasons from the first qualifying year vs from TRAIN_START_YEAR (same validation set)
QUALI_FIRST_YEAR = int(race_order.loc[race_order["raceId"].isin(clean["qualifying"]["raceId"]), "year"].min())

wide = hist.loc[hist["year"].between(QUALI_FIRST_YEAR, TRAIN_END_YEAR), model_keys + ["podium"]].reset_index(drop=True)
for frame, cols in [(grid_feats, grid_cols), (quali_feats, ["quali_position", "quali_q1_gap_pct"]), (drv_feats, drv_cols),
                    (stand_feats, stand_cols), (ctx_feats, ctx_cols)]:
    wide = add_features(wide, frame, cols)
wide = wide.merge(con_feats, on=["raceId", "constructorId"], how="left", validate="many_to_one")
wide["quali_grid_penalty"] = wide["grid_position"] - wide["quali_position"]

# the 2003+ part of `wide` must equal the training rows of model_df (same code, same values)
check = wide[wide["year"] >= TRAIN_START_YEAR].set_index(grain_key)[FEATURES]
ref = model_df[train_mask].set_index(grain_key)[FEATURES].loc[check.index]
assert len(check) == len(X_train) and np.allclose(check.values, ref.values, equal_nan=True)

rows = []
for start in [QUALI_FIRST_YEAR, TRAIN_START_YEAR]:
    part = wide[wide["year"] >= start]
    for name, pipe in [(EXP_MODEL, exp_pipeline), ("LR_baseline", fitted_models["LR_baseline"])]:
        rows.append({"training seasons": f"{start}-{TRAIN_END_YEAR}", "model": name, "train_rows": len(part),
                     "train_rows_without_qualifying": f"{part['quali_position'].isna().mean():.1%}",
                     "val_pr_auc": val_pr_auc(pipe, part[FEATURES], part["podium"])})
exp_years = pd.DataFrame(rows)
print(f"First season with qualifying data: {QUALI_FIRST_YEAR}")
exp_years.round(4)

**What the table shows:** training from 1994 adds 3,310 rows (9,673 vs 6,363), but **18.6%** of them have no qualifying data (0.3% from 2003), and those seasons come from a different era (more retirements, cell 4.8). With fewer but complete rows the 2003 start scores **better for both models**: `RF_v2_tuned` 0.748 vs 0.738, logistic regression 0.726 vs 0.720 (validation PR-AUC). The extra seasons do not help, so **Decision 4 is now backed by an experiment**, not only by the coverage chart (4.1).

## 12. Explainability (XAI)
**Goal:** explain what the models learned — **globally** (which features matter overall: permutation importance + SHAP) and **locally** (why one driver got his probability: LIME + a SHAP waterfall).

**Uses:** `best_pipeline`, `best_name`, `fitted_models`, `X_train`, `X_val`, `y_val`, `val_probs`

**Creates:** `perm_importance`, `shap_values_val`, `surprise_row`, `lime_explainer`

**Important:** "important to the model" does **not** mean "causes podiums". These methods show which inputs the model **relies on** to make its guesses; the features are correlated with each other (heatmap 8.14) and with things the data does not contain (car performance, money, team quality). Everything here is computed on **validation** rows, never on test.

### 12.1 Permutation importance (global, final model)
Shuffle one feature in the validation set, keep everything else, and measure how much the final model's PR-AUC drops (10 shuffles per feature). It works for any model (here `best_pipeline`) and measures importance **for this model on unseen seasons**.

In [ ]:
# 12.1 Permutation importance of the final model on validation (scoring = PR-AUC, 10 repeats)
perm = permutation_importance(best_pipeline, X_val, y_val, scoring="average_precision", n_repeats=10, random_state=SEED)
perm_importance = (pd.DataFrame({"feature": FEATURES, "mean_drop": perm.importances_mean, "std": perm.importances_std})
                   .sort_values("mean_drop", ascending=False).reset_index(drop=True))

fig, ax = plt.subplots(figsize=FIGSIZE)
plot_df = perm_importance.iloc[::-1]
ax.barh(plot_df["feature"], plot_df["mean_drop"], xerr=plot_df["std"], color="#2a78d6", ecolor="#52514e")
ax.axvline(0, color="#52514e", linewidth=1)
ax.set_title(f"Permutation importance on validation: {best_name}")
ax.set_xlabel("Drop in validation PR-AUC when the feature is shuffled"); ax.set_ylabel("Feature")
ax.tick_params(axis="y", labelsize=7); ax.grid(axis="x", alpha=0.3); ax.spines[["top", "right"]].set_visible(False)
plt.show()
perm_importance.head(8).round(4)

**What the chart shows (final model `RF_v2_tuned`):** the **grid position** is by far the most important input (shuffling it costs **0.062** PR-AUC), followed by the qualifying position (0.036) and the front-row flag (0.021); the form, standings and Q1-gap features each cost about 0.01. Pit-lane start, home race and field size do not matter (about 0), and a few (driver age, points before the race) are slightly negative, i.e. within noise (see the error bars: one standard deviation over 10 shuffles). Many features have small individual importances because correlated features stand in for each other when one is shuffled — the same reason the ablation drops in 11.1 are small.

**Not causal:** shuffling the grid hurts the model most because the model relies on it, not because the grid slot alone "makes" a podium: a good grid slot is the result of a fast car and driver.

### 12.2 SHAP (global): which features push the podium probability up or down
SHAP splits each prediction into one contribution per feature (in log-odds, the scale the model adds up). It is computed for the **tuned XGBoost model** (the best XGBoost attempt), the tree model SHAP was planned for (code conventions 3.1), on 500 random validation rows. The values come from XGBoost's own exact TreeSHAP (`pred_contribs=True`), which avoids version clashes between the `xgboost` and `shap` packages; the `shap` library draws the charts. Check: the contributions of each row plus the base value add up to the model's output.

In [ ]:
# 12.2 SHAP values of the best XGBoost attempt (exact TreeSHAP from XGBoost itself) on 500 validation rows
xgb_name = scores("val", [m for m in fitted_models if m.startswith("XGB_")]).loc[0, "model"]
xgb_model = fitted_models[xgb_name].named_steps["model"]
rng = np.random.default_rng(SEED)
X_shap = X_val.iloc[np.sort(rng.choice(len(X_val), size=min(500, len(X_val)), replace=False))]

def xgb_shap(X):
    """SHAP values (log-odds) for the rows of X: one column per feature + the base value in the last column."""
    booster = xgb_model.get_booster()
    contrib = booster.predict(xgb.DMatrix(X), pred_contribs=True)
    margin = booster.predict(xgb.DMatrix(X), output_margin=True)
    assert np.allclose(contrib.sum(axis=1), margin, atol=1e-3)            # contributions add up to the model output
    return shap.Explanation(values=contrib[:, :-1], base_values=contrib[:, -1], data=X.values, feature_names=FEATURES)

shap_values_val = xgb_shap(X_shap)
shap.summary_plot(shap_values_val.values, X_shap, show=False, plot_size=FIGSIZE, max_display=20)
plt.title(f"SHAP values on 500 validation rows: {xgb_name}")
plt.xlabel("SHAP value (effect on the log-odds of a podium)")
plt.show()

**What the chart shows:** for `XGB_v2_tuned`, the inputs that move predictions most are the **qualifying position** and the **grid position**: low values (blue = near the front) push the log-odds of a podium up (right), high values (red = further back) push it down by up to about −1. Next come the form features: a good team or driver average finish (blue, low = good) and a high recent podium rate (red) push towards the podium. Pit-lane start, field size and home race barely move any prediction.

In [ ]:
# 12.3 SHAP global importance: mean |SHAP value| per feature
shap.summary_plot(shap_values_val.values, X_shap, plot_type="bar", show=False, plot_size=FIGSIZE, max_display=20)
plt.title(f"Mean |SHAP value| on 500 validation rows: {xgb_name}")
plt.xlabel("Mean |SHAP value| (average effect on the log-odds)")
plt.show()

**What the chart shows:** the mean |SHAP| ranking is **quali_position** (about 0.73), **grid_position** (0.55), con_avg_finish_last10 (0.43), drv_avg_finish_last10 and drv_podium_rate_last10 (about 0.27), con_podium_rate_last10 (0.22), stand_driver_rank_before (0.16), quali_q1_gap_pct (0.12); grid_pitlane_start, ctx_field_size and ctx_home_race are close to 0. The same **groups** come first as in the permutation importance of the final forest (12.1: grid, qualifying, then form and standings), but the order of grid and qualifying position is swapped: the two are 0.97 correlated (8.14), so each model leans on one of them a bit more, and the two methods measure different things (SHAP: how much a feature moves predictions; permutation: how much the score drops without it).

### 12.4 LIME (local): why did the model rate one driver the way it did?
LIME explains **one prediction** of the final model: it creates many slightly changed copies of the driver's row, asks the model about them, and fits a small linear model around that one point. Two interesting validation cases:
1. **A surprise podium:** the podium finisher the final model rated **lowest** (complete features only, so LIME can perturb every value).
2. **A missed favourite:** the non-podium driver the final model rated **highest**.

LIME needs complete training rows to sample from, so its sampling data is the training set with missing values filled by the training medians (only for LIME's sampling; the model itself is unchanged).

In [ ]:
# 12.4 LIME setup + case 1: the surprise podium (validation)
names = (model_df.loc[X_val.index, ["raceId", "driverId", "year", "grid_position"]]
         .merge(clean["races"][["raceId", "name"]], on="raceId", how="left")
         .merge(clean["drivers"][["driverId", "forename", "surname"]], on="driverId", how="left")
         .set_index(X_val.index))
names["label"] = names["forename"] + " " + names["surname"] + ", " + names["year"].astype(str) + " " + names["name"]
complete = X_val.notna().all(axis=1)
p_final = pd.Series(val_probs[best_name], index=X_val.index)

surprise_row = p_final[(y_val == 1) & complete].idxmin()
favourite_row = p_final[(y_val == 0) & complete].idxmax()

lime_sampling_data = SimpleImputer(strategy="median").fit(X_train).transform(X_train)
lime_explainer = LimeTabularExplainer(lime_sampling_data, feature_names=FEATURES, class_names=["no podium", "podium"],
                                      mode="classification", discretize_continuous=True, random_state=SEED)
predict_fn = lambda arr: best_pipeline.predict_proba(pd.DataFrame(arr, columns=FEATURES))

def lime_case(row_id, title):
    exp = lime_explainer.explain_instance(X_val.loc[row_id].values.astype(float), predict_fn, num_features=10)
    fig = exp.as_pyplot_figure(label=1)
    fig.set_size_inches(*FIGSIZE)
    plt.title(title); plt.xlabel("Effect on the podium probability (local linear model)"); plt.ylabel("Feature condition")
    plt.tight_layout(); plt.show()
    return pd.DataFrame(exp.as_list(label=1), columns=["feature condition", "effect"])

print(f"Case 1: {names.loc[surprise_row, 'label']} — grid {names.loc[surprise_row, 'grid_position']:.0f}, "
      f"model probability {p_final[surprise_row]:.3f}, finished on the podium")
lime_case(surprise_row, f"LIME, surprise podium: {names.loc[surprise_row, 'label']} ({best_name})")

**What the chart shows:** the conditions that pushed this driver's probability down (red, negative) or up (green, positive) in the model's eyes. The surprise podium is **Daniil Kvyat, 2019 German Grand Prix**: started 14th, model probability 0.002, finished 3rd in a chaotic wet race. A surprise podium like this is a driver starting far back or with weak form who finished in the top 3 anyway: the model gave him a low chance **because** his grid and form features were weak, which is the right reasoning for a pre-race model — the race events that made the podium happen are, by design, not in the features. LIME's strongest conditions: **not on the front row** (−0.093), **grid 12–17** (−0.039), **no team podium** and **no driver podium** in the last 10 races (−0.032 each), **qualified 12th–16th** (−0.031); the only push up is "not a pit-lane start" (+0.031).

In [ ]:
# 12.5 LIME case 2: the favourite who missed the podium (validation)
print(f"Case 2: {names.loc[favourite_row, 'label']} — grid {names.loc[favourite_row, 'grid_position']:.0f}, "
      f"model probability {p_final[favourite_row]:.3f}, did NOT finish on the podium")
lime_case(favourite_row, f"LIME, missed favourite: {names.loc[favourite_row, 'label']} ({best_name})")

**What the chart shows:** the missed favourite is **Lewis Hamilton, 2020 Italian Grand Prix**: pole position, model probability 0.838, finished 7th after a stop-go penalty. For such a favourite the front grid slot, strong form and championship position all push the probability up. The model was "right for the right reasons" before the start; what it cannot see is what happened in the race (a crash, a failure, a penalty), which is exactly the information that must stay out of the features (no leakage). LIME's strongest conditions, all pushing up: **front row** (+0.097), **grid ≤ 6** (+0.093), **qualified ≤ 6th** (+0.086), driver podium share above 0.2 (+0.055) and team share above 0.25 (+0.049) in the last 10 races.

In [ ]:
# 12.6 SHAP waterfall (local) for the surprise podium of 12.4, explained by the XGBoost model
sv_one = xgb_shap(X_val.loc[[surprise_row]])[0]
shap.plots.waterfall(sv_one, max_display=10, show=False)
plt.gcf().set_size_inches(*FIGSIZE)
plt.title(f"SHAP waterfall ({xgb_name}): {names.loc[surprise_row, 'label']}")
plt.show()

**What the chart shows:** for Kvyat at the 2019 German GP, XGBoost starts from the average log-odds E[f(x)] = −1.82 (about a 14% podium chance) and ends at f(x) = **−4.65** (about 1%). The biggest pushes down are his **qualifying position 14** (−0.91) and **grid position 14** (−0.61), then his team's and his own average finish in the last 10 races (12.5 and 12.6: −0.38, −0.31) and **no podium** for him or his team in their last 10 races (−0.28, −0.23); only his age (25) pushes slightly up (+0.21). This agrees with the LIME explanation of the forest in 12.4 (same driver, different model and method): both blame the starting position first, then the weak form.

**Reminder (for the report):** both XAI methods describe **what the model relies on**, not why podiums happen. A feature can be important because it is a good **proxy** (e.g. grid position reflects car speed), not because changing it would change the result.

## 13. Inference Function
**Goal:** one function, `predict_podium(record)`, that takes the pre-race information of **one driver in one race** and returns the podium probability and a yes/no answer.

**Uses:** `best_pipeline` (final model with its preprocessing, fitted on train only), `THRESHOLD` (chosen on validation, Decision 21), `FEATURES`

**Creates:** `predict_podium()`

**How it works:** the record is turned into a one-row table with exactly the `FEATURES` columns. Anything missing (a feature left out, or `None`) becomes NaN, and the **same saved Pipeline** handles it the way it did in training (training medians + "was missing" indicators, or XGBoost's own handling of NaN). The yes/no answer uses the validation threshold, unchanged.

In [ ]:
# 13.1 Inference function: one driver, one race, information known before the start
def predict_podium(record, model=None, threshold=None):
    """Return the podium probability and a yes/no answer for one driver-race.

    record    : dict (or pandas Series) {feature name: value}; features may be missing or None -> treated as NaN
    model     : fitted Pipeline (default: best_pipeline, the final model with its training-fitted preprocessing)
    threshold : probability cut-off (default: THRESHOLD, chosen on validation)
    """
    model = best_pipeline if model is None else model
    threshold = THRESHOLD if threshold is None else threshold
    record = dict(record)
    unknown = sorted(set(record) - set(FEATURES))
    if unknown:
        raise ValueError(f"Unknown feature names (typo or post-race column?): {unknown}")
    row = pd.DataFrame([{f: record.get(f) for f in FEATURES}], columns=FEATURES).astype(float)   # None -> NaN
    probability = float(model.predict_proba(row)[0, 1])
    return {"podium_probability": round(probability, 4),
            "podium": bool(probability >= threshold),
            "threshold": round(float(threshold), 4),
            "missing_features": [f for f in FEATURES if pd.isna(row.at[0, f])]}

print(f"predict_podium uses {best_name} with THRESHOLD = {THRESHOLD:.3f} and {len(FEATURES)} features")

### 13.2 Demo 1: a complete record
A real validation row: the winner of the last race of the validation seasons (`VAL_END_YEAR`), with all 20 features known. The result must equal the Pipeline's own prediction for that row (checked with an assert).

In [ ]:
# 13.2 Demo with a complete record (last race of the validation seasons, the winner)
last_val_race = model_df.loc[val_mask].sort_values("date")["raceId"].iloc[-1]
winner_idx = model_df.index[(model_df["raceId"] == last_val_race) & (model_df["grid_position"] > 0)]
winner_idx = (model_df.loc[winner_idx].merge(clean["results"][["raceId", "driverId", "positionOrder"]], on=grain_key)
              .set_index(winner_idx).query("positionOrder == 1").index[0])
complete_record = model_df.loc[winner_idx, FEATURES].to_dict()
who = names.loc[winner_idx, "label"]

result = predict_podium(complete_record)
assert np.isclose(result["podium_probability"], best_pipeline.predict_proba(X_val.loc[[winner_idx]])[0, 1], atol=1e-4)
print(f"{who}: grid {complete_record['grid_position']:.0f}, actual result: podium = {bool(y_val.loc[winner_idx])}")
result

**What the output shows:** Max Verstappen at the 2021 Abu Dhabi Grand Prix (pole, won the race) gets a high podium probability (0.835, above `THRESHOLD`, so "podium" = True), no feature is missing, and the result equals the Pipeline's prediction for the same row (assert), so the function uses exactly the trained model and its preprocessing.

### 13.3 Demo 2: a record with missing values
A rookie in his first race (no form history, no championship position yet) whose qualifying time was not recorded: only the grid, the team's form and standings, and the context are known. The function still returns an answer and lists the missing features; the Pipeline fills them exactly as in training.

In [ ]:
# 13.3 Demo with missing values: a rookie, no qualifying time, no driver history (keys left out or None)
rookie_record = {
    "grid_position": 6, "grid_front_row": 0, "grid_pitlane_start": 0,
    "quali_position": None, "quali_q1_gap_pct": None, "quali_grid_penalty": None,      # qualifying not recorded
    f"con_podium_rate_last{FORM_WINDOW}": 0.30, f"con_avg_finish_last{FORM_WINDOW}": 6.5, f"con_dnf_rate_last{FORM_WINDOW}": 0.10,
    "stand_driver_points_before": 0, "stand_con_points_before": 40, "stand_con_rank_before": 3,
    "ctx_home_race": 0, "ctx_field_size": 20, "ctx_driver_age": 20.5, "ctx_driver_races_before": 0,
    # drv_* features and stand_driver_rank_before are simply left out (first race: no history, no rank)
}
predict_podium(rookie_record)

**What the output shows:** the function works with 7 missing features (listed in `missing_features`): it returns a probability for a driver starting 6th in a good team (0.094, below `THRESHOLD` → "no podium"), far lower than the pole-sitter in Demo 1. Missing values are never filled by hand here: the saved Pipeline does it with what it learned from the training seasons.

In [ ]:
# 13.4 Safety check: a wrong or post-race column name is refused (protects against typos and leakage)
try:
    predict_podium({**complete_record, "positionOrder": 1})
except ValueError as err:
    print("Refused:", err)

**What the output shows:** a record that contains a column the model does not know (here `positionOrder`, the race result) is refused with a clear message instead of being silently ignored, so a post-race value cannot slip into a prediction by mistake.